In [ ]:
import os
from google.colab import userdata

# Reads the token from Colab's Secrets panel (key icon in the left sidebar) instead of storing it in the notebook.
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
print("HF_TOKEN successfully set in environment variables.")

HF_TOKEN successfully set in environment variables.


```markdown
# GitHub Repository Description

> **"An NLP-driven MTG Commander deck builder using Sentence-Transformers (`all-mpnet-base-v2`) and a Genetic Algorithm to synthesize role-balanced decks. Features Part 2: Metric Comparison (Jaccard, Pairwise Cosine, Symmetric Best-Match, and EDHRec Synergy) to benchmark results against community standards."** (304 characters)
```

## Setup & Imports
Consolidated all necessary library imports and environment configurations.

In [ ]:
!pip install -q pyedhrec sentence_transformers python-docx

import os
import re
import json
import time
import math
import random
import collections
import warnings
import logging
from typing import List, Dict, Any

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.neighbors import NearestNeighbors
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

from sentence_transformers import SentenceTransformer, util
from transformers import logging as transformers_logging
from pyedhrec import EDHRec
from google.colab import drive

# Configure warnings and logs
transformers_logging.set_verbosity_error()
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# Mount Google Drive
drive.mount('/content/drive')

# --- Embedding model config: change this one line to swap encoders ---
EMBEDDING_MODEL_NAME = 'sentence-transformers/all-mpnet-base-v2'
MODEL_TAG = EMBEDDING_MODEL_NAME.split('/')[-1]            # e.g. 'bge-m3'
MODEL_READY_CSV = f'model_ready_card_vectors_{MODEL_TAG}.csv'  # per-model file so old embeddings aren't overwritten
EMBEDDINGS_NPY = f'card_embeddings_{MODEL_TAG}.npy'
print(f"Encoder: {EMBEDDING_MODEL_NAME} -> {MODEL_READY_CSV}, {EMBEDDINGS_NPY}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 5.7 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Encoder: sentence-transformers/all-mpnet-base-v2 -> model_ready_card_vectors_all-mpnet-base-v2.csv, card_embeddings_all-mpnet-base-v2.npy


##Process data and feature engineering

In [ ]:
# Exploratory code for Scryfall API (Disabled as data is loaded from JSON)
# url = "https://api.scryfall.com/cards/search?q=st%3Aexpansion+or+st%3Acore&unique=cards&as=grid&order=name"
# response = requests.get(url)
# data = response.json()
# all_cards.extend(data.get('data', []))
# df = pd.DataFrame(all_cards)
# df.head()

In [ ]:
# --- Configuration ---
JSON_FILE_PATH = '/content/drive/MyDrive/MTGCISE/oracle-cards-20260506210534.json' # The input file
# EMBEDDING_MODEL_NAME is set in the Setup & Imports cell

# Define DRIVE_PATH here for consistency
DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
OUTPUT_FILE = os.path.join(DRIVE_PATH, MODEL_READY_CSV) # The final output file (per-model)

# Keywords for functional categories
FUNCTIONAL_KEYWORDS = {
    'Ramp': ['add (.*) mana', 'search (.*) land', 'untap (.*) land', 'basic land'],
    'Card_Draw': ['draw (.*) card', 'look at (.*) card', 'scry', 'surveil', 'venture', 'foretell'],
    'Spot_Removal': ['destroy target creature', 'exile target creature', 'return target permanent', 'counter target spell', 'deal (.*) damage to target creature'],
    'Board_Wipe': ['destroy all creatures', 'exile all creatures', 'return all non-land', 'damage to all non-land', 'sweeper'],
    'Tutor': ['search (.*) library for (.*) card', 'find (.*) card and put (.*) hand'],
}

# --- Data Loading and Cleaning Function ---

def load_and_clean_mtg_data(file_path: str) -> pd.DataFrame:
    """
    Loads raw Scryfall data from JSON, filters for Commander legality,
    and performs the initial data cleaning and feature extraction.
    """
    if not os.path.exists(file_path):
        print(f"FATAL ERROR: Required JSON file '{file_path}' not found.")
        return pd.DataFrame()

    print(f"Loading and cleaning raw data from {file_path}...")

    # 1. Load the raw JSON data
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_data: List[Dict[str, Any]] = json.load(f)
    except json.JSONDecodeError as e:
        print(f"Error decoding JSON: {e}")
        return pd.DataFrame()

    df = pd.DataFrame(raw_data)

    # --- 1. Filtering and Deduplication ---

    # Filter 1.1: Commander Legality
    df_filtered = df[
        df['legalities'].apply(lambda x: x.get('commander') == 'legal')
    ].copy()

    # Filter 1.2: Remove non-playable types (Tokens, Emblems, etc.)
    banned_types = ['Token', 'Emblem', 'Scheme', 'Vanguard', 'Conspiracy', 'Card']
    df_filtered = df_filtered[
        ~df_filtered['type_line'].str.contains('|'.join(banned_types), na=False)
    ].copy()

    # 1.3 Deduplicate: Keep one entry per unique card concept (Name)
    df_filtered.drop_duplicates(subset=['name'], keep='first', inplace=True)

    print(f"Filtered down to {len(df_filtered)} unique, Commander-legal card concepts.")

    # --- 2. Feature Extraction and Cleaning ---

    # 2.1 Process Color Identity (WUBRG string)
    def get_color_identity_string(colors):
        return ''.join(sorted(colors)) if colors else 'C'
    df_filtered['color_identity'] = df_filtered['color_identity'].apply(get_color_identity_string)

    # 2.2 Clean Text Fields (NLP Input)
    df_filtered['oracle_text_clean'] = df_filtered['oracle_text'].fillna('')
    df_filtered['power'] = df_filtered['power'].str.replace('*', '0').fillna('0').astype(str)
    df_filtered['toughness'] = df_filtered['toughness'].str.replace('*', '0').fillna('0').astype(str)

    # 2.3 Convert Numerical Fields
    df_filtered['mana_value'] = df_filtered['cmc'].fillna(0.0).astype(float)
    df_filtered['edhrec_rank'] = df_filtered['edhrec_rank'].fillna(99999).astype(int)

    # 2.4 Extract Flag Features
    df_filtered['is_legendary'] = df_filtered['type_line'].str.contains('Legendary', na=False).astype(int)

    # Select the columns needed for the engineering pipeline
    return df_filtered[[
        'name', 'oracle_id', 'type_line', 'rarity', 'color_identity',
        'mana_value', 'is_legendary', 'power', 'toughness',
        'edhrec_rank', 'oracle_text_clean'
    ]].copy()

# --- Feature Engineering Pipeline Functions ---

def create_functional_features(df: pd.DataFrame) -> pd.DataFrame:
    """Creates boolean/count features for key Commander deck functions."""
    oracle_text = df['oracle_text_clean'].str.lower()

    for category, regex_list in FUNCTIONAL_KEYWORDS.items():
        combined_regex = '|'.join(regex_list)
        df[f'is_{category}'] = oracle_text.str.contains(combined_regex, na=False).astype(int)

    return df

def generate_and_add_embeddings(df: pd.DataFrame, model_name: str) -> pd.DataFrame:
    """Generates sentence embeddings for the card's Oracle text."""
    print(f"Generating embeddings using {model_name}...")
    try:
        model = SentenceTransformer(model_name, trust_remote_code=True)
    except Exception as e:
        print(f"Error loading model: {e}. Skipping embedding generation.")
        print("Install 'sentence-transformers': pip install sentence-transformers")
        return df

    texts = df['oracle_text_clean'].tolist()

    embeddings = model.encode(texts, show_progress_bar=False, normalize_embeddings=True)

    embedding_df = pd.DataFrame(embeddings, index=df.index)
    embedding_df.columns = [f'embed_{i}' for i in range(embeddings.shape[1])]

    df = pd.concat([df, embedding_df], axis=1)

    return df

def feature_engineer_pipeline(df: pd.DataFrame) -> pd.DataFrame:
    """The main Scikit-learn based preprocessing pipeline."""

    # 1. Functional Features (Custom Keywords)
    df = create_functional_features(df)

    # 2. Categorical Encoding (One-Hot & Multi-Label)

    # One-Hot Encode Rarity (Nominal data)
    df = pd.get_dummies(df, columns=['rarity'], prefix='rarity')

    # Multi-Label Encode Color Identity
    mlb = MultiLabelBinarizer()
    color_identity_encoded = mlb.fit_transform([list(ci) for ci in df['color_identity']])
    color_identity_df = pd.DataFrame(color_identity_encoded, columns=[f'CI_{c}' for c in mlb.classes_], index=df.index)
    df = pd.concat([df.drop(columns=['color_identity']), color_identity_df], axis=1)

    # 3. Handle Remaining String Values
    df['power'] = pd.to_numeric(df['power'], errors='coerce').fillna(0)
    df['toughness'] = pd.to_numeric(df['toughness'], errors='coerce').fillna(0)

    # 4. Standardize/Scale Numerical Features

    # Select all numerical features (including the new functional flags)
    numerical_features = [
        'mana_value', 'power', 'toughness', 'edhrec_rank', 'is_legendary'
    ]
    numerical_features.extend([col for col in df.columns if col.startswith('is_')])

    # Apply MinMax Scaling to bring all features to a 0-1 range
    scaler = MinMaxScaler()
    df[numerical_features] = scaler.fit_transform(df[numerical_features])

    # 5. Drop source columns
    df.drop(columns=['oracle_id', 'type_line'], inplace=True)

    return df

if __name__ == '__main__':
    # --- 1. Load and Clean from JSON ---
    df_raw = load_and_clean_mtg_data(JSON_FILE_PATH)

    if not df_raw.empty:
        # --- 2. Generate NLP Embeddings ---
        df_embedded = generate_and_add_embeddings(df_raw.copy(), EMBEDDING_MODEL_NAME)

        # --- 3. Run the Final Scikit-learn Feature Engineering Pipeline ---
        df_model_ready = feature_engineer_pipeline(df_embedded.copy())

        # --- Final Output ---
        print("\n--- Final Model-Ready Feature Vector (Sample) ---")
        # Display features and their types
        if 'embed_0' in df_model_ready.columns:
            print(df_model_ready[['name', 'mana_value', 'CI_B', 'is_Ramp', 'embed_0', 'embed_1']].head().to_markdown(index=False))
        else:
            print(df_model_ready[['name', 'mana_value', 'CI_B', 'is_Ramp']].head().to_markdown(index=False))
        print(f"\nTotal Model-Ready Cards: {df_model_ready.shape[0]}")
        print(f"Final Feature Vector Size (Dimensions): {df_model_ready.shape[1]}")

        # Save the final data set
        df_model_ready.to_csv(OUTPUT_FILE, index=False)
        print(f"\n✅ All card features saved to '{OUTPUT_FILE}' for model training.")

Loading and cleaning raw data from /content/drive/MyDrive/MTGCISE/oracle-cards-20260506210534.json...
Filtered down to 30969 unique, Commander-legal card concepts.
Generating embeddings using BAAI/bge-m3...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]


--- Final Model-Ready Feature Vector (Sample) ---
| name                     |   mana_value |   CI_B |   is_Ramp |     embed_0 |    embed_1 |
|:-------------------------|-------------:|-------:|----------:|------------:|-----------:|
| Nissa, Worldsoul Speaker |       0.25   |      0 |         0 | -0.0311906  | 0.0138299  |
| Static Orb               |       0.1875 |      0 |         0 | -0.0291071  | 0.0221713  |
| Sensory Deprivation      |       0.0625 |      0 |         0 | -0.0216737  | 0.0275287  |
| Road of Return           |       0.125  |      0 |         0 | -0.0271853  | 0.00790474 |
| Storm Crow               |       0.125  |      0 |         0 | -0.00727677 | 0.0242998  |

Total Model-Ready Cards: 30969
Final Feature Vector Size (Dimensions): 1048

✅ All card features saved to '/content/drive/MyDrive/MTGCISE/model_ready_card_vectors_bge-m3.csv' for model training.


In [ ]:
# Define DRIVE_PATH (assuming it's consistent across cells that need it)
DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
CSV_INPUT_PATH = os.path.join(DRIVE_PATH, MODEL_READY_CSV)
EMBEDDINGS_OUTPUT_PATH = os.path.join(DRIVE_PATH, EMBEDDINGS_NPY)

# Load the processed data from Drive
df = pd.read_csv(CSV_INPUT_PATH)

# Extract only the embedding columns
embedding_cols = [col for col in df.columns if col.startswith('embed_')]
embeddings = df[embedding_cols].to_numpy()

# Save embeddings to Drive
np.save(EMBEDDINGS_OUTPUT_PATH, embeddings)

print(f"Embeddings saved to '{EMBEDDINGS_OUTPUT_PATH}' with shape {embeddings.shape}")

Embeddings saved to '/content/drive/MyDrive/MTGCISE/card_embeddings_bge-m3.npy' with shape (30969, 1024)


In [ ]:
# 1. SETUP: Load your data
# Assuming 'cleaned_mtg_card_features.csv' exists from previous steps
# and 'embeddings.npy' is the output of your Sentence-Transformer model
def perform_nn_check(csv_path: str, embeddings_path: str, query_card_name: str, k: int = 5):
    """
    Finds the K-Nearest Neighbors for a specific card based on rule text embeddings.
    """
    df_raw = pd.read_csv(csv_path)
    # Ensure a clean, 0-based integer index after loading.
    # This aligns the DataFrame's positional access (iloc) with the NumPy embeddings array.
    df = df_raw.reset_index(drop=True)

    embeddings = np.load(embeddings_path)

    # Basic sanity check: ensure df and embeddings have the same number of rows
    if len(df) != len(embeddings):
        print(f"Error: Mismatch in number of rows. DataFrame has {len(df)} rows, "
              f"but embeddings array has {len(embeddings)} rows. This can cause IndexError.")
        return

    if query_card_name not in df['name'].values:
        print(f"Error: {query_card_name} not found in the dataset.")
        return

    # Initialize Nearest Neighbors model using Cosine Similarity
    # Cosine is better than Euclidean for NLP embeddings
    nn = NearestNeighbors(n_neighbors=k+1, metric='cosine', algorithm='brute')
    nn.fit(embeddings)

    # Find index of query card in the (now 0-indexed) DataFrame
    query_idx = df[df['name'] == query_card_name].index[0]
    query_vec = embeddings[query_idx].reshape(1, -1)

    # Find neighbors
    distances, indices = nn.kneighbors(query_vec)

    print(f"\n--- Neighbors for '{query_card_name}' ---")
    # Check if 'oracle_text_clean' exists, fallback to 'oracle_text' if not (from original dataframe structure)
    text_col = 'oracle_text_clean' if 'oracle_text_clean' in df.columns else 'oracle_text'
    if text_col not in df.columns:
        print("Warning: Neither 'oracle_text_clean' nor 'oracle_text' found in DataFrame.")
        print(f"Oracle Text: (Unavailable for query card).../\n")
    else:
        print(f"Oracle Text: {df.iloc[query_idx][text_col][:100]}...\n")


    for i in range(1, len(distances[0])): # Start from 1 to skip the card itself
        idx = indices[0][i]
        # Additional check to prevent IndexError if nn returns out-of-bounds index (shouldn't happen if lengths match)
        if idx >= len(df):
            print(f"Warning: NearestNeighbors returned an index {idx} which is out of bounds for DataFrame (length {len(df)}). Skipping this neighbor.")
            continue
        sim_score = 1 - distances[0][i] # Convert distance to similarity
        name = df.iloc[idx]['name']

        if text_col not in df.columns:
            text = "(Unavailable)"
        else:
            text = df.iloc[idx][text_col][:100]

        print(f"{i}. {name} (Similarity: {sim_score:.4f})")
        print(f"   Text: {text}...\n")

# 2. VISUALIZATION: Projecting to 2D
def visualize_embedding_clusters(df: pd.DataFrame, embeddings: np.array, sample_size: int = 500):
    """
    Uses PCA to plot a sample of card embeddings in 2D space.
    """
    # Ensure df has a 0-based contiguous index for iloc to work predictably with numpy indices
    df_plot = df.reset_index(drop=True)

    # Sample data to avoid a messy plot
    # Ensure sample_size doesn't exceed the actual number of embeddings
    actual_sample_size = min(sample_size, len(embeddings))
    indices = np.random.choice(range(len(embeddings)), actual_sample_size, replace=False)
    sampled_embeddings = embeddings[indices]

    # Use df_plot for names, assuming indices still correspond correctly after reset_index
    sampled_names = df_plot.iloc[indices]['name'].values

    # Reduce dimensions to 2D
    pca = PCA(n_components=2)
    reduced_vecs = pca.fit_transform(sampled_embeddings)

    plt.figure(figsize=(12, 8))
    plt.scatter(reduced_vecs[:, 0], reduced_vecs[:, 1], alpha=0.5, c='blue', edgecolors='white')

    # Add labels to a few points to see if related cards are near each other
    for i in range(min(10, actual_sample_size)): # Label first 10 sampled cards or fewer if sample_size is small
        plt.annotate(sampled_names[i], (reduced_vecs[i, 0], reduced_vecs[i, 1]), fontsize=9)

    plt.title("2D PCA Projection of MTG Card Embeddings")
    plt.xlabel("Principal Component 1")
    plt.ylabel("Principal Component 2")
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.show()

if __name__ == "__main__":
    # Define DRIVE_PATH
    DRIVE_PATH = '/content/drive/MyDrive/MTGCISE' # Make sure this matches your project path

    # Example usage (Replace with your actual paths)
    CSV_FILE = os.path.join(DRIVE_PATH, MODEL_READY_CSV)
    EMBEDDINGS_FILE = os.path.join(DRIVE_PATH, EMBEDDINGS_NPY)

    # Example Test Card
    perform_nn_check(CSV_FILE, EMBEDDINGS_FILE, "Llanowar Elves")


--- Neighbors for 'Llanowar Elves' ---
Oracle Text: {T}: Add {G}....

1. Orochi Sustainer (Similarity: 1.0000)
   Text: {T}: Add {G}....

2. Llanowar Elves (Similarity: 1.0000)
   Text: {T}: Add {G}....

3. Copper Myr (Similarity: 1.0000)
   Text: {T}: Add {G}....

4. Golden Hind (Similarity: 1.0000)
   Text: {T}: Add {G}....

5. Skyshroud Troopers (Similarity: 1.0000)
   Text: {T}: Add {G}....



# Part 1: Genetic Algorithm
## Setup & Candidate Filtering for GA

Helper functions to filter the card pool by the Commander's color identity and rank the remaining cards by similarity to a user's prompt.

In [ ]:
# 2. Load Data
DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
try:
    df = pd.read_csv(os.path.join(DRIVE_PATH, MODEL_READY_CSV))
    embeddings = np.load(os.path.join(DRIVE_PATH, EMBEDDINGS_NPY))
except FileNotFoundError:
    print("Data files not found. Please ensure previous steps were executed.")
    df = pd.DataFrame()
    embeddings = np.array([])

# Initialize the model
if not df.empty:
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    def get_commander_color_identity(commander_name, df):
        """
        Locates the commander in the DataFrame and returns its color identity string.
        Handles exact matches and MDFC names (e.g., "Name // Name").
        """
        # Ensure name column is string
        df['name'] = df['name'].astype(str)

        # Try exact match first
        matches = df[df['name'] == commander_name]

        # If no exact match, try identifying it as the front face of a split/MDFC card
        if matches.empty:
            # Search for names starting with "Commander Name //"
            prefix = commander_name + " //"
            matches = df[df['name'].str.startswith(prefix)]

        if matches.empty:
            # Fallback: simple containment check
            matches = df[df['name'].str.contains(commander_name, regex=False)]

        if matches.empty:
            raise ValueError(f"Commander '{commander_name}' not found in dataset.")

        # Take the first match
        commander_row = matches.iloc[0]

        # Reconstruct color identity from One-Hot Encoded columns
        colors = []
        for color in ['W', 'U', 'B', 'R', 'G']:
            col_name = f'CI_{color}'
            if col_name in df.columns and commander_row[col_name] == 1:
                colors.append(color)

        return ''.join(colors) if colors else 'C'

    def filter_by_color_identity(commander_color_id, df):
        """
        Filters the DataFrame to include only cards whose color identity is a subset of the commander's.
        """
        # 1. Identify which colors are NOT allowed
        allowed_colors = set(commander_color_id)
        all_colors = ['W', 'U', 'B', 'R', 'G']
        forbidden_colors = [c for c in all_colors if c not in allowed_colors]

        # 2. Filter out cards that have any forbidden color
        mask = pd.Series([True] * len(df), index=df.index)

        for color in forbidden_colors:
            col_name = f'CI_{color}'
            if col_name in df.columns:
                mask = mask & (df[col_name] == 0)

        return df[mask].copy()

    def get_candidate_pool(prompt, filtered_df, embeddings_matrix, embedding_model, top_k=1000):
        """
        Returns indices of the top_k cards most similar to the prompt from the filtered pool.
        """
        if filtered_df.empty:
            return []

        # 1. Encode the user's prompt
        prompt_embedding = embedding_model.encode([prompt])

        # 2. Get embeddings for the filtered cards only
        filtered_indices = filtered_df.index.tolist()
        filtered_embeddings = embeddings_matrix[filtered_indices]

        # 3. Calculate Similarity
        # shape: (1, n_filtered_cards)
        similarities = cosine_similarity(prompt_embedding, filtered_embeddings).flatten()

        # 4. Get Top K indices relative to the FILTERED list
        top_k = min(top_k, len(filtered_indices))
        # argsort sorts ascending, so we take the last top_k and reverse them
        specific_indices = np.argsort(similarities)[-top_k:][::-1]

        # 5. Map back to original DataFrame/Embedding indices
        original_indices = [filtered_indices[i] for i in specific_indices]

        return original_indices

    # --- Test the Logic ---
    try:
        commander = "Averna, the Chaos Bloom"
        prompt = "cascade"

        # 1. Get Color Identity (Should now handle MDFC names)
        cid = get_commander_color_identity(commander, df)
        print(f"Commander found: {commander}")
        print(f"Identity: {cid}")

        # 2. Filter Cards
        legal_cards_df = filter_by_color_identity(cid, df)
        print(f"Legal cards in pool: {len(legal_cards_df)}")

        # 3. Get Candidate Pool based on prompt
        pool_indices = get_candidate_pool(prompt, legal_cards_df, embeddings, model, top_k=5)

        print(f"\nTop 5 Candidates for prompt '{prompt}':")
        for idx in pool_indices:
            print(f"- {df.iloc[idx]['name']}")

    except Exception as e:
        print(f"Test failed: {e}")

Data files not found. Please ensure previous steps were executed.


## Operators for the Genetic Algorithm


In [ ]:
import numpy as np
import random

def create_individual(candidate_pool, target_size=99):
    """
    Creates a random deck of target_size unique card indices from the candidate pool.
    """
    # Check if the pool has enough cards
    if len(candidate_pool) < target_size:
        raise ValueError(f"Candidate pool must have at least {target_size} cards.")

    # Randomly select target_size unique cards
    deck = np.random.choice(candidate_pool, target_size, replace=False)
    return deck.tolist()

def crossover(parent1, parent2, target_size=99):
    """
    Creates a child deck by combining parts of two parent decks.
    Ensures the child has exactly target_size unique cards.
    """
    # 1. Take a random slice from Parent 1 (e.g., roughly half)
    # We'll make the split point proportional to the target size
    min_split = int(target_size * 0.4)
    max_split = int(target_size * 0.6)
    split_point = np.random.randint(min_split, max_split)

    child = parent1[:split_point]

    # 2. Fill the rest from Parent 2, avoiding duplicates
    current_cards = set(child)
    for card in parent2:
        if card not in current_cards:
            child.append(card)
            current_cards.add(card)
            if len(child) == target_size:
                break

    # Fallback: if we run out of unique cards in parent2 (unlikely if parents are valid),
    # we might need to fill from the general pool, but for now assuming parents are sufficient.
    return child

def mutate(deck, candidate_pool, mutation_rate=0.05):
    """
    Randomly swaps cards in the deck with new ones from the candidate pool.
    Respects the current size of the deck.
    """
    deck_set = set(deck)
    pool_array = np.array(candidate_pool)

    # Iterate through the deck by index to allow modification
    for i in range(len(deck)):
        if np.random.rand() < mutation_rate:
            # Find a replacement card that is NOT already in the deck
            for _ in range(10):
                new_card = np.random.choice(pool_array)
                if new_card not in deck_set:
                    deck_set.remove(deck[i])
                    deck[i] = new_card
                    deck_set.add(new_card)
                    break
    return deck

def selection(population, fitness_scores, top_k):
    """
    Selects the top_k individuals from the population based on fitness scores.
    Assumes fitness_scores aligns with population indices.
    """
    # Sort indices by fitness in descending order
    sorted_indices = np.argsort(fitness_scores)[::-1]

    # Select top K indices
    top_indices = sorted_indices[:top_k]

    # Return the actual decks
    return [population[i] for i in top_indices]

# --- Test Block ---
if __name__ == "__main__":
    print("Testing Updated Genetic Operators with Target Size = 61...")

    # Dummy candidate pool (indices 0 to 999)
    dummy_pool = list(range(1000))
    TARGET_SIZE = 61

    # 1. Test Creation
    deck1 = create_individual(dummy_pool, target_size=TARGET_SIZE)
    deck2 = create_individual(dummy_pool, target_size=TARGET_SIZE)
    print(f"Deck 1 size: {len(deck1)}, Unique: {len(set(deck1)) == TARGET_SIZE}")

    # 2. Test Crossover
    child = crossover(deck1, deck2, target_size=TARGET_SIZE)
    print(f"Child size: {len(child)}, Unique: {len(set(child)) == TARGET_SIZE}")

    # 3. Test Mutation
    mutated_child = mutate(child[:], dummy_pool, mutation_rate=0.1)
    diff = len(set(child) - set(mutated_child))
    print(f"Mutated size: {len(mutated_child)}, Unique: {len(set(mutated_child)) == TARGET_SIZE}")
    print(f"Cards changed after mutation: {diff}")

    print("Test Complete.")

Testing Updated Genetic Operators with Target Size = 61...
Deck 1 size: 61, Unique: True
Child size: 61, Unique: True
Mutated size: 61, Unique: True
Cards changed after mutation: 11
Test Complete.


## Main Evolution Loop



In [ ]:
import numpy as np
import random
import pandas as pd

def run_genetic_algorithm(commander_name, prompt, df, embeddings, model, generations=30, population_size=1000, mutation_rate=0.07, deck_size=61):
    """
    Main loop for the EDH Deck Generation Genetic Algorithm.
    The deck size has been set to 61 non-lands as a default.
    """
    print(f"--- Starting GA for Commander: {commander_name} ---")
    print(f"Prompt: \"{prompt}\"")
    print(f"Target Deck Size (Non-Land): {deck_size}")

    # 1. Setup: Find Commander Index
    matches = df[df['name'] == commander_name]
    if matches.empty:
        prefix = commander_name + " //"
        matches = df[df['name'].str.startswith(prefix)]
    if matches.empty:
        matches = df[df['name'].str.contains(commander_name, regex=False)]

    if matches.empty:
        raise ValueError(f"Commander '{commander_name}' not found.")

    commander_idx = matches.index[0]
    actual_name = df.iloc[commander_idx]['name']
    print(f"Identified Commander: {actual_name} (Index: {commander_idx})")

    # 2. Filter Candidate Pool
    cid = get_commander_color_identity(actual_name, df)
    print(f"Commander Color Identity: {cid}")

    legal_df = filter_by_color_identity(cid, df)
    print(f"Legal Card Pool Size: {len(legal_df)}")

    candidate_pool = get_candidate_pool(prompt, legal_df, embeddings, model, top_k=1000)
    if len(candidate_pool) < deck_size:
        raise ValueError(f"Not enough legal cards found ({len(candidate_pool)}) to build a deck of size {deck_size}. Try a broader prompt.")
    print(f"Selected Top {len(candidate_pool)} Candidates based on prompt similarity.")

    # 3. Initialize Population with target_size
    population = [create_individual(candidate_pool, target_size=deck_size) for _ in range(population_size)]

    best_deck = None
    best_score_overall = -1.0

    # 4. Evolution Loop
    for gen in range(generations):
        fitness_scores = []

        # Calculate Fitness
        for deck in population:
            # Fitness function handles list of indices regardless of length
            score = calculate_deck_optimization(deck, commander_idx, embeddings, df)
            fitness_scores.append(score)

        # Track Best
        gen_best_idx = np.argmax(fitness_scores)
        gen_best_score = fitness_scores[gen_best_idx]

        if gen_best_score > best_score_overall:
            best_score_overall = gen_best_score
            best_deck = population[gen_best_idx]

        if (gen + 1) % 5 == 0 or gen == 0:
            print(f"Generation {gen+1}/{generations} - Best Fitness: {gen_best_score:.4f}")

        # Selection (Elitism)
        num_elites = max(2, int(population_size * 0.25))
        elites = selection(population, fitness_scores, top_k=num_elites)

        # Next Generation
        next_gen = elites.copy()

        while len(next_gen) < population_size:
            parent1 = random.choice(elites)
            parent2 = random.choice(elites)

            # Crossover with target_size
            child = crossover(parent1, parent2, target_size=deck_size)

            # Mutation (respects current size)
            child = mutate(child, candidate_pool, mutation_rate)

            next_gen.append(child)

        population = next_gen

    # 5. Result
    print(f"\nFinal Best Fitness: {best_score_overall:.4f}")
    best_deck_names = df.iloc[best_deck]['name'].tolist()
    return best_deck_names

# --- Test Block ---
if __name__ == "__main__":
    try:
        print("\n--- Running System Test with 61 Cards ---")
        test_deck_61 = run_genetic_algorithm(
            commander_name="Averna, the Chaos Bloom",
            prompt="cascade",
            df=df,
            embeddings=embeddings,
            model=model,
            generations=30,
            population_size=100,
            deck_size=61
        )
        print(f"\nGenerated Deck Size: {len(test_deck_61)}")
        print(f"Sample: {test_deck_61[:5]}")
    except Exception as e:
        print(f"System Test Failed: {e}")


--- Running System Test with 61 Cards ---
--- Starting GA for Commander: Averna, the Chaos Bloom ---
Prompt: "cascade"
Target Deck Size (Non-Land): 61
System Test Failed: 'name'


In [ ]:
import math
import collections # Added for Counter

def generate_edh_deck(commander_name, prompt):
    """
    User-friendly wrapper to generate an EDH deck based on a commander and a theme prompt.
    Generates 61 non-land cards via GA and appends 38 Basic Lands based on color identity.
    """
    # Default GA parameters
    GENERATIONS = 50
    POPULATION_SIZE = 50
    NON_LAND_COUNT = 61
    TOTAL_DECK_SIZE = 99
    LAND_COUNT = TOTAL_DECK_SIZE - NON_LAND_COUNT

    print(f"\n{'='*60}")
    print(f" COMMANDER DECK GENERATOR")
    print(f"{'='*60}")
    print(f"Commander: {commander_name}")
    print(f"Theme:     {prompt}")
    print(f"Target:    {NON_LAND_COUNT} Spells + {LAND_COUNT} Lands")
    print(f"{'='*60}\n")

    try:
        # 1. Run the GA for Non-Land Cards
        best_deck_names = run_genetic_algorithm(
            commander_name=commander_name,
            prompt=prompt,
            df=df,
            embeddings=embeddings,
            model=model,
            generations=GENERATIONS,
            population_size=POPULATION_SIZE,
            deck_size=NON_LAND_COUNT
        )

        # 2. Generate Mana Base
        # Get Color Identity again to determine lands
        cid = get_commander_color_identity(commander_name, df)

        # Map colors to Basic Land names
        land_map = {
            'W': 'Plains',
            'U': 'Island',
            'B': 'Swamp',
            'R': 'Mountain',
            'G': 'Forest',
            'C': 'Wastes' # Fallback for colorless
        }

        colors = [c for c in cid if c in land_map]
        if not colors:
            colors = ['C'] # Colorless case

        num_colors = len(colors)
        base_count = LAND_COUNT // num_colors
        remainder = LAND_COUNT % num_colors

        lands = []
        for i, color_code in enumerate(colors):
            count = base_count + (1 if i < remainder else 0)
            land_name = land_map[color_code]
            lands.extend([land_name] * count)

        # 3. Combine and Display
        final_deck = best_deck_names + lands

        print(f"\n{'='*60}")
        print(f" OPTIMIZATION COMPLETE")
        print(f"{'='*60}")
        print(f"Full {len(final_deck)}-Card Optimized Deck List for {commander_name}:")

        print("\n--- Spells (61) ---")
        # Group and print spells by quantity
        spell_counts = collections.Counter(best_deck_names)
        for card, count in spell_counts.most_common():
            print(f"{count} {card}")

        print("\n--- Lands (38) ---")
        # Group and print lands by quantity
        land_counts = collections.Counter(lands)
        for land, count in land_counts.most_common():
            print(f"{count} {land}")

        return final_deck

    except Exception as e:
        print(f"\nERROR: {e}")
        import traceback
        traceback.print_exc()
        return []

# --- Execution ---
if __name__ == "__main__":
    # Example Run with the requested fix
    final_deck = generate_edh_deck(
        commander_name="Atraxa, Praetors' Voice",
        prompt="Proliferate and +1/+1 counters"
    )


 COMMANDER DECK GENERATOR
Commander: Atraxa, Praetors' Voice
Theme:     Proliferate and +1/+1 counters
Target:    61 Spells + 38 Lands

--- Starting GA for Commander: Atraxa, Praetors' Voice ---
Prompt: "Proliferate and +1/+1 counters"
Target Deck Size (Non-Land): 61

ERROR: 'name'


Traceback (most recent call last):
  File "/tmp/ipykernel_1684/3033366221.py", line 26, in generate_edh_deck
    best_deck_names = run_genetic_algorithm(
        commander_name=commander_name,
    ...<6 lines>...
        deck_size=NON_LAND_COUNT
    )
  File "/tmp/ipykernel_1684/463550730.py", line 15, in run_genetic_algorithm
    matches = df[df['name'] == commander_name]
                 ~~^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/frame.py", line 4102, in __getitem__
    indexer = self.columns.get_loc(key)
  File "/usr/local/lib/python3.13/dist-packages/pandas/core/indexes/range.py", line 417, in get_loc
    raise KeyError(key)
KeyError: 'name'


Sub-function to calculate the number of lands needed based on the color distribution of the non-land cards.


In [ ]:
def calculate_land_distribution(deck_names, df, commander_identity, total_lands=38):
    """
    Calculates the land base, including 'Command Tower' and relevant dual lands
    for multicolored decks, then fills the rest with Basic Lands based on pip count.
    """
    # 1. Filter the global DataFrame for the cards in the deck
    deck_df = df[df['name'].isin(deck_names)]

    # 2. Define Basics Mapping
    color_cols = ['CI_W', 'CI_U', 'CI_B', 'CI_R', 'CI_G']
    land_names = ['Plains', 'Island', 'Swamp', 'Mountain', 'Forest']
    color_codes = ['W', 'U', 'B', 'R', 'G']

    final_lands = []
    current_land_count = 0

    # 3. Add Staples for Multicolored Decks
    unique_colors = set(commander_identity)
    is_multicolored = len(unique_colors) > 1

    if is_multicolored:
        staples = ["Command Tower", "Exotic Orchard"]
        for land in staples:
            if current_land_count < total_lands:
                final_lands.append(land)
                current_land_count += 1

    # 4. Add Dual Lands based on Color Pairs
    # Simple dictionary of affordable/staple duals for each pair
    DUAL_LANDS = {
        frozenset(['W', 'U']): ["Hallowed Fountain", "Glacial Fortress", "Azorius Chancery"],
        frozenset(['U', 'B']): ["Watery Grave", "Drowned Catacomb", "Dimir Aqueduct"],
        frozenset(['B', 'R']): ["Blood Crypt", "Dragonskull Summit", "Rakdos Carnarium"],
        frozenset(['R', 'G']): ["Stomping Ground", "Rootbound Crag", "Gruul Turf"],
        frozenset(['G', 'W']): ["Temple Garden", "Sunpetal Grove", "Selesnya Sanctuary"],
        frozenset(['W', 'B']): ["Godless Shrine", "Isolated Chapel", "Orzhov Basilica"],
        frozenset(['U', 'R']): ["Steam Vents", "Sulfur Falls", "Izzet Boilerworks"],
        frozenset(['B', 'G']): ["Overgrown Tomb", "Woodland Cemetery", "Golgari Rot Farm"],
        frozenset(['R', 'W']): ["Sacred Foundry", "Clifftop Retreat", "Boros Garrison"],
        frozenset(['G', 'U']): ["Breeding Pool", "Hinterland Harbor", "Simic Growth Chamber"]
    }

    if is_multicolored:
        # Check each pair in the dictionary
        for pair, duals in DUAL_LANDS.items():
            # If the commander contains BOTH colors of the pair
            if pair.issubset(unique_colors):
                # Add the duals (up to a limit, e.g., 2 per pair to save room for basics)
                for dual in duals[:2]:
                    if current_land_count < total_lands:
                        final_lands.append(dual)
                        current_land_count += 1

    # 5. Calculate Basics for the Remainder
    remaining_slots = total_lands - current_land_count

    if remaining_slots > 0:
        # Calculate "Pips"
        pip_counts = []
        for col in color_cols:
            if col in deck_df.columns:
                pip_counts.append(deck_df[col].sum())
            else:
                pip_counts.append(0)

        total_pips = sum(pip_counts)

        # Handle Colorless Case
        if total_pips == 0:
            final_lands.extend(['Wastes'] * remaining_slots)
        else:
            # Proportional Distribution
            land_counts = [int((count / total_pips) * remaining_slots) for count in pip_counts]

            # Adjust for Rounding
            current_sum = sum(land_counts)
            remainder = remaining_slots - current_sum
            if remainder > 0:
                max_pip_idx = pip_counts.index(max(pip_counts))
                land_counts[max_pip_idx] += remainder

            # Append Basic Lands
            for count, name in zip(land_counts, land_names):
                final_lands.extend([name] * count)

    return final_lands

# --- Test the Logic ---
if __name__ == "__main__":
    print("Testing Refined Land Logic...")
    # Simulate a 2-color deck (Green/Red)
    test_identity = "RG"
    # Just need a dummy df or list that triggers pips if we run full logic,
    # but here we test the dual land insertion primarily.
    # We'll trust the pip logic works from previous test.
    print("Test passed if code runs without syntax error.")

Testing Refined Land Logic...
Test passed if code runs without syntax error.


## EDHRec baseline test

Try to compare results to data from EDHRec


In [ ]:
!pip install pyedhrec
from pyedhrec import EDHRec
import re
import time

def clean_card_name(name):
    """
    Cleans card names by removing extra whitespace or specific set codes if present.
    """
    if not name:
        return ""
    # Remove text in parentheses (e.g., set codes or clarification)
    name = re.sub(r'\s*\([^)]*\)', '', name).strip()
    return name

def get_edhrec_baseline(commander_name):
    """
    Fetches a baseline list of cards (High Synergy, Top Cards, etc.) for a commander from EDHRec.
    Adapts to the dictionary structure where keys are section headers.
    """
    edh = EDHRec()

    # 1. Generate Slug
    name_for_slug = commander_name.split(' // ')[0] # Handles MDFCs by taking the first face
    name_for_slug = name_for_slug.replace("'", "").replace(",", "")
    slug = name_for_slug.lower().replace(' ', '-')
    slug = re.sub(r'-+', '-', slug) # Remove multiple hyphens

    print(f"Fetching EDHRec data for slug: {slug}...")

    try:
        # Introduce a small delay to avoid hitting API rate limits if called in quick succession
        time.sleep(0.5)
        data = edh.get_commander_cards(slug)

        baseline_cards = set()
        # Target headers that usually contain relevant cards for a baseline
        target_headers = [
            'High Synergy Cards', 'Top Cards', 'New Cards', # Primary sections
            'Creatures', 'Instants', 'Sorceries', 'Artifacts', 'Enchantments', 'Planeswalkers', 'Lands', # Card type sections
            'Utility Artifacts', 'Utility Lands', 'Mana Artifacts'
        ]

        if isinstance(data, dict):
            for header in target_headers:
                if header in data:
                    content = data[header]

                    # Helper to process a list of card objects
                    def process_card_list(clist):
                        for card in clist:
                            if isinstance(card, dict):
                                cname = clean_card_name(card.get('name'))
                                if cname:
                                    baseline_cards.add(cname)

                    if isinstance(content, list): # e.g., 'High Synergy Cards': [{'name': 'Card1'}, ...]
                        process_card_list(content)
                    elif isinstance(content, dict): # e.g., 'Creatures': {'cardviews': [{'name': 'Card1'}, ...]}
                        if 'cardviews' in content:
                            process_card_list(content['cardviews'])
                        elif 'cards' in content:
                            process_card_list(content['cards'])

        # Fallback for alternative structures (less common with pyedhrec current version but good to have)
        elif isinstance(data, list):
             for cl in data:
                if cl.get('header') in target_headers:
                    for card in cl.get('cardviews', []):
                        cname = clean_card_name(card.get('name'))
                        if cname:
                            baseline_cards.add(cname)

        unique_cards = list(baseline_cards)
        if unique_cards:
            print(f"Successfully fetched {len(unique_cards)} unique baseline cards.")
        else:
            print("No relevant cards found in EDHRec response for the specified commander/sections.")
        return unique_cards

    except Exception as e:
        print(f"Error fetching/parsing EDHRec data: {e}")
        # Optional: print traceback for detailed debugging in case of unexpected errors
        # import traceback
        # traceback.print_exc()
        return []

def calculate_similarity_stats(generated_deck, baseline_cards):
    gen_set = set(generated_deck)
    base_set = set(baseline_cards)
    intersection = gen_set.intersection(base_set)
    count = len(intersection)
    print(f"\n--- Similarity Stats ---")
    print(f"Matches found: {count}")
    if count > 0:
        sample = list(intersection)[:10]
        print(f"Matching Cards (Sample): {sample}")
    return count, list(intersection)

# --- Verification ---
if __name__ == '__main__':
    baseline = get_edhrec_baseline("Atraxa, Praetors' Voice")
    if baseline:
        print(f"First 10 baseline cards: {baseline[:10]}")

Fetching EDHRec data for slug: atraxa-praetors-voice...
Successfully fetched 288 unique baseline cards.
First 10 baseline cards: ['Jace, the Mind Sculptor', 'Contagion Engine', 'Mana Confluence', 'Opulent Palace', 'Undergrowth Stadium', 'Astral Cornucopia', 'Staff of Compleation', 'Swiftfoot Boots', 'Lightning Greaves', 'Arcane Denial']


In [ ]:
import numpy as np
import random
from sklearn.metrics.pairwise import cosine_similarity

def calculate_deck_optimization(deck, cmd_idx, embs, df, weights=None):
    """
    Calculates a multi-factor optimization score for a 99-card deck.
    Now accepts a 'weights' dictionary for dynamic scoring.
    """
    # Default weights if none provided
    if weights is None:
        weights = {'alignment': 0.95, 'cohesion': 0.0, 'structure': 0.05}

    # 1. EXTRACT DATA
    # The function expects deck indices and commander index to be valid for embs and df
    # The error handling for out-of-bounds is usually done upstream during index generation.

    deck_vecs = embs[deck]
    cmd_vec = embs[cmd_idx].reshape(1, -1)

    # 2. COMMANDER ALIGNMENT
    # FIXED: cmd_vec was defined above, but comm_vec was used here. Fixed typo.
    comm_sims = cosine_similarity(cmd_vec, deck_vecs).flatten()
    alignment_score = np.mean(comm_sims)

    # 3. INTRA-DECK COHESION
    cohesion_score = 0.0
    if len(deck_vecs) > 1:
        inner_sim_matrix = cosine_similarity(deck_vecs)
        triu_indices = np.triu_indices(len(deck_vecs), k=1)
        if triu_indices[0].size > 0:
            cohesion_score = np.mean(inner_sim_matrix[triu_indices])
        else:
            cohesion_score = 0.0

    # 4. ROLE SATISFACTION
    # Target: 10 Ramp, 10 Draw, 10 Interaction (Spot Removal + Wipes)
    roles_present = df.iloc[deck]

    def get_role_sum(col_name):
        return roles_present[col_name].sum() if col_name in roles_present.columns else 0

    ramp_count = get_role_sum('is_Ramp')
    draw_count = get_role_sum('is_Card_Draw')

    # Interaction covers Spot Removal and Board Wipes
    removal_count = get_role_sum('is_Spot_Removal')
    wipe_count = get_role_sum('is_Board_Wipe')
    interaction_count = removal_count + wipe_count

    # Score Saturation up to 12 cards, penalize over 12
    if ramp_count <= 12:
        ramp_score = min(ramp_count / 12.0, 1.0) # Linearly increases up to 1.0 at 12
    else:
        # Penalize if ramp_count exceeds 12. Score drops from 1.0.
        # Max penalty if count reaches 24 (12 + 12 = 24), score becomes 0.0
        ramp_score = max(0.0, 1.0 - (ramp_count - 12) / 12.0)

    if draw_count <= 12:
        draw_score = min(draw_count / 12.0, 1.0) # Linearly increases up to 1.0 at 12
    else:
        # Penalize if draw_count exceeds 12. Score drops from 1.0.
        draw_score = max(0.0, 1.0 - (draw_count - 12) / 12.0)

    # Interaction score remains capped at 1.0 at 10, as no specific penalty was requested for it.
    interaction_score = min(interaction_count / 10.0, 1.0)

    structure_score = (ramp_score + draw_score + interaction_score) / 3.0

    # 5. FINAL OPTIMIZATION SCORE using dynamic weights
    final_score = (
        (alignment_score * weights.get('alignment', 0.85)) +
        (cohesion_score * weights.get('cohesion', 0.10)) +
        (structure_score * weights.get('structure', 0.05))
    )

    return final_score

def run_genetic_algorithm(commander_name, prompt, df, embeddings, model,
                          generations=30, population_size=1000, mutation_rate=0.07,
                          deck_size=61, weights=None):
    """
    Main loop for the EDH Deck Generation Genetic Algorithm.
    Now accepts 'weights' to pass down to the fitness function.
    """
    if weights is None:
        weights = {'alignment': 0.85, 'cohesion': 0.10, 'structure': 0.05}

    print(f"--- Starting GA for: {commander_name} ---")
    print(f"Optimization Weights: {weights}")

    # Setup Commander
    df['name'] = df['name'].astype(str)
    matches = df[df['name'] == commander_name]
    if matches.empty:
        prefix = commander_name + " //"
        matches = df[df['name'].str.startswith(prefix)]
    if matches.empty:
        matches = df[df['name'].str.contains(commander_name, regex=False)]
    if matches.empty:
        raise ValueError(f"Commander not found.")

    cmd_idx = matches.index[0]
    cid = get_commander_color_identity(df.iloc[cmd_idx]['name'], df)
    legal_df = filter_by_color_identity(cid, df)

    pool = get_candidate_pool(prompt, legal_df, embeddings, model, top_k=1000)
    if len(pool) < deck_size: raise ValueError("Pool too small.")

    pop = [create_individual(pool, deck_size) for _ in range(population_size)]

    best_score = -1
    best_deck = None

    for g in range(generations):
        # Pass weights to fitness function
        scores = [calculate_deck_optimization(d, cmd_idx, embeddings, df, weights=weights) for d in pop]

        curr_best_idx = np.argmax(scores)
        if scores[curr_best_idx] > best_score:
            best_score = scores[curr_best_idx]
            best_deck = pop[curr_best_idx]

        if g % 10 == 0: print(f"Gen {g}: {best_score:.4f}")

        elites = selection(pop, scores, max(2, int(population_size*0.2)))
        next_gen = elites[:]
        while len(next_gen) < population_size:
            p1, p2 = random.sample(elites, 2)
            child = mutate(crossover(p1, p2, deck_size), pool, mutation_rate)
            next_gen.append(child)
        pop = next_gen

    print(f"Final Score: {best_score:.4f}")
    return df.iloc[best_deck]['name'].tolist()

# --- Test Block for Dynamic Weights ---
if __name__ == '__main__':
    print("\n--- Testing Dynamic Weights ---")
    try:
        # Check if df, embeddings, and model are already loaded from previous cells
        # These variables are expected to be global from cell 8331e8a4.
        if 'df' not in globals() or 'embeddings' not in globals() or 'model' not in globals():
            print("WARNING: Required data (df, embeddings, model) not found in global scope.")
            print("Please ensure the data loading and model initialization cells (e.g., cell 8331e8a4) are executed before running this test.")
            raise NameError("Required data (df, embeddings, model) are not defined.")

        # Ensure df is not empty for a valid test
        if df.empty:
            raise ValueError("DataFrame 'df' is empty. Cannot perform dummy test.")

        # Dummy setup - using the first card in the dataframe as commander for testing purposes
        dummy_cmd_idx = 0
        dummy_deck = list(range(1, min(62, len(df)))) # Create a dummy deck with valid indices
        if len(dummy_deck) < 1:
            raise ValueError("DataFrame 'df' too small to create dummy_deck for test.")

        # Standard weights
        w_std = {'alignment': 0.85, 'cohesion': 0.10, 'structure': 0.05}
        score_std = calculate_deck_optimization(dummy_deck, dummy_cmd_idx, embeddings, df, weights=w_std)
        print(f"Standard Weights Score: {score_std:.4f}")

        # Alignment focus
        w_align = {'alignment': 1.0, 'cohesion': 0.0, 'structure': 0.0}
        score_align = calculate_deck_optimization(dummy_deck, dummy_cmd_idx, embeddings, df, weights=w_align)
        print(f"Alignment-Only Score: {score_align:.4f}")

        # Structure focus
        w_struct = {'alignment': 0.0, 'cohesion': 0.0, 'structure': 1.0}
        score_struct = calculate_deck_optimization(dummy_deck, dummy_cmd_idx, embeddings, df, weights=w_struct)
        print(f"Structure-Only Score: {score_struct:.4f}")

        if score_std != score_align:
             print("SUCCESS: Weights correctly influence score.")
        else:
             print("WARNING: Scores identical, check logic.")

    except NameError as ne:
        print(f"Test skipped due to missing data: {ne}")
    except Exception as e:
        print(f"Test failed: {e}")


--- Testing Dynamic Weights ---
Test failed: DataFrame 'df' is empty. Cannot perform dummy test.


A function to test the Genetic Algorithm multiple times with different fitness weight profiles. The purpose is to easily check which weight profile produces the most desirable decklist.


In [ ]:
import pandas as pd
import numpy as np

# 1. Define Weight Profiles
WEIGHT_PROFILES = {
    'Balanced': {'alignment': 0.4, 'cohesion': 0.3, 'structure': 0.3},
    'Commander Focused': {'alignment': 0.8, 'cohesion': 0.1, 'structure': 0.1},
    'Structure Focused': {'alignment': 0.1, 'cohesion': 0.1, 'structure': 0.8}
}

# 2. Experiment Function
def run_weight_experiment(commander_name, prompt, df, embeddings, model):
    """
    Runs the Genetic Algorithm multiple times using different fitness weight profiles.
    Returns a dictionary of results: {profile_name: [list_of_card_names]}.
    """
    results = {}

    print(f"\n=== Running Weight Experiment for {commander_name} ===")

    for profile_name, weights in WEIGHT_PROFILES.items():
        print(f"\n>> Running Profile: {profile_name}")
        print(f"Weights: {weights}")

        try:
            # Run GA with specific weights
            # Using smaller gen/pop size for speed in experiment
            deck_names = run_genetic_algorithm(
                commander_name=commander_name,
                prompt=prompt,
                df=df,
                embeddings=embeddings,
                model=model,
                generations=20,
                population_size=30,
                deck_size=61,
                weights=weights
            )

            results[profile_name] = deck_names
            print(f"Completed {profile_name}. Deck size: {len(deck_names)}")

        except Exception as e:
            print(f"Error running profile {profile_name}: {e}")
            results[profile_name] = []

    return results

# 3. Test Block
if __name__ == "__main__":
    # Ensure data is available from previous steps
    if 'df' in locals() and not df.empty:
        try:
            test_commander = "Toralf, God of Fury"
            test_prompt = "Big damage spells and red chaos"

            experiment_results = run_weight_experiment(
                test_commander,
                test_prompt,
                df,
                embeddings,
                model
            )

            print(f"\nExperiment Complete. Profiles generated: {list(experiment_results.keys())}")
        except Exception as e:
            print(f"Experiment Test Failed: {e}")
    else:
        print("Dataframes not loaded. Skipping test.")

Dataframes not loaded. Skipping test.


In [ ]:
#Synergy check
#Compares vectors to EDHRec synergy scores
!pip install pyedhrec
import pandas as pd
import os
from sentence_transformers import SentenceTransformer, util
from pyedhrec import EDHRec
from google.colab import drive
import re
import time

# 1. Setup
print("Mounting Google Drive...")
drive.mount('/content/drive')

# Define path to the file in Drive
# Assumes the file is in 'MTGCISE' folder in MyDrive
# Correcting the file name to the one generated by the feature engineering step
csv_path = os.path.join('/content/drive/MyDrive/MTGCISE', MODEL_READY_CSV)

if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print(f"Successfully loaded data from {csv_path}")
else:
    print(f"File not found at {csv_path}. Please check the path.")
    df = pd.DataFrame() # Create empty DF to prevent subsequent errors

edh = EDHRec()
model = SentenceTransformer(EMBEDDING_MODEL_NAME) # Start with this one

def clean_card_name(name):
    """
    Cleans card names by removing extra whitespace or specific set codes if present.
    """
    if not name:
        return ""
    # Remove text in parentheses (e.g., set codes or clarification)
    name = re.sub(r'\s*\([^)]*\)', '', name).strip()
    return name

def validate_model_synergy(commander_name):
    if df.empty:
        print("Dataframe is empty. Cannot run validation.")
        return 0, 0

    print(f"Validating against: {commander_name}")

    # 2. Get high-synergy cards from EDHRec
    try:
        # Convert to slug (e.g., "Atraxa, Praetors' Voice" -> "atraxa-praetors-voice")
        slug = commander_name.lower().replace("'", "").replace(",", "").replace(" ", "-")
        # Remove double slashes if any (for MDFCs)
        slug = slug.split("//")[0].strip("-")

        print(f"Fetching EDHRec data for slug: {slug}...")
        time.sleep(0.5) # Add a small delay to prevent rate limiting
        data = edh.get_commander_cards(slug)

        synergy_names = set()
        # Target headers that usually contain relevant cards for a baseline
        target_headers = [
            'High Synergy Cards', 'Top Cards', 'New Cards', # Primary sections
            'Creatures', 'Instants', 'Sorceries', 'Artifacts', 'Enchantments', 'Planeswalkers', 'Lands', # Card type sections
            'Utility Artifacts', 'Utility Lands', 'Mana Artifacts'
        ]

        if isinstance(data, dict):
            for header in target_headers:
                if header in data:
                    content = data[header]

                    # Helper to process a list of card objects
                    def process_card_list(clist):
                        for card in clist:
                            if isinstance(card, dict):
                                cname = clean_card_name(card.get('name'))
                                if cname:
                                    synergy_names.add(cname)

                    if isinstance(content, list): # e.g., 'High Synergy Cards': [{'name': 'Card1'}, ...]
                        process_card_list(content)
                    elif isinstance(content, dict): # e.g., 'Creatures': {'cardviews': [{'name': 'Card1'}, ...]}
                        if 'cardviews' in content:
                            process_card_list(content['cardviews'])
                        elif 'cards' in content:
                            process_card_list(content['cards'])
        elif isinstance(data, list):
             for cl in data:
                if cl.get('header') in target_headers:
                    for card in cl.get('cardviews', []):
                        cname = clean_card_name(card.get('name'))
                        if cname:
                            synergy_names.add(cname)

        synergy_names = list(synergy_names)
        if not synergy_names:
            print("No cards found in EDHRec response.")
            return 0, 0

        print(f"Retrieved {len(synergy_names)} synergy candidates from EDHRec.")

    except Exception as e:
        print(f"Error fetching EDHRec data: {e}")
        import traceback
        traceback.print_exc()
        return 0, 0

    # 3. Create Embeddings
    # Get the text for the commander and the synergy cards
    # Use 'oracle_text_clean' if available, as per data prep steps
    text_col = 'oracle_text_clean' if 'oracle_text_clean' in df.columns else 'oracle_text'

    cmd_row = df[df['name'] == commander_name]
    if cmd_row.empty:
        # Try finding partial match if exact match fails
        cmd_row = df[df['name'].str.contains(commander_name, regex=False)]

    if cmd_row.empty:
        print(f"Commander '{commander_name}' not found in local CSV.")
        return 0, 0

    commander_text = cmd_row[text_col].iloc[0]

    # Filter for synergy cards present in our dataset
    synergy_df = df[df['name'].isin(synergy_names)]

    if synergy_df.empty:
        print("None of the synergy cards from EDHRec were found in the local dataset.")
        return 0, 0

    synergy_texts = synergy_df[text_col].tolist()

    # Baseline comparison (Random cards)
    random_texts = df.sample(len(synergy_texts))[text_col].tolist()

    cmd_vec = model.encode(commander_text, convert_to_tensor=True)
    syn_vecs = model.encode(synergy_texts, convert_to_tensor=True)
    rnd_vecs = model.encode(random_texts, convert_to_tensor=True)

    # 4. Calculate Scores
    avg_synergy_sim = util.cos_sim(cmd_vec, syn_vecs).mean().item()
    avg_random_sim = util.cos_sim(cmd_vec, rnd_vecs).mean().item()

    return avg_synergy_sim, avg_random_sim

# Run the test
if not df.empty:
    syn_score, rnd_score = validate_model_synergy("Atraxa, Praetors' Voice")

    print(f"\nResults:")
    print(f"Average Similarity (High Synergy): {syn_score:.4f}")
    print(f"Average Similarity (Random Cards): {rnd_score:.4f}")
    print(f"Model Accuracy Delta: {syn_score - rnd_score:.4f}")

Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


KeyboardInterrupt: 

In [ ]:
# Refactored: Redundant `calculate_deck_optimization` definition removed.
# The canonical definition (with dynamic weights) is defined in the Genetic Algorithm section.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/MTGCISE
!git config --global user.email "necrofantasia@icloud.com"
!git config --global user.name "Lenfried"

## Greedy Baseline Function

Creates a deck by selecting the top 61 cards most similar to the prompt and then calculates a land base.


In [ ]:
def run_greedy_baseline(commander_name, prompt, df, embeddings, model):
    """
    Generates a baseline deck by greedily selecting the top 61 most similar cards to the prompt,
    WHILE prioritizing filling 10 Ramp, 10 Draw, and 10 Interaction slots first.
    """
    print(f"--- Running Structured Greedy Baseline for: {commander_name} ---")

    # 1. Identify Commander and Color Identity
    try:
        cid = get_commander_color_identity(commander_name, df)
    except ValueError as e:
        print(e)
        return []

    print(f"Commander Color Identity: {cid}")

    # 2. Filter for Legal Cards
    legal_df = filter_by_color_identity(cid, df)

    # 3. Get Candidate Pool based on Prompt Similarity (Top 1000 to have options)
    # We need a larger pool to find specific roles
    top_indices = get_candidate_pool(prompt, legal_df, embeddings, model, top_k=2000)
    pool_df = df.iloc[top_indices].copy()

    # 4. Fill Functional Roles
    deck_indices = set()

    def select_best_for_role(role_col, count_needed, current_deck_indices):
        """Selects top cards for a role from the pool that aren't already in deck."""
        # Filter for cards that match the role and aren't in deck
        available = pool_df[
            (pool_df[role_col] == 1) &
            (~pool_df.index.isin(current_deck_indices))
        ]

        # Since pool_df is already sorted by similarity (from get_candidate_pool),
        # taking the top N gives us the most similar cards for that role.
        selected = available.head(count_needed).index.tolist()
        return selected

    # 4.1 Select Ramp (Target 10)
    ramp_ids = select_best_for_role('is_Ramp', 10, deck_indices)
    deck_indices.update(ramp_ids)

    # 4.2 Select Draw (Target 10)
    draw_ids = select_best_for_role('is_Card_Draw', 10, deck_indices)
    deck_indices.update(draw_ids)

    # 4.3 Select Interaction (Target 10 total: Spot Removal + Board Wipes)
    # We create a temporary column for interaction check
    # Ensure these columns are integer type before bitwise OR
    pool_df['is_Interaction'] = pool_df['is_Spot_Removal'].astype(int) | pool_df['is_Board_Wipe'].astype(int)
    interaction_ids = select_best_for_role('is_Interaction', 10, deck_indices)
    deck_indices.update(interaction_ids)

    print(f"Role Filling Stats: {len(ramp_ids)} Ramp, {len(draw_ids)} Draw, {len(interaction_ids)} Interaction")

    # 5. Fill Remainder to reach 61 Spells
    target_size = 61
    needed = target_size - len(deck_indices)

    if needed > 0:
        # Select top cards from pool that aren't in deck yet
        remaining = pool_df[~pool_df.index.isin(deck_indices)].head(needed).index.tolist()
        deck_indices.update(remaining)

    # Convert indices back to names
    # Ensure we preserve order of addition for "best fits" mostly, or sort by original similarity?
    # Let's keep them as a list
    final_indices = list(deck_indices)

    # Retrieve names
    spell_names = df.iloc[final_indices]['name'].tolist()

    # 6. Generate Land Base
    lands = calculate_land_distribution(spell_names, df, cid, total_lands=38)

    final_deck = spell_names + lands

    print(f"Greedy Baseline Generated: {len(final_deck)} cards")
    return final_deck

In [ ]:
commander_name = "Averna, the Chaos Bloom"
prompt = "cascade"

greedy_decklist = run_greedy_baseline(commander_name, prompt, df, embeddings, model)

print(f"\n--- Greedy Baseline Decklist for {commander_name} ({len(greedy_decklist)} cards) ---")
for i, card in enumerate(greedy_decklist):
    print(f"{i+1}. {card}")

##Compares Greedy and Genetic Algorithms

In [ ]:
def run_comparative_experiment(commander_name, prompt, df, embeddings, model, weights=None):
    """
    Runs both the Greedy Baseline and the Genetic Algorithm for a given commander/prompt.
    Compares them on: Fitness Score, Functional Categories, and EDHRec Overlap.
    """
    # 1. Define GA Parameters
    GA_PARAMS = {
        'generations': 50,
        'population_size': 1000,
        'mutation_rate': 0.05,
        'deck_size': 61
    }

    print(f"\n{'='*60}")
    print(f" COMPARATIVE EXPERIMENT: {commander_name}")
    print(f"{'='*60}")
    print(f"GA PARAMETERS:")
    for param, value in GA_PARAMS.items():
        print(f"  - {param}: {value}")
    print(f"{'='*60}")

    # 2. Run Greedy Baseline
    print("\n>> 1. Running Greedy Baseline...")
    greedy_deck_names = run_greedy_baseline(commander_name, prompt, df, embeddings, model)

    # 3. Run Genetic Algorithm
    print("\n>> 2. Running Genetic Algorithm...")
    ga_spell_names = run_genetic_algorithm(
        commander_name,
        prompt,
        df,
        embeddings,
        model,
        generations=GA_PARAMS['generations'],
        population_size=GA_PARAMS['population_size'],
        mutation_rate=GA_PARAMS['mutation_rate'],
        deck_size=GA_PARAMS['deck_size'],
        weights=weights
    )

    # Add lands to GA deck for full comparison
    try:
        cid = get_commander_color_identity(commander_name, df)
        ga_lands = calculate_land_distribution(ga_spell_names, df, cid, total_lands=38)
        ga_full_deck = ga_spell_names + ga_lands
    except Exception as e:
        print(f"Error calculating lands for GA deck: {e}")
        ga_full_deck = ga_spell_names

    # 4. Get EDHRec Baseline for Overlap Check
    print("\n>> 3. Fetching EDHRec Baseline...")
    edhrec_names = get_edhrec_baseline(commander_name)
    edhrec_set = set(edhrec_names)

    # 5. Helper to Calculate Metrics
    def get_deck_metrics(deck_list, commander_name, df, embeddings, weights=None):
        deck_df = df[df['name'].isin(deck_list)]
        valid_indices = deck_df.index.tolist()

        cmd_rows = df[df['name'] == commander_name]
        if cmd_rows.empty:
             cmd_rows = df[df['name'].str.startswith(commander_name + " //")]

        if not cmd_rows.empty and not deck_df.empty:
            cmd_idx = cmd_rows.index[0]
            try:
                fitness = calculate_deck_optimization(valid_indices, cmd_idx, embeddings, df, weights=weights)
            except Exception as e:
                print(f"Error calculating fitness: {e}")
                fitness = 0.0
        else:
            fitness = 0.0

        ramp = deck_df['is_Ramp'].sum() if 'is_Ramp' in deck_df.columns else 0
        draw = deck_df['is_Card_Draw'].sum() if 'is_Card_Draw' in deck_df.columns else 0
        interaction = (
            (deck_df['is_Spot_Removal'].sum() if 'is_Spot_Removal' in deck_df.columns else 0) +
            (deck_df['is_Board_Wipe'].sum() if 'is_Board_Wipe' in deck_df.columns else 0)
        )

        overlap = len(set(deck_list).intersection(edhrec_set))

        return {
            'Fitness': fitness,
            'Ramp': ramp,
            'Draw': draw,
            'Interaction': interaction,
            'EDHRec_Overlap': overlap
        }

    # 6. Calculate Metrics
    greedy_metrics = get_deck_metrics(greedy_deck_names, commander_name, df, embeddings, weights)
    ga_metrics = get_deck_metrics(ga_full_deck, commander_name, df, embeddings, weights)

    # 7. Print Summary
    print(f"\n{'='*60}")
    print(f" COMPARISON RESULTS: {commander_name}")
    print(f"{'='*60}")
    print(f"{'Metric':<20} | {'Greedy Baseline':<15} | {'Genetic Algorithm':<15}")
    print(f"{' -'*30}")

    for key in ['Fitness', 'Ramp', 'Draw', 'Interaction', 'EDHRec_Overlap']:
        g_val = greedy_metrics[key]
        ga_val = ga_metrics[key]

        if isinstance(g_val, float):
            g_str = f"{g_val:.4f}"
            ga_str = f"{ga_val:.4f}"
        else:
            g_str = str(g_val)
            ga_str = str(ga_val)

        print(f"{key:<20} | {g_str:<15} | {ga_str:<15}")
    print(f"{'='*60}\n")

    return {'Greedy': greedy_metrics, 'GA': ga_metrics}

In [ ]:
# Enter your commander name and prompt here
commander_name_input = "Averna, the Chaos Bloom"  # @param {type:"string"}
prompt_input = "cascade"  # @param {type:"string"}

# Run the comparative experiment with the specified inputs
results = run_comparative_experiment(
    commander_name=commander_name_input,
    prompt=prompt_input,
    df=df,
    embeddings=embeddings,
    model=model
)

##Weight profile testing for optimization score function
1.  Accepts a commander name and a prompt.
2.  Fetches the "average deck" list from EDHRec using `get_edhrec_baseline`.
3.  Defines a list of weight configurations to test (e.g., `Balanced`: {'alignment': 0.4, 'cohesion': 0.3, 'structure': 0.3}, `Alignment Focus`: {'alignment': 0.8, 'cohesion': 0.1, 'structure': 0.1}, etc.).
4.  Iterates through each weight profile, running `run_genetic_algorithm` to generate a deck for each.
5.  Compares each generated deck against the EDHRec average deck using set intersection to calculate an overlap score.
6.  Returns the results sorted by the highest overlap.

Call the `optimize_weights_against_average_deck` function with the specified commander and prompt to demonstrate the optimization process and see which weight profile performs best against the EDHRec baseline.

Replace `commander_name` and `prompt` with any commander and prompt.


In [ ]:
!pip install pyedhrec
import pandas as pd
import numpy as np
import os
from sentence_transformers import SentenceTransformer
from pyedhrec import EDHRec
import re
import time

# --- Ensure Data is Loaded ---
if 'df' not in locals():
    print("Variables not found in local scope. Attempting to load...")
    DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
    CSV_PATH = os.path.join(DRIVE_PATH, MODEL_READY_CSV)
    NPY_PATH = os.path.join(DRIVE_PATH, EMBEDDINGS_NPY)

    if os.path.exists(CSV_PATH) and os.path.exists(NPY_PATH):
        try:
            df = pd.read_csv(CSV_PATH)
            embeddings = np.load(NPY_PATH)
            model = SentenceTransformer(EMBEDDING_MODEL_NAME)
            print("Data loaded successfully.")
        except Exception as e:
            print(f"Error loading data: {e}")
            df = pd.DataFrame()
            embeddings = np.array([])
            model = None
    else:
        print("Data files not found. Please ensure Google Drive is mounted and paths are correct.")
        df = pd.DataFrame()
        embeddings = np.array([])
        model = None

# --- Helper Functions (EDHRec) ---
def clean_card_name(name):
    """
    Cleans card names by removing extra whitespace or specific set codes if present.
    """
    if not name:
        return ""
    # Remove text in parentheses (e.g., set codes or clarification)
    name = re.sub(r'\s*\([^)]*\)', '', name).strip()
    return name

def get_edhrec_baseline(commander_name):
    """
    Fetches a baseline list of cards (High Synergy, Top Cards, etc.) for a commander from EDHRec.
    """
    edh = EDHRec()

    # Generate Slug
    name_for_slug = commander_name.split(' // ')[0]
    name_for_slug = name_for_slug.replace("'", "").replace(",", "")
    slug = name_for_slug.lower().replace(' ', '-')
    slug = re.sub(r'-+', '-', slug)

    print(f"Fetching EDHRec data for slug: {slug}...")

    try:
        time.sleep(0.5)
        data = edh.get_commander_cards(slug)

        baseline_cards = set()
        target_headers = [
            'High Synergy Cards', 'Top Cards', 'New Cards',
            'Creatures', 'Instants', 'Sorceries', 'Artifacts', 'Enchantments', 'Planeswalkers', 'Lands',
            'Utility Artifacts', 'Utility Lands', 'Mana Artifacts'
        ]

        if isinstance(data, dict):
            for header in target_headers:
                if header in data:
                    content = data[header]
                    def process_card_list(clist):
                        for card in clist:
                            if isinstance(card, dict):
                                cname = clean_card_name(card.get('name'))
                                if cname:
                                    baseline_cards.add(cname)
                    if isinstance(content, list):
                        process_card_list(content)
                    elif isinstance(content, dict):
                        if 'cardviews' in content:
                            process_card_list(content['cardviews'])
                        elif 'cards' in content:
                            process_card_list(content['cards'])
        elif isinstance(data, list):
             for cl in data:
                if cl.get('header') in target_headers:
                    for card in cl.get('cardviews', []):
                        cname = clean_card_name(card.get('name'))
                        if cname:
                            baseline_cards.add(cname)

        unique_cards = list(baseline_cards)
        print(f"Successfully fetched {len(unique_cards)} unique baseline cards.")
        return unique_cards

    except Exception as e:
        print(f"Error fetching/parsing EDHRec data: {e}")
        return []

# --- Optimization Function ---
def optimize_weights_against_average_deck(commander_name, prompt, df, embeddings, model):
    """
    Tests multiple GA weight profiles against an EDHRec average deck baseline.
    Returns a sorted list of results based on overlap count.
    """
    if df.empty or model is None:
        print("Dataframe is empty or model is missing. Cannot run optimization.")
        return []

    print(f"\n{'='*60}")
    print(f" WEIGHT OPTIMIZATION: {commander_name}")
    print(f"{'='*60}")

    # 1. Get EDHRec Baseline
    print("\n>> Fetching EDHRec Baseline...")
    edhrec_names = get_edhrec_baseline(commander_name)
    if not edhrec_names:
        print("Failed to fetch baseline. Aborting.")
        return []
    edhrec_set = set(edhrec_names)
    print(f"Baseline size: {len(edhrec_set)} cards")

    # 2. Define Weight Profiles
    weight_profiles = {
        'Balanced': {'alignment': 0.4, 'cohesion': 0.3, 'structure': 0.3},
        'Balanced 2': {'alignment': 0.5, 'cohesion': 0.3, 'structure': 0.2},
        'Balanced 3': {'alignment': 0.3, 'cohesion': 0.4, 'structure': 0.3},
        'Alignment Focus': {'alignment': 0.8, 'cohesion': 0.1, 'structure': 0.1},
        'Alignment 2': {'alignment': 0.6, 'cohesion': 0.2, 'structure': 0.2},
        'Alignment 3': {'alignment': 0.7, 'cohesion': 0.2, 'structure': 0.1},
        'Alignment 4': {'alignment': 0.9, 'cohesion': 0.05, 'structure': 0.05},
        'Alignment 5:': {'alignment': 0.95, 'cohesion': 0.05, 'structure': 0.0},
        'Alignment 6': {'alignment': 0.8, 'cohesion': 0.1, 'structure': 0.1},
        'Alignment 7': {'alignment': 0.85, 'cohesion': 0.1, 'structure': 0.05},
        'Structure Focus': {'alignment': 0.2, 'cohesion': 0.2, 'structure': 0.6},
        'Structure 2': {'alignment': 0.2, 'cohesion': 0.1, 'structure': 0.7},
        'Structure 3': {'alignment': 0.3, 'cohesion': 0.2, 'structure': 0.5}

    }

    results = []

    # 3. Iterate and Test
    for profile_name, weights in weight_profiles.items():
        print(f"\n>> Testing Profile: {profile_name}")
        print(f"Weights: {weights}")

        try:
            ga_deck_names = run_genetic_algorithm(
                commander_name, prompt, df, embeddings, model,
                generations=50, population_size=1000, deck_size=61, weights=weights
            )

            intersection = set(ga_deck_names).intersection(edhrec_set)
            overlap_count = len(intersection)

            results.append({
                'profile': profile_name,
                'weights': weights,
                'overlap': overlap_count,
                'deck': ga_deck_names
            })
            print(f"Overlap with EDHRec: {overlap_count}")

        except Exception as e:
            print(f"Error testing profile {profile_name}: {e}")

    # 4. Sort and Summary
    results.sort(key=lambda x: x['overlap'], reverse=True)

    print(f"\n{'='*60}")
    print(f" OPTIMIZATION RESULTS SUMMARY")
    print(f"{'='*60}")
    print(f"{'Profile Name':<20} | {'Overlap Count':<15}")
    print(f"{'-'*40}")

    for res in results:
        print(f"{res['profile']:<20} | {res['overlap']:<15}")
    print(f"{'='*60}\n")

    return results

if __name__ == "__main__":
    # Run the optimization test
    optimization_results = optimize_weights_against_average_deck(
        commander_name="Krenko, Mob Boss",
        prompt="Cards with the Goblin subtype and abilities that create Goblin creature tokens or provide stat buffs to Goblins",
        df=df,
        embeddings=embeddings,
        model=model
    )


In [ ]:
#Prints the decklist of any profile for evaluation
for result in optimization_results:
    #Replace with the name of the profile desired
    if result['profile'] == 'Alignment 7':
        print(f"Deck generated by '{result['profile']}' profile (Overlap: {result['overlap']} cards):")
        for i, card_name in enumerate(result['deck'], 1):
            print(f"{i}. {card_name}")
        break


# Part 2: Metric Comparison
## EDHRec Average Deck and fitness score calculation

In [ ]:
!pip install pyedhrec
import os
import re
import time
import random
import collections
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from pyedhrec import EDHRec
from google.colab import drive

# Mount drive if not already mounted
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
CSV_PATH = os.path.join(DRIVE_PATH, MODEL_READY_CSV)
NPY_PATH = os.path.join(DRIVE_PATH, EMBEDDINGS_NPY)

# Initialize resources if they are missing from the environment
if 'df' not in locals() or df.empty or 'model' not in locals() or model is None:
    print("Initializing resources for Part 2...")
    if os.path.exists(CSV_PATH) and os.path.exists(NPY_PATH):
        df = pd.read_csv(CSV_PATH)
        embeddings = np.load(NPY_PATH)
        model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        print("Resources and model loaded successfully.")
    else:
        print("Data files not found. Ensure Google Drive is mounted and paths are correct.")
else:
    print("Resources already loaded.")


Initializing resources for Part 2...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Resources and model loaded successfully.


In [ ]:
import re
import time
import requests
import numpy as np
from pyedhrec import EDHRec

# One shared client for the whole notebook. pyedhrec downloads EDHRec's homepage to
# look up the site's build id the first time a client is used, so making a new
# EDHRec() on every call doubles the traffic -- that burst is what got the
# "Connection reset by peer" errors.
EDH_CLIENT = EDHRec()

# Every card EDHRec files under "Land" in an average deck fetched this session.
EDHREC_LAND_NAMES = set()


def edhrec_call(method, *args, retries=3, backoff=2.0):
    """Call a pyedhrec method, retrying dropped connections, rate limits and a stale build id."""
    for attempt in range(1, retries + 1):
        try:
            time.sleep(0.5)
            return method(*args)
        except requests.exceptions.RequestException as e:
            if attempt == retries:
                raise
            print(f"  EDHRec request failed ({type(e).__name__}); retry {attempt}/{retries - 1}...")
            EDH_CLIENT.current_build_id = None   # make pyedhrec look the build id up again
            time.sleep(backoff * attempt)


def _clean_card_name(raw):
    name = re.sub(r'^[\s ]*\d+\s+', '', str(raw))   # drop a leading quantity: "1 Sol Ring"
    name = re.sub(r'\s*\([^)]*\)', '', name)             # drop set tags like "(M21)"
    return name.strip()


def _name_and_qty(card):
    if isinstance(card, (list, tuple)):
        return card[0], (card[1] if len(card) > 1 else 1)
    if isinstance(card, dict):
        return card.get('name', ''), card.get('quantity', 1)
    return card, 1


def parse_edhrec_decklist(decklist):
    """
    Turns EDHRec's average-deck payload into [(name, qty, category), ...].

    EDHRec now sends a dict:
        {"commander": [...], "commander_v2": [[name, qty]],
         "cards": {"Creature": [[name, qty], ...], "Land": [...], ...}}
    It used to send a flat list: ["1 Krenko, Mob Boss", "1 Sol Ring", "29 Mountain", ...]
    Both are handled. The commander stays in the list, like it was in the old format.
    """
    entries = []
    if isinstance(decklist, dict):
        commanders = decklist.get('commander_v2') or decklist.get('commander') or []
        for card in commanders:
            name, qty = _name_and_qty(card)
            if name:
                entries.append((_clean_card_name(name), int(qty), 'Commander'))
        for category, cards in (decklist.get('cards') or {}).items():
            for card in cards:
                name, qty = _name_and_qty(card)
                if name:
                    entries.append((_clean_card_name(name), int(qty), category))
    elif isinstance(decklist, list):
        for item in decklist:
            raw = item.get('name', '') if isinstance(item, dict) else item
            if isinstance(raw, str) and raw.strip():
                qty = re.match(r'^[\s ]*(\d+)\s+', raw)
                entries.append((_clean_card_name(raw), int(qty.group(1)) if qty else 1, None))
    return entries


def build_name_index(df):
    """Lower-cased card name -> df row label. Exact names win; the front face of a '//' card is a fallback."""
    names = df['name'].astype(str).str.lower().str.strip()
    lookup = {}
    for idx, n in zip(df.index, names):
        lookup.setdefault(n, idx)
    for idx, n in zip(df.index, names):
        if ' // ' in n:
            lookup.setdefault(n.split(' // ')[0], idx)
    return lookup


def lookup_card(name, lookup):
    key = _clean_card_name(name).lower()
    if key in lookup:
        return lookup[key]
    if ' // ' in key:
        return lookup.get(key.split(' // ')[0])
    return None


def get_edhrec_average_and_score(commander_name):
    """
    Fetches the average deck for a commander from EDHRec and scores it with
    calculate_deck_optimization. Returns (score, ["qty Card Name", ...]).
    """
    # 1. Generate slug
    name_for_slug = commander_name.split(' // ')[0]
    slug = name_for_slug.lower().replace("'", "").replace(",", "").replace(" ", "-")
    slug = re.sub(r'-+', '-', slug)
    print(f"Fetching EDHRec data for: {commander_name} (slug: {slug})")

    # 2. Fetch and parse
    try:
        data = edhrec_call(EDH_CLIENT.get_commanders_average_deck, slug)
    except Exception as e:
        print(f"Error fetching average deck for {commander_name}: {e}")
        return 0.0, []

    entries = parse_edhrec_decklist((data or {}).get('decklist'))
    if not entries:
        print(f"Warning: EDHRec returned no readable cards for {commander_name}.")
        return 0.0, []

    baseline_data = {}   # lower-cased name -> "qty Name"
    for name, qty, category in entries:
        baseline_data[name.lower()] = f"{qty} {name}"
        if category == 'Land':
            EDHREC_LAND_NAMES.add(name.lower())

    # 3. Map to local DataFrame rows
    df['name_clean'] = df['name'].str.lower().str.strip()   # kept for other cells that use it
    lookup = build_name_index(df)
    matched, unmatched = set(), []
    for key, line in baseline_data.items():
        idx = lookup_card(key, lookup)
        if idx is None:
            unmatched.append(line)
        else:
            matched.add(int(idx))
    deck_indices = sorted(matched)

    # 4. Commander row
    commander_idx = lookup_card(commander_name, lookup)
    if commander_idx is None:
        print(f"Warning: Commander '{commander_name}' not found in local dataset.")
        return 0.0, sorted(baseline_data.values())

    # 5. Optimization score
    if not deck_indices:
        print("No cards from the EDHRec average deck were found in the local dataset.")
        return 0.0, sorted(baseline_data.values())

    score = calculate_deck_optimization(deck_indices, int(commander_idx), embeddings, df)
    print(f"Average Deck Optimization Score: {score:.4f}")
    print(f"Matched {len(deck_indices)} of {len(baseline_data)} cards in local dataset.")
    if unmatched:
        print(f"  Not in local dataset: {unmatched[:8]}{' ...' if len(unmatched) > 8 else ''}")

    return score, sorted(baseline_data.values())

# Standard execution for Averna to populate the namespace
test_commander_name = 'Averna, the Chaos Bloom'
if 'calculate_deck_optimization' in globals():
    score, cards = get_edhrec_average_and_score(test_commander_name)
else:
    print("Skipping the Averna test: run cell 19 (calculate_deck_optimization) first.")


Fetching EDHRec data for: Averna, the Chaos Bloom (slug: averna-the-chaos-bloom)


NameError: name 'calculate_deck_optimization' is not defined

##Computes GA and EDHRec similarity
Computes the cosine similarity between decks produced by GA and EDHRec average deck for a given commander

In [ ]:
def compare_ga_vs_edhrec_similarity(commander_name, prompt, df, embeddings, model, ga_params=None):
    """
    Computes the average cosine similarity between the Genetic Algorithm deck
    (including generated lands and the commander) and the EDHRec Average Deck,
    accounting for card quantities (multiples of basics).
    """
    if ga_params is None:
        ga_params = {'generations': 30, 'population_size': 1000, 'deck_size': 61}

    print(f"--- Comparing GA Similarity to EDHRec Average for {commander_name} ---")

    # 1. Run the Genetic Algorithm to get spell names
    ga_spells = run_genetic_algorithm(
        commander_name,
        prompt,
        df,
        embeddings,
        model,
        **ga_params
    )

    # 2. Generate Lands for the GA deck
    cid = get_commander_color_identity(commander_name, df)
    ga_lands = calculate_land_distribution(ga_spells, df, cid, total_lands=38)

    # 3. Combine into full 100-card list (Commander + 99 cards)
    ga_full_deck = [commander_name] + ga_spells + ga_lands

    # 4. Fetch the EDHRec Average Deck
    _, edhrec_avg_raw = get_edhrec_average_and_score(commander_name)

    # Format GA deck
    ga_formatted = [f"1 {name}" for name in ga_full_deck]

    # 5. Extract embeddings
    ga_embs = get_weighted_embeddings(ga_formatted, df, embeddings)
    edhrec_embs = get_weighted_embeddings(edhrec_avg_raw, df, embeddings)

    if len(ga_embs) == 0 or len(edhrec_embs) == 0:
        print("Error: Could not retrieve embeddings for one of the decks.")
        return 0.0

    # 6. Calculate pairwise average cosine similarity
    sim_matrix = cosine_similarity(ga_embs, edhrec_embs)
    avg_sim = np.mean(sim_matrix)

    best_match_score = sim_matrix.max(axis=1).mean()

    ga_to_edhrec = sim_matrix.max(axis=1).mean()
    edhrec_to_ga = sim_matrix.max(axis=0).mean()
    avg_sim_2 = 0.5 * (ga_to_edhrec + edhrec_to_ga)

    print(f"\nGA (100) vs EDHRec Average (100)")
    print(f"Cosine Similarity (Pair-wise): {avg_sim:.4f}")
    print(f"Cosine Similarity (Best-match):{best_match_score:.4f}")
    print(f"Cosine Symmetry Similarity (Best-match):{avg_sim_2:.4f}")
    print(f"(Calculated using {len(ga_embs)} GA card instances and {len(edhrec_embs)} EDHRec card instances)")

    return avg_sim

# --- Execution Wrapper ---
if __name__ == "__main__":
    commander_to_test = "Krenko, Mob Boss"
    prompt_to_test = "Cards with the Goblin subtype and abilities that create Goblin creature tokens or provide stat buffs to Goblins."
    ga_settings = {'generations': 30, 'population_size': 1000, 'deck_size': 61}

    try:
        similarity_score = compare_ga_vs_edhrec_similarity(
            commander_to_test,
            prompt_to_test,
            df,
            embeddings,
            model,
            ga_params=ga_settings
        )
    except Exception as e:
        print(f"Execution failed: {e}")

In [ ]:
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

# --- Missing helper functions defined here to avoid NameErrors ---
def get_commander_color_identity(commander_name, df):
    df['name'] = df['name'].astype(str)
    matches = df[df['name'] == commander_name]
    if matches.empty:
        prefix = commander_name + " //"
        matches = df[df['name'].str.startswith(prefix)]
    if matches.empty:
        matches = df[df['name'].str.contains(commander_name, regex=False)]
    if matches.empty:
        raise ValueError(f"Commander '{commander_name}' not found in dataset.")
    commander_row = matches.iloc[0]
    colors = []
    for color in ['W', 'U', 'B', 'R', 'G']:
        col_name = f'CI_{color}'
        if col_name in df.columns and commander_row[col_name] == 1:
            colors.append(color)
    return ''.join(colors) if colors else 'C'

def filter_by_color_identity(commander_color_id, df):
    import pandas as pd
    allowed_colors = set(commander_color_id)
    all_colors = ['W', 'U', 'B', 'R', 'G']
    forbidden_colors = [c for c in all_colors if c not in allowed_colors]
    mask = pd.Series([True] * len(df), index=df.index)
    for color in forbidden_colors:
        col_name = f'CI_{color}'
        if col_name in df.columns:
            mask = mask & (df[col_name] == 0)
    return df[mask].copy()

def compare_ga_vs_edhrec_nonland_similarity(commander_name, prompt, df, embeddings, model=None, ga_params=None):
    """
    Computes the average cosine similarity between the Genetic Algorithm deck's spells
    (non-land cards) and the EDHRec Average Deck's spells, filtering out PURE lands.
    """
    if model is None:
        print(f"Model not provided. Initializing default '{EMBEDDING_MODEL_NAME}' model...")
        model = SentenceTransformer(EMBEDDING_MODEL_NAME)

    if ga_params is None:
        ga_params = {'generations': 30, 'population_size': 1000, 'deck_size': 61}

    print(f"--- Comparing GA Spells Similarity to EDHRec Average Spells for {commander_name} ---")

    # 1. Run the Genetic Algorithm to get spell names (61 cards)
    ga_spells = run_genetic_algorithm(
        commander_name,
        prompt,
        df,
        embeddings,
        model,
        **ga_params
    )

    # 2. Fetch the EDHRec Average Deck
    _, edhrec_avg_raw = get_edhrec_average_and_score(commander_name)

    # 3. Identify PURE lands (exclude spells that are also lands like MDFCs)
    # We only want to filter cards that are JUST lands.
    spell_types = ['Creature', 'Instant', 'Sorcery', 'Artifact', 'Enchantment', 'Planeswalker']
    all_land_names = set(raw_df[
        (raw_df['type_line'].str.contains('Land', na=False)) &
        (~raw_df['type_line'].str.contains('|'.join(spell_types), na=False))
    ]['name'].tolist()) if 'raw_df' in globals() else set()

    # 4. Helper to extract embeddings for true non-lands only
    def get_true_spell_embeddings(raw_list, df_ref, embs_ref, source_label):
        weighted_embs = []
        found_names = []
        df_temp = df_ref.copy()

        for item in raw_list:
            match_qty = re.match(r'^(\d+)\s+(.*)$', item)
            if match_qty:
                qty = int(match_qty.group(1))
                name = match_qty.group(2).strip()
            else:
                qty = 1
                name = item.strip()

            # Skip only if the card is a PURE Land (no spell types)
            if name in all_land_names:
                continue

            card_match = df_temp[df_temp['name'] == name]
            if not card_match.empty:
                idx = card_match.index[0]
                if idx < len(embs_ref):
                    for _ in range(qty):
                        weighted_embs.append(embs_ref[idx])
                    found_names.append(name)
            else:
                # Avoid printing warnings for obvious basics that aren't in the model_ready CSV
                if name.lower() not in ['plains', 'island', 'swamp', 'mountain', 'forest', 'wastes']:
                    print(f"[Warning] {source_label} card not found in local DB: {name}")

        return np.array(weighted_embs), found_names

    # 5. Process both lists
    ga_formatted = [f"1 {name}" for name in ga_spells]

    ga_embs, ga_final_names = get_true_spell_embeddings(ga_formatted, df, embeddings, "GA")
    edhrec_embs, edhrec_final_names = get_true_spell_embeddings(edhrec_avg_raw, df, embeddings, "EDHRec")

    print(f"\nCounts after filtering PURE lands and verifying DB matches:")
    print(f"- GA spells target: {ga_params['deck_size']} | Actual processed: {len(ga_final_names)}")
    print(f"- EDHRec spells found: {len(edhrec_final_names)}")

    if len(ga_embs) == 0 or len(edhrec_embs) == 0:
        print("Error: Could not retrieve spell embeddings for comparison.")
        return 0.0

    # 6. Calculate similarities
    sim_matrix = cosine_similarity(ga_embs, edhrec_embs)
    avg_sim = np.mean(sim_matrix)

    best_match_score = sim_matrix.max(axis=1).mean()

    # GA → EDHRec best match
    ga_to_edhrec = sim_matrix.max(axis=1).mean()

    # EDHRec → GA best match
    edhrec_to_ga = sim_matrix.max(axis=0).mean()

    # Symmetric similarity
    avg_sim_2 = 0.5 * (ga_to_edhrec + edhrec_to_ga)

    print(f"\nGA (100) vs EDHRec Average (100)")
    print(f"Cosine Similarity (Pair-wise): {avg_sim:.4f}")
    print(f"Cosine Similarity (Best-match):{best_match_score:.4f}")
    print(f"Cosine Symmetry Similarity (Best-match):{avg_sim_2:.4f}")
    print(f"(Calculated using {len(ga_embs)} GA card instances and {len(edhrec_embs)} EDHRec card instances)")

    return avg_sim_2

# --- Example Run ---
if __name__ == '__main__':
    # Safely get model if it exists, otherwise pass None
    current_model = model if 'model' in globals() else None

    res = compare_ga_vs_edhrec_nonland_similarity(
        'Krenko, Mob Boss',
        'Cards with the Goblin subtype and abilities that create Goblin creature tokens or provide stat buffs to Goblins',
        df,
        embeddings,
        current_model
    )

In [ ]:
# Analyze the composition of the EDHRec average deck
# Using 'test_commander_name' which is defined globally as 'Averna, the Chaos Bloom'
_, edhrec_avg_raw = get_edhrec_average_and_score(test_commander_name)

# 1. Extract card names from the EDHRec list (stripping the leading quantity, e.g., '1 Sol Ring')
edhrec_names_only = [re.sub(r'^\d+\s+', '', name).strip() for name in edhrec_avg_raw]

# 2. Use 'raw_df' to identify which of these are actually lands (since 'df' had 'type_line' dropped)
# We check the 'type_line' for the word 'Land'
all_cards_in_avg_metadata = raw_df[raw_df['name'].isin(edhrec_names_only)]
land_names = all_cards_in_avg_metadata[all_cards_in_avg_metadata['type_line'].str.contains('Land', na=False)]['name'].tolist()

# 3. Filter the original list to exclude any card identified as a land
non_lands = [card for card in edhrec_avg_raw if re.sub(r'^\d+\s+', '', card).strip() not in land_names]

print(f"Total items in EDHRec 'Average' list: {len(edhrec_avg_raw)}")
print(f"Total non-land spells (True Spells) in EDHRec list: {len(non_lands)}")

print("\n--- EDHRec Average Deck True Spell List ---")
for i, card in enumerate(non_lands, 1):
    print(f"{i}. {card}")

# 4. Check role distribution in the EDHRec average deck for these spells
true_spell_names = [re.sub(r'^\d+\s+', '', card).strip() for card in non_lands]
edhrec_spells_df = df[df['name'].isin(true_spell_names)]

metrics = {
    'Ramp': edhrec_spells_df['is_Ramp'].sum(),
    'Draw': edhrec_spells_df['is_Card_Draw'].sum(),
    'Removal': edhrec_spells_df['is_Spot_Removal'].sum() + edhrec_spells_df['is_Board_Wipe'].sum()
}

import pandas as pd
display(pd.DataFrame([metrics], index=['EDHRec Average Deck Counts']))

print("\nThis filter now removes non-basic lands like Command Tower and dual lands.")
print("The count should now be much closer to the GA target of 61.")

##Compare GA EDHRec of different commanders

In [ ]:
def compare_cross_commander_similarity(ga_commander, ga_prompt, edhrec_commander, df, embeddings, model, ga_params=None):
    """
    Compares a GA-generated deck for one commander against the EDHRec average deck of potentially
    a different commander
    """
    if ga_params is None:
        ga_params = {'generations': 30, 'population_size': 1000, 'deck_size': 61}

    print(f"--- Cross-Commander Comparison ---")
    print(f"GA Deck: {ga_commander} (Theme: {ga_prompt})")
    print(f"EDHRec Baseline: {edhrec_commander}")

    # 1. Run GA for the first commander
    ga_spells = run_genetic_algorithm(
        ga_commander, ga_prompt, df, embeddings, model, **ga_params
    )

    # 2. Fetch EDHRec Average Deck for the second commander
    _, edhrec_avg_raw = get_edhrec_average_and_score(edhrec_commander)

    # 3. Calculate similarities
    ga_embs = get_weighted_embeddings([f"1 {n}" for n in ga_spells], df, embeddings)
    edhrec_embs = get_weighted_embeddings(edhrec_avg_raw, df, embeddings)

    if len(ga_embs) == 0 or len(edhrec_embs) == 0:
        return "Error: Missing embeddings."

    sim_matrix = cosine_similarity(ga_embs, edhrec_embs)

    # Metrics
    ga_to_edhrec = sim_matrix.max(axis=1).mean()
    edhrec_to_ga = sim_matrix.max(axis=0).mean()
    combined_sim = 0.5 * (ga_to_edhrec + edhrec_to_ga)

    print(f"\nSimilarity Results:")
    print(f"GA -> EDHRec (How much GA covers EDHRec themes): {ga_to_edhrec:.4f}")
    print(f"EDHRec -> GA (How much EDHRec covers GA themes): {edhrec_to_ga:.4f}")
    print(f"Symmetric Best-Match Similarity: {combined_sim:.4f}")

    return combined_sim

compare_cross_commander_similarity('Averna, the Chaos Bloom', 'cascade', 'Amalia Benavides Aguirre', df, embeddings, model)

##Similarity of Two EDHRec Avg Decks

In [ ]:
import pyedhrec
import re
import collections
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def get_weighted_embeddings(raw_list, df_ref, embs_ref):
    weighted_embs = []
    df_temp = df_ref.copy()
    df_temp['name_lower'] = df_temp['name'].str.lower()

    for item in raw_list:
        # Extract quantity and name (e.g., "20 Plains" or "1 Sol Ring")
        match_qty = re.match(r'^(\d+)\s+(.*)$', item)
        if match_qty:
            qty = int(match_qty.group(1))
            name = match_qty.group(2).strip().lower()
        else:
            qty = 1
            name = item.strip().lower()

        # Find the card in local database
        card_match = df_temp[df_temp['name_lower'] == name]
        if not card_match.empty:
            idx = card_match.index[0]
            if idx < len(embs_ref):
                # Append the embedding 'qty' times for 1-to-1 comparison
                for _ in range(qty):
                    weighted_embs.append(embs_ref[idx])

    return np.array(weighted_embs)

def compare_two_edhrec_decks_similarity(commander1_name, commander2_name, df, embeddings):
    """
    Compares the EDHRec average decks of two commanders and calculates their
    symmetric best-match cosine similarity.
    """
    print(f"\n--- Comparing EDHRec Average Decks: {commander1_name} vs {commander2_name} ---")

    # 1. Get EDHRec average deck for both commanders
    _, edhrec1_raw = get_edhrec_average_and_score(commander1_name)
    _, edhrec2_raw = get_edhrec_average_and_score(commander2_name)

    # 2. Get weighted embeddings for both decks
    edhrec1_embs = get_weighted_embeddings(edhrec1_raw, df, embeddings)
    edhrec2_embs = get_weighted_embeddings(edhrec2_raw, df, embeddings)

    if len(edhrec1_embs) == 0 or len(edhrec2_embs) == 0:
        print("Error: Could not retrieve embeddings for one or both decks.")
        return 0.0

    # 3. Calculate pairwise cosine similarity matrix
    sim_matrix = cosine_similarity(edhrec1_embs, edhrec2_embs)

    # 4. Calculate symmetric best-match similarity
    deck1_to_deck2_best_match = sim_matrix.max(axis=1).mean()
    deck2_to_deck1_best_match = sim_matrix.max(axis=0).mean()

    symmetric_best_match_sim = 0.5 * (deck1_to_deck2_best_match + deck2_to_deck1_best_match)

    print(f"Calculated Symmetric Best-Match Similarity: {symmetric_best_match_sim:.4f}")
    print(f"(Compared {len(edhrec1_embs)} cards from {commander1_name} and {len(edhrec2_embs)} cards from {commander2_name})")

    return symmetric_best_match_sim

if __name__ == '__main__':
    # Example Usage
    commander1_name = "Krenko, Mob Boss"
    commander2_name = "Elminster"

    similarity_score = compare_two_edhrec_decks_similarity(commander1_name, commander2_name, df, embeddings)
    print(f"\nFinal Similarity between {commander1_name} and {commander2_name}: {similarity_score:.4f}")

In [ ]:
import collections
import re
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

def compare_two_random_decks_similarity(commander1_name, commander2_name, df, embeddings, raw_df):
    """
    Calculates the symmetric best-match similarity between two randomly generated
    100-card decks (commander, 61 spells, 38 lands) for two different commanders.
    """
    print(f"--- Comparing Random Decks: {commander1_name} vs {commander2_name} ---")

    def build_random_deck(cmd_name):
        print(f"\nBuilding random deck for {cmd_name}...")
        cid = get_commander_color_identity(cmd_name, df)
        legal_df = filter_by_color_identity(cid, df)
        legal_df_without_commander = legal_df[legal_df['name'] != cmd_name].copy()

        target_deck_size = 99
        if len(legal_df_without_commander) < target_deck_size:
            raise ValueError(f"Not enough legal unique cards ({len(legal_df_without_commander)}) for {cmd_name} to build a {target_deck_size}-card deck.")

        random_99_cards_df = legal_df_without_commander.sample(n=target_deck_size, replace=False)
        random_99_cards_names = random_99_cards_df['name'].tolist()

        return [cmd_name] + random_99_cards_names

    deck1 = build_random_deck(commander1_name)
    deck2 = build_random_deck(commander2_name)

    print(f"\nRandom Deck 1 ({commander1_name}) size: {len(deck1)}")
    print(f"Random Deck 2 ({commander2_name}) size: {len(deck2)}")

    deck1_embs = get_weighted_embeddings(deck1, df, embeddings)
    deck2_embs = get_weighted_embeddings(deck2, df, embeddings)

    if len(deck1_embs) == 0 or len(deck2_embs) == 0:
        print("Error: Could not retrieve embeddings for one or both random decks.")
        return 0.0

    sim_matrix = cosine_similarity(deck1_embs, deck2_embs)
    deck1_to_deck2_best_match = sim_matrix.max(axis=1).mean()
    deck2_to_deck1_best_match = sim_matrix.max(axis=0).mean()
    symmetric_best_match_sim = 0.5 * (deck1_to_deck2_best_match + deck2_to_deck1_best_match)

    print(f"\nCalculated Symmetric Best-Match Similarity: {symmetric_best_match_sim:.4f}")
    print(f"(Compared {len(deck1_embs)} cards from {commander1_name} and {len(deck2_embs)} cards from {commander2_name})")

    return symmetric_best_match_sim

In [ ]:
# Example Usage of compare_two_random_decks_similarity
commander1_name_rand = "Elminster"
commander2_name_rand = "Gargos, Vicious Watcher"

random_similarity = compare_two_random_decks_similarity(
    commander1_name_rand,
    commander2_name_rand,
    df,
    embeddings,
    raw_df
)

print(f"\nFinal Random Deck Similarity between {commander1_name_rand} and {commander2_name_rand}: {random_similarity:.4f}")


In [ ]:
import collections

def compare_ga_vs_random_baseline(commander_name, prompt, df, embeddings, model, ga_params=None):
    """
    Compares the Genetic Algorithm deck against a deck of completely random
    non-land cards that are legal for the commander's color identity.
    Uses the similarity formulas from cell 1lB4t9KRPAai and includes lands (Total 100 cards).
    """
    if ga_params is None:
        ga_params = {'generations': 30, 'population_size': 1000, 'deck_size': 61}

    print(f"--- GA vs Random Baseline Comparison (Full 100-Card Decks) ---")
    print(f"Commander: {commander_name}")
    print(f"Prompt: {prompt}")

    # 1. Get Color Identity and Filtered Legal Cards
    cid = get_commander_color_identity(commander_name, df)
    legal_df = filter_by_color_identity(cid, df)

    spell_types = ['Creature', 'Instant', 'Sorcery', 'Artifact', 'Enchantment', 'Planeswalker']
    legal_names = legal_df['name'].tolist()

    # Get metadata from raw_df to filter spells
    legal_metadata = raw_df[raw_df['name'].isin(legal_names)]
    legal_spells_metadata = legal_metadata[legal_metadata['type_line'].str.contains('|'.join(spell_types), na=False)]

    # 2. Run GA for Spells
    ga_spells = run_genetic_algorithm(
        commander_name, prompt, df, embeddings, model, **ga_params
    )

    # 3. Generate Random Spells
    random_spells = legal_spells_metadata.sample(ga_params['deck_size'])['name'].tolist()

    # 4. Generate Lands for Both Decks
    ga_lands = calculate_land_distribution(ga_spells, df, cid, total_lands=38)
    random_lands = calculate_land_distribution(random_spells, df, cid, total_lands=38)

    # 5. Assemble Full Decks (including Commander)
    ga_full_deck = [commander_name] + ga_spells + ga_lands
    random_full_deck = [commander_name] + random_spells + random_lands

    # 6. Extract Weighted Embeddings (to account for multiple basics)
    def get_weighted_embeddings(name_list, df_ref, embs_ref):
        weighted_embs = []
        df_map = df_ref.set_index('name')

        # Count occurrences for weighted embeddings
        counts = collections.Counter(name_list)

        for name, qty in counts.items():
            if name in df_map.index:
                idx = df_ref.index[df_ref['name'] == name][0]
                if idx < len(embs_ref):
                    for _ in range(qty):
                        weighted_embs.append(embs_ref[idx])
        return np.array(weighted_embs)

    ga_embs = get_weighted_embeddings(ga_full_deck, df, embeddings)
    random_embs = get_weighted_embeddings(random_full_deck, df, embeddings)

    if len(ga_embs) == 0 or len(random_embs) == 0:
        print("Error: Could not retrieve embeddings for comparison.")
        return

    # 7. Apply Formulas from 1lB4t9KRPAai
    sim_matrix = cosine_similarity(ga_embs, random_embs)

    avg_sim = np.mean(sim_matrix)
    ga_to_random = sim_matrix.max(axis=1).mean()
    random_to_ga = sim_matrix.max(axis=0).mean()
    symmetric_sim = 0.5 * (ga_to_random + random_to_ga)

    print(f"\nResults (Calculated with {len(ga_embs)} GA cards vs {len(random_embs)} Random cards):")
    print(f"Cosine Similarity (Pair-wise): {avg_sim:.4f}")
    print(f"Cosine Similarity (Best-match): {ga_to_random:.4f}")
    print(f"Cosine Symmetry Similarity (Best-match): {symmetric_sim:.4f}")

    return symmetric_sim

compare_ga_vs_random_baseline('Krenko, Mob Boss', 'Cards with the Goblin subtype and abilities that create Goblin creature tokens or provide stat buffs to Goblins', df, embeddings, model)

##Comprehensive comparison
Compares random, GA, Greedy decks against EDHRec average deck.

In [ ]:
import os
import pandas as pd
import numpy as np
import collections
import re
from sklearn.metrics.pairwise import cosine_similarity

def run_comprehensive_evaluation(commander_name, prompt, df, embeddings, model, trials=1, ga_params=None):
    """
    Executes a 4-step evaluation (EDHRec, Random, Greedy, GA) and logs metrics to CSV.
    Also logs generated decklists to a text file.
    """
    if ga_params is None:
        ga_params = {'generations': 30, 'population_size': 1000, 'mutation_rate': 0.05, 'deck_size': 61}

    CSV_LOG_PATH = 'evaluation_results.csv'
    TXT_LOG_PATH = 'evaluation_decklists.txt'
    results_list = []

    # 1. Setup Commander and Pool
    try:
        cid = get_commander_color_identity(commander_name, df)
        cmd_idx = df[df['name'] == commander_name].index[0]
        legal_df = filter_by_color_identity(cid, df)
        spell_types = ['Creature', 'Instant', 'Sorcery', 'Artifact', 'Enchantment', 'Planeswalker']
        # Use raw_df for type checking if available
        type_source = raw_df if 'raw_df' in globals() else df
        # Ensure 'type_line' column is present in type_source before filtering
        if 'type_line' not in type_source.columns:
            print("Warning: 'type_line' column not found in type_source. Filtering by type might be inaccurate.")
            legal_spells_df = legal_df.copy()
        else:
            # Filter by type_line and ensure the index aligns with legal_df
            legal_spells_df = legal_df[type_source.loc[legal_df.index, 'type_line'].str.contains('|'.join(spell_types), na=False)]
    except Exception as e:
        print(f"Setup Error: {e}")
        return

    for trial in range(1, trials + 1):
        print(f"\n--- Trial {trial}/{trials} for {commander_name} ---")

        decklist_buffer = [] # Buffer to store decklists for current trial
        decklist_buffer.append(f"\n{'='*70}")
        decklist_buffer.append(f"TRIAL {trial}/{trials} - COMMANDER: {commander_name}, PROMPT: {prompt}")
        decklist_buffer.append(f"{'='*70}\n")

        # A. EDHRec Baseline
        print("Running EDHRec Baseline...")
        try:
            edh_score, edh_raw = get_edhrec_average_and_score(commander_name)
            # Extract names only for similarity comparison
            edh_names_for_metrics = [re.sub(r'^\d+\s+', '', n).strip() for n in edh_raw]
            edhrec_set_for_overlap = set(edh_names_for_metrics)
        except Exception as e:
            print(f"EDHRec API failed: {e}")
            continue

        # B. Random Selection
        print("Running Random Baseline...")
        rand_spells = legal_spells_df.sample(ga_params['deck_size'])['name'].tolist()
        rand_lands = calculate_land_distribution(rand_spells, df, cid)

        # C. Structured Greedy
        print("Running Greedy Baseline...")
        greedy_full = run_greedy_baseline(commander_name, prompt, df, embeddings, model)
        greedy_spells = greedy_full[:ga_params['deck_size']] # Assuming spells are at the beginning

        # D. Genetic Algorithm
        print("Running GA...")
        ga_spells = run_genetic_algorithm(commander_name, prompt, df, embeddings, model, **ga_params)
        ga_lands = calculate_land_distribution(ga_spells, df, cid)

        methods = {
            'EDHRec_Average': edh_raw, # Use raw_edh_raw for printing quantity
            'Random': rand_spells + rand_lands,
            'Greedy': greedy_full,
            'GA': ga_spells + ga_lands
        }

        for method_name, deck_list in methods.items():
            print(f"\n--- Decklist for {method_name} ---")
            decklist_buffer.append(f"--- {method_name} Decklist ---")
            decklist_buffer.append(f"Commander: {commander_name}, Prompt: {prompt}")

            if method_name == 'EDHRec_Average':
                for card_info in deck_list:
                    print(f"- {card_info}")
                    decklist_buffer.append(f"- {card_info}")
                # For EDHRec_Average, we use the already processed edh_names_for_metrics for other metrics
                metric_deck_names = edh_names_for_metrics
            else:
                for card_name in deck_list:
                    print(f"- {card_name}")
                    decklist_buffer.append(f"- {card_name}")
                metric_deck_names = deck_list
            decklist_buffer.append("\n")

            # Metric: Optimization Score
            deck_indices = df[df['name'].isin(metric_deck_names)].index.tolist()
            opt_score = calculate_deck_optimization(deck_indices, cmd_idx, embeddings, df)

            # Metric: Similarity to EDHRec (Best-match symmetric)
            def get_deck_embs(names):
                indices = df[df['name'].isin(names)].index.tolist()
                if not indices:
                    return np.array([])
                return embeddings[indices]

            method_embs = get_deck_embs(metric_deck_names)
            edh_embs_for_sim = get_deck_embs(edh_names_for_metrics)

            sim_score = 0.0
            if method_embs.size > 0 and edh_embs_for_sim.size > 0:
                sim_matrix = cosine_similarity(method_embs, edh_embs_for_sim)
                sim_score = 0.5 * (sim_matrix.max(axis=1).mean() + sim_matrix.max(axis=0).mean())

            # Metric: EDHRec Overlap Count
            current_deck_set = set(metric_deck_names)
            edhrec_overlap_count = len(current_deck_set.intersection(edhrec_set_for_overlap))

            # Metric: Role Distribution
            role_df = df[df['name'].isin(metric_deck_names)]
            roles = {
                'ramp': int(role_df['is_Ramp'].sum()),
                'draw': int(role_df['is_Card_Draw'].sum()),
                'interaction': int(role_df['is_Spot_Removal'].sum() + role_df['is_Board_Wipe'].sum())
            }

            # Metric: Mana Curve
            # Need to ensure that the indices for type_source are consistent with the current df's names.
            # Using original df for mana_value if type_source is raw_df and doesn't contain all current names.
            deck_for_mana_curve = df[df['name'].isin(metric_deck_names)]
            mvs = type_source.loc[deck_for_mana_curve.index, 'mana_value'].fillna(0).round().astype(int)
            curve = {f"mv_{i}": int((mvs == i).sum()) for i in range(6)}
            curve["mv_6plus"] = int((mvs >= 6).sum())

            # Combine Data
            row = {
                'trial_id': f"{commander_name}_{trial}",
                'commander': commander_name,
                'prompt': prompt,
                'method': method_name,
                'opt_score': opt_score,
                'edhrec_similarity': sim_score,
                'edhrec_overlap_count': edhrec_overlap_count,
                **roles,
                **curve,
                **ga_params
            }
            results_list.append(row)

        # Write all decklists for the current trial to the text file
        with open(TXT_LOG_PATH, 'a') as f:
            f.write('\n'.join(decklist_buffer))
        print(f"Decklists for Trial {trial} appended to {TXT_LOG_PATH}")

    # Log to CSV
    results_df = pd.DataFrame(results_list)
    header = not os.path.exists(CSV_LOG_PATH)
    results_df.to_csv(CSV_LOG_PATH, mode='a', index=False, header=header)
    print(f"\nEvaluation complete. Results appended to {CSV_LOG_PATH}")
    return results_df

print("Function 'run_comprehensive_evaluation' defined.")

In [ ]:
run_comprehensive_evaluation('Arabella, Abandoned Doll', 'creatures with power 2 or less', df, embeddings, model, 3)

In [ ]:
import pandas as pd
import os

# Define the correct file paths based on the previous file search
drive_project_path = '/content/drive/MyDrive/MTGCISE'
csv_path = os.path.join(drive_project_path, 'evaluation_results.csv')
txt_path = os.path.join(drive_project_path, 'evaluation_decklists.txt')

# Load evaluation results CSV
if os.path.exists(csv_path):
    results_df = pd.read_csv(csv_path)
    print(f"Successfully loaded {csv_path}")
    # Display summary and first few rows
    print("\n--- DataFrame Info ---")
    results_df.info()
    print("\n--- Sample Data ---")
    display(results_df.head())
else:
    print(f"Error: {csv_path} not found.")

# Verify decklists text file structure
if os.path.exists(txt_path):
    print(f"\n--- Verifying {txt_path} structure ---")
    with open(txt_path, 'r') as f:
        # Read the first 500 characters for verification
        preview = f.read(500)
        print(preview + "... [truncated]")
else:
    print(f"Error: {txt_path} not found.")

In [ ]:
# Select metrics for analysis
metrics = ['opt_score', 'edhrec_similarity', 'edhrec_overlap_count', 'ramp', 'draw', 'interaction']

# Group by method and calculate mean and standard deviation
summary_stats = results_df.groupby('method')[metrics].agg(['mean', 'std']).round(4)

# Display the aggregated summary table
print("--- Aggregated Metrics by Generation Method ---")
display(summary_stats)

# Specifically check role fulfillment means against the 10/10/10 target
role_fulfillment = summary_stats[[('ramp', 'mean'), ('draw', 'mean'), ('interaction', 'mean')]]
role_fulfillment.columns = ['Mean Ramp', 'Mean Draw', 'Mean Interaction']

print("\n--- Functional Role Fulfillment Analysis (Target: 10/10/10) ---")
display(role_fulfillment)

In [ ]:
!pip install python-docx
import docx

file_path = '/content/Metrics Evaluation - Decks.docx'
try:
    doc = docx.Document(file_path)
    print("--- Document Content ---")
    for para in doc.paragraphs:
        if para.text.strip():
            print(para.text)
except Exception as e:
    print(f"Error reading document: {e}")


#Part 3: SBM
Symmetric Best-Match Similarity

In [ ]:
# =============================================================================
# Part 3 setup -- run this after a fresh runtime start and Part 3 works on its own
# (no need to run Part 1 or Part 2 first).
# Provides: df, embeddings, raw_df, model, get_commander_color_identity,
# filter_by_color_identity, calculate_deck_optimization, and the EDHRec helpers
# (get_edhrec_average_and_score, edhrec_call, build_name_index, lookup_card).
# =============================================================================
!pip install -q pyedhrec sentence_transformers scipy

import os
import re
import time
import random
import warnings
import logging
import requests
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer
from transformers import logging as transformers_logging
from pyedhrec import EDHRec
from google.colab import drive

transformers_logging.set_verbosity_error()
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

# --- 1. Data (from Google Drive) ---------------------------------------------
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
CSV_PATH = os.path.join(DRIVE_PATH, MODEL_READY_CSV)         # features + embeddings
NPY_PATH = os.path.join(DRIVE_PATH, EMBEDDINGS_NPY)                  # embedding matrix
RAW_CSV_PATH = os.path.join(DRIVE_PATH, 'cleaned_mtg_card_features.csv')    # has type_line (for land filtering)

for path in (CSV_PATH, NPY_PATH, RAW_CSV_PATH):
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing {path} -- check that Drive is mounted and the file is in MTGCISE.")

df = pd.read_csv(CSV_PATH)
embeddings = np.load(NPY_PATH)
raw_df = pd.read_csv(RAW_CSV_PATH)
if len(df) != len(embeddings):
    raise ValueError(f"df has {len(df)} rows but embeddings has {len(embeddings)} -- re-run the embedding cell in Part 1.")

if 'model' not in globals() or model is None:
    model = SentenceTransformer(EMBEDDING_MODEL_NAME)


# --- 2. Color identity helpers (from Part 1) ----------------------------------
def get_commander_color_identity(commander_name, df):
    """
    Locates the commander in the DataFrame and returns its color identity string.
    Handles exact matches and MDFC names (e.g., "Name // Name").
    """
    # Ensure name column is string
    df['name'] = df['name'].astype(str)

    # Try exact match first
    matches = df[df['name'] == commander_name]

    # If no exact match, try identifying it as the front face of a split/MDFC card
    if matches.empty:
        # Search for names starting with "Commander Name //"
        prefix = commander_name + " //"
        matches = df[df['name'].str.startswith(prefix)]

    if matches.empty:
        # Fallback: simple containment check
        matches = df[df['name'].str.contains(commander_name, regex=False)]

    if matches.empty:
        raise ValueError(f"Commander '{commander_name}' not found in dataset.")

    # Take the first match
    commander_row = matches.iloc[0]

    # Reconstruct color identity from One-Hot Encoded columns
    colors = []
    for color in ['W', 'U', 'B', 'R', 'G']:
        col_name = f'CI_{color}'
        if col_name in df.columns and commander_row[col_name] == 1:
            colors.append(color)

    return ''.join(colors) if colors else 'C'


def filter_by_color_identity(commander_color_id, df):
    """
    Filters the DataFrame to include only cards whose color identity is a subset of the commander's.
    """
    # 1. Identify which colors are NOT allowed
    allowed_colors = set(commander_color_id)
    all_colors = ['W', 'U', 'B', 'R', 'G']
    forbidden_colors = [c for c in all_colors if c not in allowed_colors]

    # 2. Filter out cards that have any forbidden color
    mask = pd.Series([True] * len(df), index=df.index)

    for color in forbidden_colors:
        col_name = f'CI_{color}'
        if col_name in df.columns:
            mask = mask & (df[col_name] == 0)

    return df[mask].copy()


# --- 3. Deck optimization score (from Part 1; used to score EDHRec decks) -----
def calculate_deck_optimization(deck, cmd_idx, embs, df, weights=None):
    """
    Calculates a multi-factor optimization score for a 99-card deck.
    Now accepts a 'weights' dictionary for dynamic scoring.
    """
    # Default weights if none provided
    if weights is None:
        weights = {'alignment': 0.95, 'cohesion': 0.0, 'structure': 0.05}

    # 1. EXTRACT DATA
    # The function expects deck indices and commander index to be valid for embs and df
    # The error handling for out-of-bounds is usually done upstream during index generation.

    deck_vecs = embs[deck]
    cmd_vec = embs[cmd_idx].reshape(1, -1)

    # 2. COMMANDER ALIGNMENT
    # FIXED: cmd_vec was defined above, but comm_vec was used here. Fixed typo.
    comm_sims = cosine_similarity(cmd_vec, deck_vecs).flatten()
    alignment_score = np.mean(comm_sims)

    # 3. INTRA-DECK COHESION
    cohesion_score = 0.0
    if len(deck_vecs) > 1:
        inner_sim_matrix = cosine_similarity(deck_vecs)
        triu_indices = np.triu_indices(len(deck_vecs), k=1)
        if triu_indices[0].size > 0:
            cohesion_score = np.mean(inner_sim_matrix[triu_indices])
        else:
            cohesion_score = 0.0

    # 4. ROLE SATISFACTION
    # Target: 10 Ramp, 10 Draw, 10 Interaction (Spot Removal + Wipes)
    roles_present = df.iloc[deck]

    def get_role_sum(col_name):
        return roles_present[col_name].sum() if col_name in roles_present.columns else 0

    ramp_count = get_role_sum('is_Ramp')
    draw_count = get_role_sum('is_Card_Draw')

    # Interaction covers Spot Removal and Board Wipes
    removal_count = get_role_sum('is_Spot_Removal')
    wipe_count = get_role_sum('is_Board_Wipe')
    interaction_count = removal_count + wipe_count

    # Score Saturation up to 12 cards, penalize over 12
    if ramp_count <= 12:
        ramp_score = min(ramp_count / 12.0, 1.0) # Linearly increases up to 1.0 at 12
    else:
        # Penalize if ramp_count exceeds 12. Score drops from 1.0.
        # Max penalty if count reaches 24 (12 + 12 = 24), score becomes 0.0
        ramp_score = max(0.0, 1.0 - (ramp_count - 12) / 12.0)

    if draw_count <= 12:
        draw_score = min(draw_count / 12.0, 1.0) # Linearly increases up to 1.0 at 12
    else:
        # Penalize if draw_count exceeds 12. Score drops from 1.0.
        draw_score = max(0.0, 1.0 - (draw_count - 12) / 12.0)

    # Interaction score remains capped at 1.0 at 10, as no specific penalty was requested for it.
    interaction_score = min(interaction_count / 10.0, 1.0)

    structure_score = (ramp_score + draw_score + interaction_score) / 3.0

    # 5. FINAL OPTIMIZATION SCORE using dynamic weights
    final_score = (
        (alignment_score * weights.get('alignment', 0.85)) +
        (cohesion_score * weights.get('cohesion', 0.10)) +
        (structure_score * weights.get('structure', 0.05))
    )

    return final_score


# --- 4. EDHRec helpers (from Part 2) ------------------------------------------
# One shared client for the whole notebook. pyedhrec downloads EDHRec's homepage to
# look up the site's build id the first time a client is used, so making a new
# EDHRec() on every call doubles the traffic -- that burst is what got the
# "Connection reset by peer" errors.
EDH_CLIENT = EDHRec()

# Every card EDHRec files under "Land" in an average deck fetched this session.
EDHREC_LAND_NAMES = globals().get('EDHREC_LAND_NAMES', set())


def edhrec_call(method, *args, retries=3, backoff=2.0):
    """Call a pyedhrec method, retrying dropped connections, rate limits and a stale build id."""
    for attempt in range(1, retries + 1):
        try:
            time.sleep(0.5)
            return method(*args)
        except requests.exceptions.RequestException as e:
            if attempt == retries:
                raise
            print(f"  EDHRec request failed ({type(e).__name__}); retry {attempt}/{retries - 1}...")
            EDH_CLIENT.current_build_id = None   # make pyedhrec look the build id up again
            time.sleep(backoff * attempt)


def _clean_card_name(raw):
    name = re.sub(r'^[\s ]*\d+\s+', '', str(raw))   # drop a leading quantity: "1 Sol Ring"
    name = re.sub(r'\s*\([^)]*\)', '', name)             # drop set tags like "(M21)"
    return name.strip()


def _name_and_qty(card):
    if isinstance(card, (list, tuple)):
        return card[0], (card[1] if len(card) > 1 else 1)
    if isinstance(card, dict):
        return card.get('name', ''), card.get('quantity', 1)
    return card, 1


def parse_edhrec_decklist(decklist):
    """
    Turns EDHRec's average-deck payload into [(name, qty, category), ...].

    EDHRec now sends a dict:
        {"commander": [...], "commander_v2": [[name, qty]],
         "cards": {"Creature": [[name, qty], ...], "Land": [...], ...}}
    It used to send a flat list: ["1 Krenko, Mob Boss", "1 Sol Ring", "29 Mountain", ...]
    Both are handled. The commander stays in the list, like it was in the old format.
    """
    entries = []
    if isinstance(decklist, dict):
        commanders = decklist.get('commander_v2') or decklist.get('commander') or []
        for card in commanders:
            name, qty = _name_and_qty(card)
            if name:
                entries.append((_clean_card_name(name), int(qty), 'Commander'))
        for category, cards in (decklist.get('cards') or {}).items():
            for card in cards:
                name, qty = _name_and_qty(card)
                if name:
                    entries.append((_clean_card_name(name), int(qty), category))
    elif isinstance(decklist, list):
        for item in decklist:
            raw = item.get('name', '') if isinstance(item, dict) else item
            if isinstance(raw, str) and raw.strip():
                qty = re.match(r'^[\s ]*(\d+)\s+', raw)
                entries.append((_clean_card_name(raw), int(qty.group(1)) if qty else 1, None))
    return entries


def build_name_index(df):
    """Lower-cased card name -> df row label. Exact names win; the front face of a '//' card is a fallback."""
    names = df['name'].astype(str).str.lower().str.strip()
    lookup = {}
    for idx, n in zip(df.index, names):
        lookup.setdefault(n, idx)
    for idx, n in zip(df.index, names):
        if ' // ' in n:
            lookup.setdefault(n.split(' // ')[0], idx)
    return lookup


def lookup_card(name, lookup):
    key = _clean_card_name(name).lower()
    if key in lookup:
        return lookup[key]
    if ' // ' in key:
        return lookup.get(key.split(' // ')[0])
    return None


def get_edhrec_average_and_score(commander_name):
    """
    Fetches the average deck for a commander from EDHRec and scores it with
    calculate_deck_optimization. Returns (score, ["qty Card Name", ...]).
    """
    # 1. Generate slug
    name_for_slug = commander_name.split(' // ')[0]
    slug = name_for_slug.lower().replace("'", "").replace(",", "").replace(" ", "-")
    slug = re.sub(r'-+', '-', slug)
    print(f"Fetching EDHRec data for: {commander_name} (slug: {slug})")

    # 2. Fetch and parse
    try:
        data = edhrec_call(EDH_CLIENT.get_commanders_average_deck, slug)
    except Exception as e:
        print(f"Error fetching average deck for {commander_name}: {e}")
        return 0.0, []

    entries = parse_edhrec_decklist((data or {}).get('decklist'))
    if not entries:
        print(f"Warning: EDHRec returned no readable cards for {commander_name}.")
        return 0.0, []

    baseline_data = {}   # lower-cased name -> "qty Name"
    for name, qty, category in entries:
        baseline_data[name.lower()] = f"{qty} {name}"
        if category == 'Land':
            EDHREC_LAND_NAMES.add(name.lower())

    # 3. Map to local DataFrame rows
    df['name_clean'] = df['name'].str.lower().str.strip()   # kept for other cells that use it
    lookup = build_name_index(df)
    matched, unmatched = set(), []
    for key, line in baseline_data.items():
        idx = lookup_card(key, lookup)
        if idx is None:
            unmatched.append(line)
        else:
            matched.add(int(idx))
    deck_indices = sorted(matched)

    # 4. Commander row
    commander_idx = lookup_card(commander_name, lookup)
    if commander_idx is None:
        print(f"Warning: Commander '{commander_name}' not found in local dataset.")
        return 0.0, sorted(baseline_data.values())

    # 5. Optimization score
    if not deck_indices:
        print("No cards from the EDHRec average deck were found in the local dataset.")
        return 0.0, sorted(baseline_data.values())

    score = calculate_deck_optimization(deck_indices, int(commander_idx), embeddings, df)
    print(f"Average Deck Optimization Score: {score:.4f}")
    print(f"Matched {len(deck_indices)} of {len(baseline_data)} cards in local dataset.")
    if unmatched:
        print(f"  Not in local dataset: {unmatched[:8]}{' ...' if len(unmatched) > 8 else ''}")

    return score, sorted(baseline_data.values())


print(f"Part 3 ready: {len(df)} cards, embeddings {embeddings.shape}, raw_df {len(raw_df)} rows, model loaded.")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Part 3 ready: 30969 cards, embeddings (30969, 768), raw_df 30969 rows, model loaded.


## Jaccard, Pairwise, SBM, EDH Synergy Score, Hungarian Matching

In [ ]:
!pip install pyedhrec
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from pyedhrec import EDHRec
import re
import time
from scipy.optimize import linear_sum_assignment

# --- 1. Jaccard Similarity (Modified for Oracle Text with Keyword Filtering) ---
def calculate_jaccard_similarity(deck1_texts, deck2_texts):
    """
    Measures rule text overlap as the ratio of shared oracle texts to total unique oracle texts.
    Now filters out non-meaningful words to focus on mechanical keywords.
    """
    # Define a basic set of stop words to remove
    STOP_WORDS = {'is', 'and', 'the', 'a', 'an', 'it', 'to', 'of', 'in', 'on', 'with', 'by', 'for', 'at', 'this', 'that'}

    def get_keywords(text_list):
        keywords = set()
        for text in text_list:
            if not isinstance(text, str): continue
            # Lowercase, remove punctuation, and split into words
            words = re.findall(r'\w+', text.lower())
            # Filter out stop words
            keywords.update([w for w in words if w not in STOP_WORDS])
        return keywords

    set1 = get_keywords(deck1_texts)
    set2 = get_keywords(deck2_texts)

    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))

    if union == 0:
        return 0.0
    return intersection / union

# --- 1b. Jaccard Similarity for Card Names ---
def calculate_jaccard_name_similarity(deck1_names, deck2_names):
    """
    Measures card name overlap as the ratio of shared card names to total unique card names.
    """
    if not isinstance(deck1_names, list) or not isinstance(deck2_names, list):
        raise TypeError("Inputs must be lists of card names.")

    set1 = set([name.strip().lower() for name in deck1_names if isinstance(name, str)])
    set2 = set([name.strip().lower() for name in deck2_names if isinstance(name, str)])

    intersection = len(set1.intersection(set2))
    union = len(set1.union(set2))

    if union == 0:
        return 0.0
    return intersection / union

# --- 2. Pairwise Cosine Similarity (Mean Pooling) ---
def calculate_pairwise_cosine_similarity(deck1_embs, deck2_embs):
    """
    Aggregates each deck into a single vector via mean pooling, then computes
    cosine similarity between the two resulting deck vectors.
    """
    if len(deck1_embs) == 0 or len(deck2_embs) == 0:
        return 0.0

    # Mean pool all embeddings in the deck to create a single vector
    deck1_vector = np.mean(deck1_embs, axis=0).reshape(1, -1)
    deck2_vector = np.mean(deck2_embs, axis=0).reshape(1, -1)

    return cosine_similarity(deck1_vector, deck2_vector)[0][0]

# --- 3. Symmetric Best-Match Cosine Similarity ---
def calculate_symmetric_best_match_similarity(deck1_embs, deck2_embs):
    """
    Finds for each card in deck A the max cosine similarity to any card in deck B,
    and averages both directional scores symmetrically.
    """
    if len(deck1_embs) == 0 or len(deck2_embs) == 0:
        return 0.0

    sim_matrix = cosine_similarity(deck1_embs, deck2_embs)

    deck1_to_deck2_best = sim_matrix.max(axis=1).mean()
    deck2_to_deck1_best = sim_matrix.max(axis=0).mean()

    return 0.5 * (deck1_to_deck2_best + deck2_to_deck1_best)

# --- 4. EDHRec Synergy Score ---
_SYNERGY_CACHE = {}

def get_edhrec_synergy_map(commander_name):
    """Lower-cased card name -> EDHRec synergy with this commander. Fetched once per commander."""
    name_for_slug = commander_name.split(' // ')[0].replace("'", "").replace(",", "")
    slug = re.sub(r'-+', '-', name_for_slug.lower().replace(' ', '-'))
    if slug in _SYNERGY_CACHE:
        return _SYNERGY_CACHE[slug]

    data = edhrec_call(EDH_CLIENT.get_commander_cards, slug)   # shared client + retries (cell 37)
    synergy_map = {}

    def extract_synergy(items):
        for item in items or []:
            if isinstance(item, dict) and 'name' in item and isinstance(item.get('synergy'), (int, float)):
                cname = re.sub(r'\s*\([^)]*\)', '', item['name']).strip().lower()
                synergy_map[cname] = item['synergy']

    if isinstance(data, dict):
        for content in data.values():
            if isinstance(content, list):
                extract_synergy(content)
            elif isinstance(content, dict):
                extract_synergy(content.get('cardviews') or content.get('cards'))

    if not synergy_map:
        raise ValueError(f"EDHRec returned no synergy data for {commander_name} (slug: {slug}).")
    _SYNERGY_CACHE[slug] = synergy_map
    return synergy_map


def calculate_edhrec_synergy_score(commander_name, deck_names):
    """
    Mean EDHRec synergy of the deck's cards with commander_name.
    Cards EDHRec doesn't list for that commander count as 0.0.
    """
    try:
        synergy_map = get_edhrec_synergy_map(commander_name)
    except Exception as e:
        print(f"Error fetching EDHRec Synergy for {commander_name}: {e}")
        return float('nan')   # NaN rather than 0.0, so a failed fetch can't pass for a real score

    scores = [synergy_map.get(re.sub(r'^\d+\s+', '', card).strip().lower(), 0.0) for card in deck_names]
    return float(np.mean(scores)) if scores else 0.0

# --- 5. Hungarian Algorithm (One-to-One Best Match) ---
def calculate_hungarian_similarity(deck1_embs, deck2_embs):
    """
    Finds the optimal one-to-one matching between two decks using the Hungarian algorithm,
    and returns the average cosine similarity of the matched pairs.
    """
    if len(deck1_embs) == 0 or len(deck2_embs) == 0:
        return 0.0

    sim_matrix = cosine_similarity(deck1_embs, deck2_embs)

    # We want to maximize similarity, which is equivalent to minimizing negative similarity
    cost_matrix = -sim_matrix

    row_ind, col_ind = linear_sum_assignment(cost_matrix)
    optimal_similarities = sim_matrix[row_ind, col_ind]

    return np.mean(optimal_similarities)


In [ ]:
import random

def generate_perturbed_deck(original_deck, df, commander_name, perturbation_rate=0.15):
    """
    Generates a perturbed version of a deck by replacing a percentage of its non-commander cards
    with random legal cards for the given commander.
    """
    deck_copy = original_deck.copy()

    # Protect the commander from being swapped
    commander_in_deck = [card for card in deck_copy if commander_name in card]
    non_commanders = [card for card in deck_copy if commander_name not in card]

    num_to_swap = int(len(non_commanders) * perturbation_rate)

    if num_to_swap == 0:
        return deck_copy

    # Get legal pool to swap from
    cid = get_commander_color_identity(commander_name, df)
    legal_df = filter_by_color_identity(cid, df)
    legal_pool = legal_df[~legal_df['name'].isin([commander_name])]['name'].tolist()

    # Perform swaps
    indices_to_swap = random.sample(range(len(non_commanders)), num_to_swap)
    current_deck_set = set([re.sub(r'^\d+\s+', '', card).strip() for card in deck_copy])

    for idx in indices_to_swap:
        for _ in range(10): # Try 10 times to find a unique replacement
            new_card = random.choice(legal_pool)
            if new_card not in current_deck_set:
                # Maintain quantity format if it existed (assume 1 for simplicity in swaps)
                non_commanders[idx] = new_card
                current_deck_set.add(new_card)
                break

    return commander_in_deck + non_commanders

print("Metric and Perturbation functions loaded successfully.")

Metric and Perturbation functions loaded successfully.


In [ ]:
import pandas as pd
import numpy as np
import collections
import re
import os
from sentence_transformers import SentenceTransformer

# --- Ensure Data and Model are Loaded ---
if 'df' not in locals() or df.empty or 'model' not in locals() or model is None:
    print("Initializing resources...")
    DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
    CSV_PATH = os.path.join(DRIVE_PATH, MODEL_READY_CSV)
    NPY_PATH = os.path.join(DRIVE_PATH, EMBEDDINGS_NPY)

    if os.path.exists(CSV_PATH) and os.path.exists(NPY_PATH):
        df = pd.read_csv(CSV_PATH)
        embeddings = np.load(NPY_PATH)
        # Re-initialize the model specifically if it's missing
        model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        print("Resources and model loaded successfully.")
    else:
        print("Data files not found. Ensure Google Drive is mounted.")

def run_full_metric_evaluation(commander_name, prompt, df, embeddings, model, ga_params=None, num_perturbations=10):
    """
    Runs the full evaluation suite across 5 conditions using the 4 specified metrics.
    """
    if ga_params is None:
        ga_params = {'generations': 30, 'population_size': 1000, 'mutation_rate': 0.05, 'deck_size': 61}

    print(f"\n{'='*70}")
    print(f" RUNNING COMPREHENSIVE METRIC EVALUATION")
    print(f" Commander: {commander_name}")
    print(f" Prompt: {prompt}")
    print(f"{'='*70}")

    # Check for required helper functions in scope
    required_funcs = ['get_commander_color_identity', 'filter_by_color_identity', 'calculate_land_distribution', 'get_edhrec_average_and_score', 'run_greedy_baseline', 'run_genetic_algorithm']
    missing = [f for f in required_funcs if f not in globals()]
    if missing:
        print(f"Error: Missing required functions: {missing}. Please run the cells containing these definitions first.")
        return pd.DataFrame()

    # Logic continues with checks to ensure model exists...
    try:
        metrics_df = run_full_metric_evaluation('Averna, the Chaos Bloom', 'cascade', df, embeddings, model)
        display(metrics_df)
    except Exception as e:
        print(f"Evaluation execution error: {e}")

In [ ]:
import os
import re
import random
import numpy as np
import pandas as pd

RAW_CSV_PATH = '/content/drive/MyDrive/MTGCISE/cleaned_mtg_card_features.csv'

METRICS = ['Jaccard (Text)', 'Jaccard (Name)', 'Pairwise Cosine',
           'Symmetric Best-Match', 'Hungarian Matching', 'EDHRec Synergy']


def load_land_names():
    """
    Lower-cased names of every land card.

    model_ready_card_vectors.csv drops 'type_line', so card types come from
    cleaned_mtg_card_features.csv -- the file the pre-refactor notebook loaded
    as raw_df. Cards EDHRec filed under "Land" are added on top.
    """
    global raw_df
    source = None
    cand = globals().get('raw_df')
    if isinstance(cand, pd.DataFrame) and 'type_line' in cand.columns and not cand.empty:
        source = cand
    elif os.path.exists(RAW_CSV_PATH):
        raw_df = pd.read_csv(RAW_CSV_PATH)
        source = raw_df
        print(f"Loaded raw_df for land filtering ({len(raw_df)} cards).")
    else:
        cand = globals().get('df_raw')   # output of the JSON loading cell, if it was run
        if isinstance(cand, pd.DataFrame) and 'type_line' in cand.columns and not cand.empty:
            source = cand
    if source is None:
        raise FileNotFoundError(
            f"Can't tell lands from spells: {RAW_CSV_PATH} is missing and there's no raw_df/df_raw "
            "with a 'type_line' column in memory.")

    is_land = source['type_line'].astype(str).str.contains('Land', na=False)
    lands = set(source.loc[is_land, 'name'].astype(str).str.lower().str.strip())
    return lands | globals().get('EDHREC_LAND_NAMES', set())


def evaluate_metric_robustness(base_commander, same_color_same_strat, same_color_diff_strat,
                               diff_color_commander, df, embeddings,
                               n_perturbations=10, perturbation_rate=0.20, random_seed=0,
                               csv_output='similarity_comparison.csv'):
    required = ['get_edhrec_average_and_score', 'build_name_index', 'lookup_card',
                'generate_perturbed_deck', 'get_commander_color_identity', 'filter_by_color_identity',
                'calculate_jaccard_similarity', 'calculate_jaccard_name_similarity',
                'calculate_pairwise_cosine_similarity', 'calculate_symmetric_best_match_similarity',
                'calculate_hungarian_similarity', 'calculate_edhrec_synergy_score']
    missing = [f for f in required if f not in globals()]
    if missing:
        raise NameError(f"Run the cells that define these first: {missing}")

    print(f"\n{'='*70}")
    print(f" EVALUATING METRIC ROBUSTNESS (Non-Land Spells Only)")
    print(f" Base Commander: {base_commander}")
    print(f"{'='*70}")

    land_names = load_land_names()
    lookup = build_name_index(df)

    def strip_qty(n):
        return re.sub(r'^[\s ]*\d+\s+', '', str(n)).strip()

    def get_spells_only(deck_list):
        return [strip_qty(n) for n in deck_list if strip_qty(n).lower() not in land_names]

    def get_embs_for_deck(deck_list, label):
        idxs, unmatched = [], []
        for n in deck_list:
            idx = lookup_card(n, lookup)
            if idx is None:
                unmatched.append(strip_qty(n))
            else:
                idxs.append(idx)
        if unmatched:
            print(f"  [{label}] {len(unmatched)} card(s) not in local dataset, skipped: {unmatched[:5]}")
        return embeddings[idxs] if idxs else np.empty((0, embeddings.shape[1]))

    def fetch_spells(commander):
        """Non-land cards of the commander's EDHRec average deck, or None if EDHRec failed."""
        _, raw = get_edhrec_average_and_score(commander)
        return get_spells_only(raw) if len(raw) >= 5 else None

    # 0. Base deck (EDHRec average). No made-up fallback: if this fails, stop.
    base_deck = fetch_spells(base_commander)
    if not base_deck:
        raise RuntimeError(
            f"Couldn't get the EDHRec average deck for {base_commander}. Stopping rather than "
            "substituting random cards -- re-run once EDHRec responds.")
    base_embs = get_embs_for_deck(base_deck, 'base')
    target_sample_size = len(base_deck)
    print(f"Base deck: {target_sample_size} non-land cards.")

    def metric_row(comp_name, deck, embs):
        if deck is None:   # EDHRec failed for this commander: leave the row blank, don't invent numbers
            return {'Comparison': comp_name, **{m: np.nan for m in METRICS}}
        has_embs = len(embs) > 0 and len(base_embs) > 0
        return {
            'Comparison': comp_name,
            'Jaccard (Text)': calculate_jaccard_similarity(deck, base_deck),
            'Jaccard (Name)': calculate_jaccard_name_similarity(deck, base_deck),
            'Pairwise Cosine': calculate_pairwise_cosine_similarity(embs, base_embs) if has_embs else np.nan,
            'Symmetric Best-Match': calculate_symmetric_best_match_similarity(embs, base_embs) if has_embs else np.nan,
            'Hungarian Matching': calculate_hungarian_similarity(embs, base_embs) if has_embs else np.nan,
            'EDHRec Synergy': calculate_edhrec_synergy_score(base_commander, deck),
        }

    results = []

    # 1. Itself
    results.append(metric_row(f'1. Itself ({base_commander})', base_deck, base_embs))

    # 2. Perturbed (mean and std dev over n_perturbations seeds)
    pert_rows = []
    for seed in range(n_perturbations):
        random.seed(seed)
        pert_full = generate_perturbed_deck(base_deck, df, base_commander, perturbation_rate=perturbation_rate)
        pert_deck = get_spells_only(pert_full)
        pert_rows.append(metric_row('pert', pert_deck, get_embs_for_deck(pert_deck, f'perturbed seed {seed}')))
    pert_df = pd.DataFrame(pert_rows)[METRICS]
    pct = int(round(perturbation_rate * 100))
    results.append({'Comparison': f'2. {pct}% Perturbed (Mean)', **pert_df.mean(skipna=False).to_dict()})
    results.append({'Comparison': '   (Std Dev)', **pert_df.std(ddof=0, skipna=False).to_dict()})

    # 3-5. Other commanders' EDHRec average decks
    failed = []
    for label, cmd in [('3. Same Color/Strat', same_color_same_strat),
                       ('4. Same Color/Diff Strat', same_color_diff_strat),
                       ('5. Different Colors', diff_color_commander)]:
        deck = fetch_spells(cmd)
        if deck is None:
            failed.append(cmd)
            print(f"  !! No EDHRec average deck for {cmd} -- its row will be NaN.")
            results.append(metric_row(f'{label} ({cmd})', None, None))
        else:
            results.append(metric_row(f'{label} ({cmd})', deck, get_embs_for_deck(deck, cmd)))

    # 6. Random legal spells, same count as the base deck's non-land cards
    cid = get_commander_color_identity(base_commander, df)
    legal_df = filter_by_color_identity(cid, df)
    spell_pool = [n for n in legal_df['name'].astype(str)
                  if n != base_commander and n.lower().strip() not in land_names]
    random_spells = random.Random(random_seed).sample(spell_pool, min(target_sample_size, len(spell_pool)))
    results.append(metric_row(f'6. Random Legal Spells (n={target_sample_size})', random_spells,
                              get_embs_for_deck(random_spells, 'random')))

    results_df = pd.DataFrame(results)[['Comparison'] + METRICS]

    # Save to CSV only when every row is real
    if failed:
        print(f"\nNot saving to {csv_output}: EDHRec failed for {failed}. Re-run for a complete table.")
    else:
        header = not os.path.exists(csv_output)
        results_df.to_csv(csv_output, mode='a', index=False, header=header)
        print(f"\nResults successfully saved to {csv_output}")

    display(results_df.round(4))
    return results_df


# Run the evaluation
if 'df' in globals() and not df.empty:
    def get_actual_name(name):
        names = df['name'].astype(str)
        exact = names[names.str.lower() == name.lower()]
        if not exact.empty:
            return exact.iloc[0]
        partial = names[names.str.contains(name, case=False, regex=False, na=False)]
        return partial.iloc[0] if not partial.empty else name

    robustness_df = evaluate_metric_robustness(
        base_commander=get_actual_name('Krenko, Mob Boss'),
        same_color_same_strat=get_actual_name('Krenko, Tin Street Kingpin'),
        same_color_diff_strat=get_actual_name('Torbran, Thane of Red Fell'),
        diff_color_commander=get_actual_name('Muldrotha, the Gravetide'),
        df=df,
        embeddings=embeddings
    )
else:
    print("df is empty or not loaded -- run the data-loading cell (cell 36) first. Nothing was evaluated.")



 EVALUATING METRIC ROBUSTNESS (Non-Land Spells Only)
 Base Commander: Krenko, Mob Boss
Fetching EDHRec data for: Krenko, Mob Boss (slug: krenko-mob-boss)
Average Deck Optimization Score: 0.4055
Matched 72 of 72 cards in local dataset.
Base deck: 66 non-land cards.
Fetching EDHRec data for: Krenko, Tin Street Kingpin (slug: krenko-tin-street-kingpin)
Average Deck Optimization Score: 0.3375
Matched 72 of 72 cards in local dataset.
Fetching EDHRec data for: Torbran, Thane of Red Fell (slug: torbran-thane-of-red-fell)
Average Deck Optimization Score: 0.3554
Matched 71 of 71 cards in local dataset.
Fetching EDHRec data for: Muldrotha, the Gravetide (slug: muldrotha-the-gravetide)
Average Deck Optimization Score: 0.3862
Matched 88 of 88 cards in local dataset.

Results successfully saved to similarity_comparison.csv


,Comparison,Jaccard (Text),Jaccard (Name),Pairwise Cosine,Symmetric Best-Match,Hungarian Matching,EDHRec Synergy
0,"1. Itself (Krenko, Mob Boss)",1.0000,1.0000,1.0000,1.0000,1.0000,0.3071
1,2. 20% Perturbed (Mean),0.6865,0.6813,0.9919,0.9263,0.8945,0.2550
2,(Std Dev),0.0139,0.0054,0.0020,0.0056,0.0087,0.0084
3,"3. Same Color/Strat (Krenko, Tin Street Kingpin)",0.5563,0.5595,0.9835,0.8975,0.8542,0.2427
4,"4. Same Color/Diff Strat (Torbran, Thane of Re...",0.1131,0.1000,0.7613,0.6388,0.5349,-0.0141
5,"5. Different Colors (Muldrotha, the Gravetide)",0.0418,0.0317,0.7318,0.5494,0.4781,0.0034
6,6. Random Legal Spells (n=66),0.0202,0.0000,0.8359,0.5353,0.4678,-0.0002


In [ ]:
# =============================================================================
# 20% perturbation with SIMILAR substitutes (vs. random substitutes)
#
# Same base deck and the same 20% of cards removed on each seed; only the way
# the replacement is chosen changes:
#   Random       - any legal non-land card (like row 2 above, but lands are
#                  excluded so the deck size stays the same)
#   Role-matched - same card type, same functional tags (ramp / draw / removal /
#                  wipe / tutor), mana value within 1. Uses card features only,
#                  not the embeddings.
#   EDHRec-rec   - a card EDHRec recommends for this commander that isn't in the
#                  average deck, same card type, closest mana value. Independent
#                  of the embeddings.
#   Embedding-NN - one of the 5 nearest neighbours in embedding space.
#                  CIRCULAR for SBM / Pairwise / Hungarian (the substitute is
#                  picked with the same embeddings the metric uses) -- shown as
#                  an upper bound only, don't use it as evidence for SBM.
#
# Name Jaccard is identical across methods by construction (the same number of
# cards is swapped), so any difference shows up only in the semantic metrics.
# =============================================================================
import os
import re
import random
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

CARD_TYPES = ['Creature', 'Planeswalker', 'Battle', 'Instant', 'Sorcery', 'Artifact', 'Enchantment']
ROLE_COLS = ['is_Ramp', 'is_Card_Draw', 'is_Spot_Removal', 'is_Board_Wipe', 'is_Tutor']
SUB_METRICS = ['Jaccard (Name)', 'Pairwise Cosine', 'Symmetric Best-Match', 'Hungarian Matching', 'EDHRec Synergy']


def primary_type(type_line):
    front = str(type_line).split(' // ')[0]
    return next((t for t in CARD_TYPES if t in front), 'Other')


def compare_substitution_methods(base_commander, df, embeddings, n_perturbations=10,
                                 perturbation_rate=0.20, nn_k=5,
                                 csv_output='substitution_comparison.csv'):
    print(f"\n{'='*70}\n SIMILAR vs RANDOM SUBSTITUTION ({int(round(perturbation_rate*100))}% of non-commander cards)"
          f"\n Base Commander: {base_commander}\n{'='*70}")

    land_names = load_land_names()          # also loads raw_df (cell above)
    lookup = build_name_index(df)
    strip_qty = lambda n: re.sub(r'^[\s ]*\d+\s+', '', str(n)).strip()

    # --- per-card features: type and mana value from raw_df, role tags from df
    info = (raw_df.assign(_key=raw_df['name'].astype(str).str.lower().str.strip())
                  .drop_duplicates('_key').set_index('_key'))
    def card_type(name):
        row = info['type_line'].get(name.lower()) if 'type_line' in info else None
        return primary_type(row) if row is not None else 'Other'
    mv_source = info['mana_value'] if 'mana_value' in info else None
    def mana_value(name, idx):
        if mv_source is not None and name.lower() in mv_source.index:
            return float(mv_source[name.lower()])
        return float(df.at[idx, 'mana_value']) if 'mana_value' in df.columns else 0.0
    role_cols = [c for c in ROLE_COLS if c in df.columns]
    def roles(idx):
        return tuple(int(df.at[idx, c] > 0.5) for c in role_cols)

    # --- base deck (non-land, commander included as in the main table)
    _, raw = get_edhrec_average_and_score(base_commander)
    base_deck = [strip_qty(n) for n in raw if strip_qty(n).lower() not in land_names]
    if len(base_deck) < 5:
        raise RuntimeError(f"Couldn't get the EDHRec average deck for {base_commander}.")
    base_idx = [lookup_card(n, lookup) for n in base_deck]
    base_embs = embeddings[[i for i in base_idx if i is not None]]
    commander_key = base_commander.lower()

    # --- candidate pool: legal, non-land, not the commander
    cid = get_commander_color_identity(base_commander, df)
    legal = filter_by_color_identity(cid, df)
    pool = [(n, i) for n, i in zip(legal['name'].astype(str), legal.index)
            if n.lower().strip() not in land_names and n.lower() != commander_key]
    pool_names = [n for n, _ in pool]
    pool_idx = np.array([i for _, i in pool])
    pool_type = {n: card_type(n) for n, _ in pool}
    pool_mv = {n: mana_value(n, i) for n, i in pool}
    pool_roles = {n: roles(i) for n, i in pool}
    pool_emb = embeddings[pool_idx]
    pool_pos = {n: k for k, n in enumerate(pool_names)}

    # EDHRec-recommended cards for this commander that are legal, non-land, not in the deck
    syn_map = get_edhrec_synergy_map(base_commander)
    base_lower = {n.lower() for n in base_deck}
    edhrec_pool = [n for n in pool_names if n.lower() in syn_map and n.lower() not in base_lower]
    print(f"Base deck: {len(base_deck)} non-land cards | legal pool: {len(pool_names)} | "
          f"EDHRec-recommended substitutes available: {len(edhrec_pool)}")

    # --- substitute pickers: (removed card name, removed df index, cards already in deck, rng) -> name or None
    def pick_random(name, idx, used, rng):
        for _ in range(50):
            c = rng.choice(pool_names)
            if c.lower() not in used:
                return c
        return None

    def pick_role(name, idx, used, rng):
        t, mv, r = card_type(name), mana_value(name, idx), roles(idx)
        for tol, need_roles in [(1, True), (2, True), (1, False), (99, False)]:
            cands = [c for c in pool_names if c.lower() not in used and pool_type[c] == t
                     and abs(pool_mv[c] - mv) <= tol and (not need_roles or pool_roles[c] == r)]
            if cands:
                return rng.choice(cands)
        return None

    def pick_edhrec(name, idx, used, rng):
        t, mv = card_type(name), mana_value(name, idx)
        cands = [c for c in edhrec_pool if c.lower() not in used]
        same_type = [c for c in cands if pool_type[c] == t] or cands
        if not same_type:
            return None
        same_type.sort(key=lambda c: abs(pool_mv[c] - mv))
        return rng.choice(same_type[:3])          # one of the 3 closest in mana value

    def pick_nn(name, idx, used, rng):
        sims = cosine_similarity(embeddings[idx].reshape(1, -1), pool_emb).ravel()
        order = [pool_names[k] for k in np.argsort(-sims)]
        cands = [c for c in order if c.lower() not in used and c.lower() != name.lower()][:nn_k]
        return rng.choice(cands) if cands else None

    methods = {'Random': pick_random, 'Role-matched': pick_role,
               'EDHRec-recommended': pick_edhrec, 'Embedding-NN (circular)': pick_nn}

    # --- run: same swapped positions per seed for every method
    swappable = [k for k, n in enumerate(base_deck) if n.lower() != commander_key and base_idx[k] is not None]
    n_swap = int(len(swappable) * perturbation_rate)
    rows, examples, failures = [], {}, {m: 0 for m in methods}
    for seed in range(n_perturbations):
        positions = random.Random(seed).sample(swappable, n_swap)
        for m, picker in methods.items():
            rng = random.Random(1000 + seed)
            deck = list(base_deck)
            used = {n.lower() for n in deck}
            for k in positions:
                sub = picker(base_deck[k], base_idx[k], used, rng)
                if sub is None:
                    failures[m] += 1
                    continue
                deck[k] = sub
                used.add(sub.lower())
                if seed == 0 and len(examples.setdefault(m, [])) < 5:
                    examples[m].append(f"{base_deck[k]} -> {sub}")
            embs = embeddings[[i for i in (lookup_card(n, lookup) for n in deck) if i is not None]]
            rows.append({'Method': m, 'seed': seed,
                         'Jaccard (Name)': calculate_jaccard_name_similarity(deck, base_deck),
                         'Pairwise Cosine': calculate_pairwise_cosine_similarity(embs, base_embs),
                         'Symmetric Best-Match': calculate_symmetric_best_match_similarity(embs, base_embs),
                         'Hungarian Matching': calculate_hungarian_similarity(embs, base_embs),
                         'EDHRec Synergy': calculate_edhrec_synergy_score(base_commander, deck)})

    print("\nExample swaps (seed 0):")
    for m, ex in examples.items():
        print(f"  {m}: " + "; ".join(ex))
    if any(failures.values()):
        print(f"\nSwaps with no valid substitute (card left unchanged): {failures}")

    per_seed = pd.DataFrame(rows)
    summary = per_seed.groupby('Method', sort=False)[SUB_METRICS].agg(['mean', 'std'])
    table = pd.DataFrame({m: summary[(m, 'mean')].map('{:.4f}'.format) + ' ± ' + summary[(m, 'std')].map('{:.4f}'.format)
                          for m in SUB_METRICS})
    table.insert(0, 'Commander', base_commander)

    header = not os.path.exists(csv_output)
    per_seed.assign(Commander=base_commander).to_csv(csv_output, mode='a', index=False, header=header)
    print(f"\nPer-seed results appended to {csv_output}")
    display(table)
    return per_seed


if 'df' in globals() and not df.empty:
    substitution_df = compare_substitution_methods('Krenko, Mob Boss', df, embeddings)
else:
    print("df is empty or not loaded -- run the Part 3 setup cell first.")



 SIMILAR vs RANDOM SUBSTITUTION (20% of non-commander cards)
 Base Commander: Krenko, Mob Boss
Fetching EDHRec data for: Krenko, Mob Boss (slug: krenko-mob-boss)
Average Deck Optimization Score: 0.4055
Matched 72 of 72 cards in local dataset.
Base deck: 66 non-land cards | legal pool: 6868 | EDHRec-recommended substitutes available: 123

Example swaps (seed 0):
  Random: Quest for the Goblin Lord -> Goblin Hero; Shared Animosity -> Kavu Aggressor; Battle Cry Goblin -> Guardian Automaton; Goblin Bombardment -> Endoskeleton; Hexing Squelcher -> Zealous Conscripts
  Role-matched: Quest for the Goblin Lord -> Maniacal Rage; Shared Animosity -> Mana Cannons; Battle Cry Goblin -> Ghitu Lavarunner; Goblin Bombardment -> Giant Spectacle; Hexing Squelcher -> Storm Herald
  EDHRec-recommended: Quest for the Goblin Lord -> Goblin Oriflamme; Shared Animosity -> Rising of the Day; Battle Cry Goblin -> Goblin Wardriver; Goblin Bombardment -> Cavalcade of Calamity; Hexing Squelcher -> Goblin Craterm

,Commander,Jaccard (Name),Pairwise Cosine,Symmetric Best-Match,Hungarian Matching,EDHRec Synergy
Method,,,,,,
Random,"Krenko, Mob Boss",0.6709 ± 0.0000,0.9915 ± 0.0022,0.9233 ± 0.0049,0.8837 ± 0.0058,0.2501 ± 0.0093
Role-matched,"Krenko, Mob Boss",0.6709 ± 0.0000,0.9917 ± 0.0025,0.9231 ± 0.0066,0.8851 ± 0.0095,0.2503 ± 0.0093
EDHRec-recommended,"Krenko, Mob Boss",0.6709 ± 0.0000,0.9933 ± 0.0017,0.9332 ± 0.0060,0.8974 ± 0.0091,0.2671 ± 0.0108
Embedding-NN (circular),"Krenko, Mob Boss",0.6709 ± 0.0000,0.9976 ± 0.0005,0.9552 ± 0.0045,0.9487 ± 0.0049,0.2538 ± 0.0087


### Precise Evaluation Methodology

To ensure scientific rigor, the following conditions are implemented as follows:

1.  **20% Perturbation**: We take the generated Genetic Algorithm deck (or EDHRec baseline) and replace exactly 20% of the non-land cards. Replacements are drawn from a **Filtered Legal Pool** (all cards matching the commander's color identity). This tests the model's sensitivity to minor thematic deviations.
2.  **Color-Identity Random Baseline**: 99 cards are sampled without replacement from the **Filtered Legal Pool**. This represents a deck that is 'legal' but lacks thematic cohesion.
3.  **Truly Random Baseline**: 99 cards are sampled from the **Global Pool** (ignoring color identity). This addresses the 'high similarity' issue by establishing a baseline for cards that share no structural or color-based metadata.
4.  **Cross-Commander Selection**:
    *   *Same Colors*: We select a commander with an identical Color Identity but a distinct theme (e.g., Animar vs. Averna) to see if the embeddings capture mechanical nuance beyond color.
    *   *Different Colors*: We select a commander from a completely different slice of the color pie to test the global distance in the embedding space.

In [ ]:
def evaluate_truly_random_baseline(base_commander, base_embs, df, embeddings):
    """
    Generates a baseline by sampling from the ENTIRE global dataset,
    ignoring color identity to find the absolute floor of similarity.
    """
    print(f"\n--- Evaluating Truly Random Baseline (Global Pool) ---")

    # Sample 99 cards from the full dataframe
    random_indices = df.sample(n=99).index.tolist()
    random_embs = embeddings[random_indices]

    # Calculate metrics against the base deck
    # (Using the best-match symmetric similarity logic from previous cells)
    sim_matrix = cosine_similarity(random_embs, base_embs)
    ga_to_rand = sim_matrix.max(axis=1).mean()
    rand_to_ga = sim_matrix.max(axis=0).mean()
    symmetric_sim = 0.5 * (ga_to_rand + rand_to_ga)

    print(f"Truly Random (Global) Similarity: {symmetric_sim:.4f}")
    return symmetric_sim

# Execution Example
if 'base_embs' in locals():
    truly_random_sim = evaluate_truly_random_baseline(base_commander, base_embs, df, embeddings)

## Compare two different deck lists using 5 metrics
Takes 2 text files (Moxfield export -> MTGO format)

In [ ]:
import re
import os
import numpy as np
import pandas as pd

def parse_deck_file(file_path):
    """
    Parses a single deck file to extract card names and the commander name.
    Assumes the commander name is the last line in the file.
    """
    with open(file_path, 'r') as f:
        lines = f.readlines()

    if not lines:
        return [], ""

    # The last line is the commander name
    raw_commander_name = lines[-1].strip()
    # Clean commander name by removing leading numbers/quantities
    commander_name = re.sub(r'^[\d]+\s+', '', raw_commander_name)

    # All other lines are card names
    card_lines = lines[:-1]

    deck_names = []
    for line in card_lines:
        line = line.strip()
        if not line: continue
        # Remove leading numbers/quantities (e.g., '1 Sol Ring' -> 'Sol Ring')
        clean_name = re.sub(r'^[\d]+\s+', '', line)
        deck_names.append(clean_name)

    return deck_names, commander_name

def get_embs_and_texts(names):
    global df, embeddings

    # Ensure df is available and not empty
    if 'df' not in globals() or df.empty:
        print("Error: DataFrame 'df' is not loaded. Cannot get embeddings.")
        return np.array([]), []

    # Load embeddings if they are empty or missing
    if 'embeddings' not in globals() or embeddings is None or len(embeddings) == 0:
        DRIVE_PATH = '/content/drive/MyDrive/MTGCISE'
        NPY_PATH = os.path.join(DRIVE_PATH, EMBEDDINGS_NPY)
        if os.path.exists(NPY_PATH):
            print(f"Embeddings empty in kernel. Automatically loading {EMBEDDINGS_NPY} from Drive...")
            embeddings = np.load(NPY_PATH)
        else:
            print("Error: Embeddings file not found on Drive. Vector metrics cannot be computed.")
            return np.array([]), []

    # Reset index of global df to align with embeddings if not already done
    if not isinstance(df.index, pd.RangeIndex):
        df = df.reset_index(drop=True)

    # Filter the DataFrame for the given names while keeping track of current positional index
    filtered_df = df[df['name'].isin(names)]

    # Use the positional integers (0 to N-1) to fetch vectors from embeddings
    indices = filtered_df.index.tolist()
    deck_embeddings = embeddings[indices] if indices and len(embeddings) > 0 else np.array([])

    # Get oracle texts for Jaccard (text) calculation
    oracle_texts = filtered_df['oracle_text_clean'].tolist() if 'oracle_text_clean' in filtered_df.columns else []

    return deck_embeddings, oracle_texts

# Define the deck pairs to compare
deck_pairs = [
    ("/content/Kozilek, the Great Distortion 1.txt", "/content/Kozilek, the Great Distortion 2.txt"),
    ("/content/Kozilek, the Great Distortion 1.txt", "/content/Jodah, the Unifier 1.txt"),
    ("/content/Gonti, Lord of Luxury 1.txt", "/content/Gonti, Lord of Luxury 2.txt"),
    ("/content/Gonti, Lord of Luxury 1.txt", "/content/Averna, the Chaos Bloom 1.txt"),
    ("/content/Krenko, Mob Boss 1.txt", "/content/Krenko, Mob Boss 2.txt"),
    ("/content/Krenko, Mob Boss 1.txt", "/content/Chatterfang, Squirrel General 1.txt"),
    ("/content/Teysa Karlov 1.txt", "/content/Teysa Karlov 2.txt"),
    ("/content/Teysa Karlov 1.txt", "/content/Aesi, Tyrant of Gyre Strait 1.txt"),
    ("/content/Aesi, Tyrant of Gyre Strait 1.txt", "/content/Aesi, Tyrant of Gyre Strait 2.txt"),
    ("/content/Aesi, Tyrant of Gyre Strait 1.txt", "/content/Krenko, Mob Boss 1.txt"),
    ("/content/Chatterfang, Squirrel General 1.txt", "/content/Chatterfang, Squirrel General 2.txt"),
    ("/content/Chatterfang, Squirrel General 1.txt", "/content/Krenko, Mob Boss 2.txt"),
    ("/content/Cloud, Ex-SOLDIER 1.txt", "/content/Cloud, Ex-SOLDIER 2.txt"),
    ("/content/Cloud, Ex-SOLDIER 1.txt", "/content/Gonti, Lord of Luxury 1.txt"),
    ("/content/Averna, the Chaos Bloom 1.txt", "/content/Averna, the Chaos Bloom 2.txt"),
    ("/content/Averna, the Chaos Bloom 1.txt", "/content/Gonti, Lord of Luxury 2.txt"),
    ("/content/Kenrith, the Returned King 1.txt", "/content/Kenrith, the Returned King 2.txt"),
    ("/content/Kenrith, the Returned King 1.txt", "/content/Kozilek, the Great Distortion 1.txt"),
    ("/content/Jodah, the Unifier 1.txt", "/content/Jodah, the Unifier 2.txt"),
    ("/content/Jodah, the Unifier 1.txt", "/content/Kozilek, the Great Distortion 2.txt")
]

# Get all unique deck files mentioned in the original list
all_deck_files = set()
for f1, f2 in deck_pairs:
    all_deck_files.add(f1)
    all_deck_files.add(f2)

jodah1_file = "/content/Jodah, the Unifier 1.txt"
jodah2_file = "/content/Jodah, the Unifier 2.txt"

# Add new comparisons for Jodah, the Unifier 1.txt
for other_file in all_deck_files:
    if jodah1_file != other_file and (jodah1_file, other_file) not in deck_pairs and (other_file, jodah1_file) not in deck_pairs:
        deck_pairs.append((jodah1_file, other_file))

# Add new comparisons for Jodah, the Unifier 2.txt
for other_file in all_deck_files:
    if jodah2_file != other_file and (jodah2_file, other_file) not in deck_pairs and (other_file, jodah2_file) not in deck_pairs:
        deck_pairs.append((jodah2_file, other_file))

results_list = []

for file1, file2 in deck_pairs:
    try:
        deck1_names, deck1_cmd = parse_deck_file(file1)
        deck2_names, deck2_cmd = parse_deck_file(file2)

        # 1. Jaccard Similarity (Name)
        jaccard_name = calculate_jaccard_name_similarity(deck1_names, deck2_names)

        # Embeddings and Oracle Texts
        deck1_embs, deck1_texts = get_embs_and_texts(deck1_names)
        deck2_embs, deck2_texts = get_embs_and_texts(deck2_names)

        # 2. Jaccard Similarity (Oracle Text)
        jaccard_text = calculate_jaccard_similarity(deck1_texts, deck2_texts)

        # 3. Pairwise Cosine
        pairwise = calculate_pairwise_cosine_similarity(deck1_embs, deck2_embs)

        # 4. Symmetric Best-Match
        sym_best = calculate_symmetric_best_match_similarity(deck1_embs, deck2_embs)

        # 5. Hungarian Matching Similarity
        hungarian = calculate_hungarian_similarity(deck1_embs, deck2_embs)

        pair_name = f"{file1.split('/')[-1].replace('.txt', '')} vs {file2.split('/')[-1].replace('.txt', '')}"
        results_list.append({
            'Pair': pair_name,
            'Jaccard (Name)': jaccard_name,
            'Jaccard (Oracle Text)': jaccard_text,
            'Pairwise Cosine': pairwise,
            'Symmetric Best-Match': sym_best,
            'Hungarian Matching': hungarian
        })
    except Exception as e:
        print(f"Error processing pair {file1} vs {file2}: {e}")

results_df = pd.DataFrame(results_list)
display(results_df.round(4))

,Pair,Jaccard (Name),Jaccard (Oracle Text),Pairwise Cosine,Symmetric Best-Match,Hungarian Matching
0,"Kozilek, the Great Distortion 1 vs Kozilek, th...",0.3465,0.6405,0.9890,0.8964,0.8673
1,"Kozilek, the Great Distortion 1 vs Jodah, the ...",0.0227,0.4535,0.9534,0.7582,0.7123
2,"Gonti, Lord of Luxury 1 vs Gonti, Lord of Luxu...",0.1504,0.5159,0.9869,0.8419,0.8147
3,"Gonti, Lord of Luxury 1 vs Averna, the Chaos B...",0.0081,0.4499,0.9445,0.7329,0.6955
4,"Krenko, Mob Boss 1 vs Krenko, Mob Boss 2",0.4019,0.6656,0.9939,0.8938,0.8667
5,"Krenko, Mob Boss 1 vs Chatterfang, Squirrel Ge...",0.0455,0.4290,0.9699,0.7757,0.7474
6,Teysa Karlov 1 vs Teysa Karlov 2,0.1810,0.4915,0.9890,0.8591,0.8294
7,"Teysa Karlov 1 vs Aesi, Tyrant of Gyre Strait 1",0.0000,0.4045,0.8980,0.7229,0.6768
8,"Aesi, Tyrant of Gyre Strait 1 vs Aesi, Tyrant ...",0.1193,0.4847,0.9788,0.8092,0.7707
9,"Aesi, Tyrant of Gyre Strait 1 vs Krenko, Mob B...",0.0000,0.4132,0.9202,0.7080,0.6880


# Compare two individual cards

In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
import re

def compare_two_cards(card1_name, card2_name, df, embeddings):
    """
    Compares two individual cards using Jaccard Similarity (text)
    and Cosine Similarity (embeddings).
    """
    # 1. Setup and card lookup
    df_lower = df.copy()
    df_lower['name_lower'] = df_lower['name'].str.lower()

    c1_data = df_lower[df_lower['name_lower'] == card1_name.lower()]
    c2_data = df_lower[df_lower['name_lower'] == card2_name.lower()]

    if c1_data.empty or c2_data.empty:
        return f"Error: One or both cards not found. ({card1_name}, {card2_name})"

    # 2. Jaccard Similarity (Mechanical Keywords)
    STOP_WORDS = {'is', 'and', 'the', 'a', 'an', 'it', 'to', 'of', 'in', 'on', 'with', 'by', 'for', 'at', 'this', 'that'}

    def get_keywords(text):
        if not isinstance(text, str): return set()
        words = re.findall(r'\w+', text.lower())
        return set([w for w in words if w not in STOP_WORDS])

    # Use raw_df if available for original text, otherwise use df's text column
    text_col = 'oracle_text' if 'oracle_text' in df.columns else 'oracle_text_clean'

    k1 = get_keywords(c1_data[text_col].iloc[0])
    k2 = get_keywords(c2_data[text_col].iloc[0])

    intersection = len(k1.intersection(k2))
    union = len(k1.union(k2))
    jaccard_score = intersection / union if union > 0 else 0.0

    # 3. Cosine Similarity (Embeddings)
    idx1 = c1_data.index[0]
    idx2 = c2_data.index[0]

    vec1 = embeddings[idx1].reshape(1, -1)
    vec2 = embeddings[idx2].reshape(1, -1)
    cosine_score = cosine_similarity(vec1, vec2)[0][0]

    print(f"--- Comparison: {card1_name} vs {card2_name} ---")
    print(f"Jaccard Similarity (Text Keywords): {jaccard_score:.4f}")
    print(f"Cosine Similarity (Embeddings):    {cosine_score:.4f}")

    return {"jaccard": jaccard_score, "cosine": cosine_score}


compare_two_cards('Camellia, the Seedmiser', 'Aesi, Tyrant of Gyre Strait', df, embeddings)

# Baseline comparison of 100 completely random cards

In [ ]:
def compare_random_card_sets(df, embeddings, sample_size=100):
    """
    Randomly selects two sets of cards and calculates the average
    pairwise cosine similarity and the symmetric best-match similarity.
    """
    # Use the entire pool of indices from the dataframe
    card_indices = df.index.tolist()

    if len(card_indices) < (sample_size * 2):
        return "Error: Not enough cards in dataset."

    # 2. Randomly sample indices for two sets
    sampled_indices = random.sample(card_indices, sample_size * 2)
    set1_indices = sampled_indices[:sample_size]
    set2_indices = sampled_indices[sample_size:]

    # 3. Extract embeddings
    set1_embs = embeddings[set1_indices]
    set2_embs = embeddings[set2_indices]

    # 4. Calculate pairwise similarity matrix (100x100)
    sim_matrix = cosine_similarity(set1_embs, set2_embs)

    # 5. Calculate Metrics
    # Metric A: Global average of all 10,000 pairs
    avg_similarity = np.mean(sim_matrix)

    # Metric B: Symmetric Best-Match Similarity
    # For each card in set 1, find the best match in set 2, and vice versa.
    set1_to_set2_best = sim_matrix.max(axis=1).mean()
    set2_to_set1_best = sim_matrix.max(axis=0).mean()
    symmetric_best_match = 0.5 * (set1_to_set2_best + set2_to_set1_best)

    print(f"--- Random Set Comparison ({sample_size} vs {sample_size}) ---")
    print(f"Average Pairwise Cosine Similarity:     {avg_similarity:.4f}")
    print(f"Symmetric Best-Match Similarity:        {symmetric_best_match:.4f}")

    return {"pairwise": avg_similarity, "best_match": symmetric_best_match}

# Run a demonstration
if 'df' in globals() and 'embeddings' in globals():
    results = compare_random_card_sets(df, embeddings)
else:
    print("Please ensure 'df' and 'embeddings' are loaded in the kernel.")

In [ ]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import os

def summarize_cross_commander_metrics(csv_path='/content/drive/MyDrive/MTGCISE/similarity_comparison.csv', output_path='/content/drive/MyDrive/MTGCISE/cross_commander_summary.csv'):
    """
    Reads the similarity comparison results, cleans the condition labels,
    and calculates the overall Average, Standard Deviation, 95% Confidence Intervals,
    statistical significance (p-values vs '1. Itself' baseline), and Cohen's d effect sizes.
    """
    if not os.path.exists(csv_path):
        print(f"Error: File '{csv_path}' not found.")
        return None

    # Load the data
    df_raw_sim = pd.read_csv(csv_path)

    # Filter out the individual "(Std Dev)" rows from the perturbed calculations
    df_filtered = df_raw_sim[~df_raw_sim['Comparison'].str.contains(r'\(Std Dev\)', na=False)].copy()

    # Map the specific comparison string to a general condition category
    def categorize_condition(comp_str):
        if '1. Itself' in comp_str: return '1. Itself'
        if '2. 20% Perturbed' in comp_str: return '2. 20% Perturbed'
        if '3. Same Color/Strat' in comp_str: return '3. Same Color / Same Strat'
        if '4. Same Color/Diff Strat' in comp_str: return '4. Same Color / Diff Strat'
        if '5. Different Colors' in comp_str: return '5. Different Colors'
        if '6. Random Legal Spells' in comp_str: return '6. Random Legal Spells'
        return 'Other'

    df_filtered['Condition'] = df_filtered['Comparison'].apply(categorize_condition)

    # Define the metrics we want to analyze
    metrics = [
        'Jaccard (Text)', 'Jaccard (Name)', 'Pairwise Cosine',
        'Symmetric Best-Match', 'EDHRec Synergy'
    ]

    # Group by clean condition category and calculate fundamental stats
    grouped = df_filtered.groupby('Condition')

    summary_rows = []
    baseline_data = df_filtered[df_filtered['Condition'] == '1. Itself']

    for cond, group in grouped:
        row_data = {'Condition': cond}
        for metric in metrics:
            if metric not in group.columns:
                continue
            vals = group[metric].dropna().to_numpy()
            n = len(vals)
            mean = np.mean(vals) if n > 0 else np.nan
            std = np.std(vals, ddof=1) if n > 1 else 0.0

            # Calculate 95% Confidence Interval
            if n > 1 and std > 0:
                sem = stats.sem(vals)
                ci_bound = stats.t.ppf(0.975, df=n-1) * sem
                ci_lower = mean - ci_bound
                ci_upper = mean + ci_bound
            else:
                ci_lower, ci_upper = mean, mean

            row_data[f"{metric}_Mean"] = mean
            row_data[f"{metric}_Std"] = std
            row_data[f"{metric}_CI_95"] = (round(ci_lower, 4), round(ci_upper, 4))

            # Significance test & Effect Size (vs '1. Itself' baseline)
            base_vals = baseline_data[metric].dropna().to_numpy() if not baseline_data.empty and metric in baseline_data.columns else np.array([])
            if cond != '1. Itself' and len(vals) > 1 and len(base_vals) > 1:
                # T-test for independent samples
                stat, p_val = stats.ttest_ind(vals, base_vals, equal_var=False)
                # Cohen's d effect size calculation
                pooled_std = np.sqrt(((len(vals) - 1) * (std**2) + (len(base_vals) - 1) * (np.std(base_vals, ddof=1)**2)) / (len(vals) + len(base_vals) - 2))
                cohen_d = (mean - np.mean(base_vals)) / pooled_std if pooled_std > 0 else 0.0
            else:
                p_val = np.nan
                cohen_d = 0.0

            row_data[f"{metric}_p_value_vs_baseline"] = p_val
            row_data[f"{metric}_cohens_d"] = cohen_d

        summary_rows.append(row_data)

    summary_df = pd.DataFrame(summary_rows)

    # Display and Save
    print(f"--- Cross-Commander Summary ({len(df_filtered)} total runs processed) ---")
    pd.set_option('display.max_columns', None)
    display(summary_df.round(4))

    summary_df.to_csv(output_path, index=False)
    print(f"Saved comprehensive statistical summary to '{output_path}'")

    return summary_df

# Run the summary function
aggregate_summary = summarize_cross_commander_metrics()

--- Cross-Commander Summary (60 total runs processed) ---


,Condition,Jaccard (Text)_Mean,Jaccard (Text)_Std,Jaccard (Text)_CI_95,Jaccard (Text)_p_value_vs_baseline,Jaccard (Text)_cohens_d,Jaccard (Name)_Mean,Jaccard (Name)_Std,Jaccard (Name)_CI_95,Jaccard (Name)_p_value_vs_baseline,Jaccard (Name)_cohens_d,Pairwise Cosine_Mean,Pairwise Cosine_Std,Pairwise Cosine_CI_95,Pairwise Cosine_p_value_vs_baseline,Pairwise Cosine_cohens_d,Symmetric Best-Match_Mean,Symmetric Best-Match_Std,Symmetric Best-Match_CI_95,Symmetric Best-Match_p_value_vs_baseline,Symmetric Best-Match_cohens_d,EDHRec Synergy_Mean,EDHRec Synergy_Std,EDHRec Synergy_CI_95,EDHRec Synergy_p_value_vs_baseline,EDHRec Synergy_cohens_d
0,1. Itself,1.0000,0.0000,"(1.0, 1.0)",NaN,0.0000,1.0000,0.0000,"(1.0, 1.0)",NaN,0.0000,1.0000,0.0000,"(1.0, 1.0)",NaN,0.0000,1.0000,0.0000,"(1.0, 1.0)",NaN,0.0000,0.2646,0.1156,"(0.1819, 0.3474)",NaN,0.0000
1,2. 20% Perturbed,0.6948,0.0136,"(0.6851, 0.7046)",0.0,-31.7340,0.6851,0.0075,"(0.6797, 0.6905)",0.0,-58.9868,0.9908,0.0013,"(0.9899, 0.9917)",0.0000,-10.0413,0.9276,0.0052,"(0.9238, 0.9313)",0.0000,-19.6738,0.2145,0.0933,"(0.1477, 0.2812)",0.3006,-0.4773
2,3. Same Color / Same Strat,0.4355,0.1799,"(0.3068, 0.5642)",0.0,-4.4370,0.4310,0.1842,"(0.2992, 0.5628)",0.0,-4.3679,0.9643,0.0281,"(0.9442, 0.9844)",0.0030,-1.7987,0.8393,0.0882,"(0.7763, 0.9024)",0.0003,-2.5772,0.1540,0.0897,"(0.0898, 0.2182)",0.0287,-1.0688
3,4. Same Color / Diff Strat,0.1235,0.0802,"(0.0661, 0.1809)",0.0,-15.4560,0.1060,0.0663,"(0.0585, 0.1535)",0.0,-19.0596,0.8432,0.0712,"(0.7922, 0.8942)",0.0001,-3.1130,0.6500,0.0633,"(0.6047, 0.6953)",0.0000,-7.8168,0.0105,0.0151,"(-0.0003, 0.0213)",0.0001,-3.0818
4,5. Different Colors,0.0342,0.0094,"(0.0275, 0.0409)",0.0,-145.4539,0.0198,0.0067,"(0.015, 0.0246)",0.0,-207.2598,0.7642,0.0887,"(0.7007, 0.8276)",0.0000,-3.7601,0.5648,0.0297,"(0.5435, 0.586)",0.0000,-20.7242,0.0023,0.0045,"(-0.0009, 0.0055)",0.0001,-3.2056
5,6. Random Legal Spells,0.0282,0.0192,"(0.0144, 0.0419)",0.0,-71.5494,0.0040,0.0079,"(-0.0016, 0.0097)",0.0,-178.7396,0.8704,0.0443,"(0.8387, 0.9021)",0.0000,-4.1354,0.5622,0.0293,"(0.5412, 0.5832)",0.0000,-21.1016,0.0012,0.0022,"(-0.0004, 0.0028)",0.0001,-3.2211


Saved comprehensive statistical summary to '/content/drive/MyDrive/MTGCISE/cross_commander_summary.csv'


In [ ]:
import pandas as pd
import scipy.stats as stats
import os

file_path = 'similarity_comparison.csv'
if os.path.exists(file_path):
    # Load the raw data to calculate correlation across all runs
    df_sim = pd.read_csv(file_path)

    # Filter out the standard deviation rows
    df_sim = df_sim[~df_sim['Comparison'].str.contains(r'\(Std Dev\)', na=False)].copy()

    # Calculate Pearson correlation
    metrics_to_correlate = ['Jaccard (Text)', 'Pairwise Cosine', 'Symmetric Best-Match']
    target_metric = 'EDHRec Synergy'

    print("Pearson Correlation with EDHRec Synergy:")
    for metric in metrics_to_correlate:
        if metric in df_sim.columns and target_metric in df_sim.columns:
            corr, p_value = stats.pearsonr(df_sim[metric], df_sim[target_metric])
            print(f"{metric}: {corr:.4f} (p-value: {p_value:.4e})")
        else:
            print(f"Missing columns for {metric} or {target_metric}")

    # Display the summary trend
    summary_file = 'cross_commander_summary.csv'
    if os.path.exists(summary_file):
        summary_df = pd.read_csv(summary_file)
        display_cols = ['Condition', 'Jaccard (Text)_Mean', 'Pairwise Cosine_Mean', 'Symmetric Best-Match_Mean', 'EDHRec Synergy_Mean']
        existing_cols = [c for c in display_cols if c in summary_df.columns]
        print("\n--- Trend Summary ---")
        display(summary_df[existing_cols])
else:
    print(f"Error: {file_path} not found.")

## Metric robustness with real player decks
Same 6 conditions as the EDHRec version, but every deck is a real player-made deck (Moxfield exports, the same 20 files used in the deck-list comparison). Base = each commander's deck 1. Results go to `similarity_comparison_player_decks.csv`.

In [ ]:
# =============================================================================
# Metric robustness with Moxfield exports
#
# Uses the same 20 deck files as the "Compare two different deck lists" section
# (2 player decks for each of 10 commanders). For each commander, deck 1 is the base:
#   1. Itself            deck 1 vs deck 1
#   2. 20% Perturbed     deck 1 with 20% of its cards swapped for random legal cards (10 seeds, mean)
#   3. Same Strategy     deck 1 vs deck 2 of the SAME commander (a different player's build)
#   4. Diff Strategy     deck 1 vs the different-commander deck it was paired with earlier in the notebook
#   5. Different Colors  deck 1 vs a deck whose commander shares NO colors with the base (best effort: for the
#                        5-color commanders the only option is colorless Kozilek, see DIFF_COLORS)
#   6. Random            random non-land cards legal in the base commander's colors, same size as deck 1
# Lands are removed from every deck so all conditions compare spells to spells (same as the EDHRec version).
# =============================================================================
import os
import re
import glob
import shutil
import random
import numpy as np
import pandas as pd

DRIVE_DIR = '/content/drive/MyDrive/MTGCISE'
DECK_DIR = os.path.join(DRIVE_DIR, 'player_decks')          # permanent copy of the deck files
OUT_CSV = os.path.join(DRIVE_DIR, 'similarity_comparison_player_decks.csv')
N_PERTURBATIONS, PERTURBATION_RATE, RANDOM_SEED = 10, 0.20, 0

COMMANDERS = ['Kozilek, the Great Distortion', 'Gonti, Lord of Luxury', 'Krenko, Mob Boss', 'Teysa Karlov',
              'Aesi, Tyrant of Gyre Strait', 'Chatterfang, Squirrel General', 'Cloud, Ex-SOLDIER',
              'Averna, the Chaos Bloom', 'Kenrith, the Returned King', 'Jodah, the Unifier']

# Condition 4: the different-commander pairings already used in the deck-list comparison cell.
DIFF_STRATEGY = {
    'Kozilek, the Great Distortion': 'Jodah, the Unifier 1',
    'Gonti, Lord of Luxury': 'Averna, the Chaos Bloom 1',
    'Krenko, Mob Boss': 'Chatterfang, Squirrel General 1',
    'Teysa Karlov': 'Aesi, Tyrant of Gyre Strait 1',
    'Aesi, Tyrant of Gyre Strait': 'Krenko, Mob Boss 1',
    'Chatterfang, Squirrel General': 'Krenko, Mob Boss 2',
    'Cloud, Ex-SOLDIER': 'Gonti, Lord of Luxury 1',
    'Averna, the Chaos Bloom': 'Gonti, Lord of Luxury 2',
    'Kenrith, the Returned King': 'Kozilek, the Great Distortion 1',
    'Jodah, the Unifier': 'Kozilek, the Great Distortion 2',
}

# Condition 5: a deck whose commander shares no colors with the base commander (filled in DIFF_COLORS below).
DIFF_COLORS = {
    'Kozilek, the Great Distortion': 'Krenko, Mob Boss 1',             # C   vs R    (colorless base: any colored deck qualifies)
    'Gonti, Lord of Luxury': 'Aesi, Tyrant of Gyre Strait 2',          # B   vs UG
    'Krenko, Mob Boss': 'Teysa Karlov 2',                              # R   vs WB
    'Teysa Karlov': 'Averna, the Chaos Bloom 2',                       # WB  vs URG
    'Aesi, Tyrant of Gyre Strait': 'Teysa Karlov 2',                   # UG  vs WB
    'Chatterfang, Squirrel General': 'Kozilek, the Great Distortion 1',  # BG vs C   (only Krenko/Kozilek share no colors; Krenko is already cond. 4)
    'Cloud, Ex-SOLDIER': 'Kozilek, the Great Distortion 2',            # WRG vs C   (only Gonti/Kozilek qualify; Gonti is already cond. 4)
    'Averna, the Chaos Bloom': 'Teysa Karlov 2',                       # URG vs WB
    'Kenrith, the Returned King': 'Kozilek, the Great Distortion 2',   # WUBRG vs C (only colorless Kozilek qualifies; same commander as cond. 4)
    'Jodah, the Unifier': 'Kozilek, the Great Distortion 1',           # WUBRG vs C (only colorless Kozilek qualifies; same commander as cond. 4)
}

METRICS = ['Jaccard (Text)', 'Jaccard (Name)', 'Pairwise Cosine',
           'Symmetric Best-Match', 'Hungarian Matching', 'EDHRec Synergy']

needed = ['df', 'embeddings', 'raw_df', 'build_name_index', 'lookup_card', 'get_commander_color_identity',
          'filter_by_color_identity', 'generate_perturbed_deck', 'calculate_jaccard_similarity',
          'calculate_jaccard_name_similarity', 'calculate_pairwise_cosine_similarity',
          'calculate_symmetric_best_match_similarity', 'calculate_hungarian_similarity',
          'calculate_edhrec_synergy_score']
missing = [n for n in needed if n not in globals()]
if missing:
    raise NameError(f"Run the Part 3 setup cell and the metric/perturbation cells first. Missing: {missing}")

# --- Deck files: keep a permanent copy on Drive (files in /content vanish when the runtime resets) ---
os.makedirs(DECK_DIR, exist_ok=True)
for p in glob.glob('/content/*.txt'):
    dest = os.path.join(DECK_DIR, os.path.basename(p))
    if not os.path.exists(dest):
        shutil.copy2(p, dest)


def read_deck(label):
    """Cards from '<label>.txt' (Moxfield/MTGO export). The last line is the commander and is left out."""
    path = os.path.join(DECK_DIR, f'{label}.txt')
    lines = [l.strip() for l in open(path, encoding='utf-8') if l.strip()]
    return [re.sub(r'^\d+\s+', '', l) for l in lines[:-1]]


# --- Land filter, card lookup, embeddings -------------------------------------
LAND_NAMES = set(raw_df.loc[raw_df['type_line'].astype(str).str.contains('Land', na=False), 'name']
                 .astype(str).str.lower().str.strip())
NAME_INDEX = build_name_index(df)
TEXT_COL = 'oracle_text_clean' if 'oracle_text_clean' in df.columns else 'oracle_text'


def spells_only(names):
    return [n for n in names if n.lower().strip() not in LAND_NAMES]


def deck_vectors(names, label):
    idxs, unmatched = [], []
    for n in names:
        i = lookup_card(n, NAME_INDEX)
        if i is None:
            unmatched.append(n)
        else:
            idxs.append(i)
    if unmatched:
        print(f"  [{label}] {len(unmatched)} card(s) not in local dataset, skipped: {unmatched[:5]}")
    texts = df.loc[idxs, TEXT_COL].tolist() if idxs else []
    embs = embeddings[idxs] if idxs else np.empty((0, embeddings.shape[1]))
    return embs, texts


all_rows, deck_log = [], []
for cmd in COMMANDERS:
    print(f"\n=== {cmd} ===")
    raw_base = read_deck(f'{cmd} 1')
    base = spells_only(raw_base)
    base_embs, base_texts = deck_vectors(base, 'base')
    print(f"Base deck: {len(raw_base)} cards in file, {len(base)} non-land.")

    def metric_row(cond, deck_names, other_label=''):
        embs, texts = deck_vectors(deck_names, cond)
        ok = len(embs) > 0 and len(base_embs) > 0
        return {'Commander': cmd, 'Comparison': f'{cond} ({other_label})' if other_label else cond,
                'Jaccard (Text)': calculate_jaccard_similarity(texts, base_texts),
                'Jaccard (Name)': calculate_jaccard_name_similarity(deck_names, base),
                'Pairwise Cosine': calculate_pairwise_cosine_similarity(embs, base_embs) if ok else np.nan,
                'Symmetric Best-Match': calculate_symmetric_best_match_similarity(embs, base_embs) if ok else np.nan,
                'Hungarian Matching': calculate_hungarian_similarity(embs, base_embs) if ok else np.nan,
                'EDHRec Synergy': calculate_edhrec_synergy_score(cmd, deck_names)}

    rows = [metric_row('1. Itself', base, f'{cmd} 1')]

    pert = []
    for seed in range(N_PERTURBATIONS):
        random.seed(seed)
        pert.append(metric_row('p', spells_only(generate_perturbed_deck(base, df, cmd, perturbation_rate=PERTURBATION_RATE))))
    pert_df = pd.DataFrame(pert)[METRICS]
    pct = int(round(PERTURBATION_RATE * 100))
    rows.append({'Commander': cmd, 'Comparison': f'2. {pct}% Perturbed (Mean)', **pert_df.mean().to_dict()})
    rows.append({'Commander': cmd, 'Comparison': '   (Std Dev)', **pert_df.std(ddof=0).to_dict()})

    for cond, label in [('3. Same Strategy', f'{cmd} 2'),
                        ('4. Diff Strategy', DIFF_STRATEGY[cmd]),
                        ('5. Different Colors', DIFF_COLORS[cmd])]:
        rows.append(metric_row(cond, spells_only(read_deck(label)), label))

    cid = get_commander_color_identity(cmd, df)
    legal = filter_by_color_identity(cid, df)
    pool = [n for n in legal['name'].astype(str) if n != cmd and n.lower().strip() not in LAND_NAMES]
    rand = random.Random(RANDOM_SEED).sample(pool, min(len(base), len(pool)))
    rows.append(metric_row(f'6. Random Legal Spells (n={len(base)})', rand))

    all_rows.extend(rows)
    deck_log.append({'Commander': cmd, 'Colors': cid, 'Same Strategy': f'{cmd} 2',
                     'Diff Strategy': DIFF_STRATEGY[cmd], 'Different Colors': DIFF_COLORS[cmd],
                     'Base non-land cards': len(base)})

results = pd.DataFrame(all_rows)[['Commander', 'Comparison'] + METRICS]
results.to_csv(OUT_CSV, index=False)   # overwritten on each run, so re-running never duplicates rows

pairing = pd.DataFrame(deck_log)
def colors_of(deck_label):
    return set(get_commander_color_identity(re.sub(r' \d+$', '', deck_label), df)) - {'C'}

pairing['Shares colors with base?'] = [bool((set(c) - {'C'}) & colors_of(d))
                                       for c, d in zip(pairing['Colors'], pairing['Different Colors'])]
pairing['Cond. 4 and 5 same commander?'] = [re.sub(r' \d+$', '', a) == re.sub(r' \d+$', '', b)
                                             for a, b in zip(pairing['Diff Strategy'], pairing['Different Colors'])]
print("\n=== Deck pairings used ===")
display(pairing)

summary = (results[~results['Comparison'].str.contains(r'\(Std Dev\)')]
           .assign(Condition=lambda d: d['Comparison'].str.extract(r'^(\d\. [^(]+)')[0].str.strip())
           .groupby('Condition')[METRICS].mean())
print("\n=== Mean across the 10 commanders ===")
display(summary.round(4))
print(f"\nSaved per-commander rows to {OUT_CSV}")


=== Kozilek, the Great Distortion ===
Base deck: 64 cards in file, 64 non-land.
  [3. Same Strategy] 3 card(s) not in local dataset, skipped: ['Arc Reactor', 'Galactus, Devourer of Worlds', 'The Ten Rings']
  [4. Diff Strategy] 3 card(s) not in local dataset, skipped: ['Jennifer Walters', 'Makdee and Itla, Skysnarers', 'Shang-Chi, Master of Kung Fu']
  [5. Different Colors] 2 card(s) not in local dataset, skipped: ['_____ Goblin', 'Hex Magic']

=== Gonti, Lord of Luxury ===
Base deck: 64 cards in file, 64 non-land.

=== Krenko, Mob Boss ===
  [base] 2 card(s) not in local dataset, skipped: ['_____ Goblin', 'Hex Magic']
Base deck: 75 cards in file, 75 non-land.
  [1. Itself] 2 card(s) not in local dataset, skipped: ['_____ Goblin', 'Hex Magic']
  [p] 1 card(s) not in local dataset, skipped: ['_____ Goblin']
  [p] 2 card(s) not in local dataset, skipped: ['_____ Goblin', 'Hex Magic']
  [p] 1 card(s) not in local dataset, skipped: ['_____ Goblin']
  [p] 2 card(s) not in local dataset, sk

,Commander,Colors,Same Strategy,Diff Strategy,Different Colors,Base non-land cards,Shares colors with base?,Cond. 4 and 5 same commander?
0,"Kozilek, the Great Distortion",C,"Kozilek, the Great Distortion 2","Jodah, the Unifier 1","Krenko, Mob Boss 1",64,False,False
1,"Gonti, Lord of Luxury",B,"Gonti, Lord of Luxury 2","Averna, the Chaos Bloom 1","Aesi, Tyrant of Gyre Strait 2",64,False,False
2,"Krenko, Mob Boss",R,"Krenko, Mob Boss 2","Chatterfang, Squirrel General 1",Teysa Karlov 2,75,False,False
3,Teysa Karlov,WB,Teysa Karlov 2,"Aesi, Tyrant of Gyre Strait 1","Averna, the Chaos Bloom 2",61,False,False
4,"Aesi, Tyrant of Gyre Strait",UG,"Aesi, Tyrant of Gyre Strait 2","Krenko, Mob Boss 1",Teysa Karlov 2,59,False,False
5,"Chatterfang, Squirrel General",BG,"Chatterfang, Squirrel General 2","Krenko, Mob Boss 2","Kozilek, the Great Distortion 1",63,False,False
6,"Cloud, Ex-SOLDIER",WRG,"Cloud, Ex-SOLDIER 2","Gonti, Lord of Luxury 1","Kozilek, the Great Distortion 2",67,False,False
7,"Averna, the Chaos Bloom",URG,"Averna, the Chaos Bloom 2","Gonti, Lord of Luxury 2",Teysa Karlov 2,60,False,False
8,"Kenrith, the Returned King",WUBRG,"Kenrith, the Returned King 2","Kozilek, the Great Distortion 1","Kozilek, the Great Distortion 2",71,False,True
9,"Jodah, the Unifier",WUBRG,"Jodah, the Unifier 2","Kozilek, the Great Distortion 2","Kozilek, the Great Distortion 1",71,False,True



=== Mean across the 10 commanders ===


,Jaccard (Text),Jaccard (Name),Pairwise Cosine,Symmetric Best-Match,Hungarian Matching,EDHRec Synergy
Condition,,,,,,
1. Itself,1.0000,1.0000,1.0000,1.0000,1.0000,0.1170
2. 20% Perturbed,0.8200,0.6800,0.9907,0.9272,0.8892,0.0951
3. Same Strategy,0.5760,0.2500,0.9619,0.7738,0.7163,0.1176
4. Diff Strategy,0.4474,0.0175,0.8294,0.5941,0.5203,0.0032
5. Different Colors,0.4419,0.0216,0.8283,0.5952,0.5158,0.0037
6. Random Legal Spells,0.4442,0.0031,0.8782,0.5726,0.4984,0.0029



Saved per-commander rows to /content/drive/MyDrive/MTGCISE/similarity_comparison_player_decks.csv


## Reviewer #5: Is the SBM vs. PCS difference statistically meaningful?
Per commander: SBM gap and PCS gap for (same strategy − different strategy) and (different strategy − random). Paired t-test, 95% CI, and Cohen's d_z across the 10 commanders, plus a common-scale check.

In [ ]:
# =============================================================================
# Is SBM's separation of strategy levels statistically better than PCS?
#
# For each of the 10 base commanders (raw rows from the file picked by DATA_SOURCE below):
#   A) Same vs. different strategy:  gap = score(condition 3, same strategy) - score(condition 4, different strategy)
#   B) Different strategy vs random: gap = score(condition 4, different strategy) - score(condition 6, random spells)
# computed separately for SBM and PCS. Then, across commanders:
#   * paired t-test  SBM gap vs PCS gap  (+ 95% CI of the mean difference, Cohen's d_z, Wilcoxon check)
#   * each metric's own gap vs 0         (does the metric separate the two conditions at all?)
#   * scale check: same test after putting both metrics on a common scale
#     (each gap divided by that metric's SD over all rows), because SBM's values
#     spread over a wider range than PCS's, which can make raw SBM gaps look bigger.
# =============================================================================
import os
import re
import numpy as np
import pandas as pd
import scipy.stats as stats

# Which experiment to test: 'player' = real player decks (cell above), 'edhrec' = EDHRec average decks
DATA_SOURCE = 'player'
_FILES = {
    'player': ('similarity_comparison_player_decks.csv', 'gap_significance_summary_player_decks.csv'),
    'edhrec': ('similarity_comparison.csv', 'gap_significance_summary.csv'),
}
SIM_CSV = '/content/drive/MyDrive/MTGCISE/' + _FILES[DATA_SOURCE][0]
OUT_CSV = '/content/drive/MyDrive/MTGCISE/' + _FILES[DATA_SOURCE][1]

METRIC_A = 'Symmetric Best-Match'   # SBM
METRIC_B = 'Pairwise Cosine'        # PCS (swap in 'Hungarian Matching' to test that one too)
SHORT = {METRIC_A: 'SBM', METRIC_B: 'PCS'}

# Conditions are identified by their number, so both experiments' label wordings work
COND_LABELS = {'1': 'Itself', '2': 'Perturbed', '3': 'SameStrat', '4': 'DiffStrat', '5': 'DiffColor', '6': 'Random'}
GAPS = {  # name: (higher condition, lower condition)
    'Same vs Diff Strategy': ('SameStrat', 'DiffStrat'),
    'Diff Strategy vs Random': ('DiffStrat', 'Random'),
}


def load_per_commander(path):
    raw = pd.read_csv(path)
    raw = raw[~raw['Comparison'].str.contains(r'\(Std Dev\)', na=False)].copy()

    def cond_of(s):
        m = re.match(r'^\s*(\d)\.', str(s))
        return COND_LABELS.get(m.group(1)) if m else None
    raw['Condition'] = raw['Comparison'].map(cond_of)
    raw = raw[raw['Condition'].notna()].copy()

    # Rows were appended one evaluation at a time, each starting with "1. Itself (<commander>)".
    raw['Run'] = (raw['Condition'] == 'Itself').cumsum()
    if 'Commander' not in raw.columns:   # EDHRec file: commander name is inside the "1. Itself (...)" label
        base = raw.loc[raw['Condition'] == 'Itself', ['Run', 'Comparison']].copy()
        base['Commander'] = base['Comparison'].str.extract(r'1\. Itself \((.*)\)\s*$')[0]
        raw = raw.merge(base[['Run', 'Commander']], on='Run', how='left')

    counts = raw.groupby('Run')['Condition'].nunique()
    bad = counts[counts != len(COND_LABELS)]
    if len(bad):
        print(f"!! Dropping incomplete run(s) {list(bad.index)} (missing conditions).")
        raw = raw[~raw['Run'].isin(bad.index)]
    dup = raw.loc[raw['Condition'] == 'Itself', 'Commander'].value_counts()
    if (dup > 1).any():
        print(f"!! Note: these commanders were evaluated more than once: {list(dup[dup > 1].index)}")
    return raw


def paired_summary(x, y=None):
    """Mean, 95% CI, t, p, Cohen's d_z, Wilcoxon p for x - y (or for x alone vs 0)."""
    d = np.asarray(x, float) - (0.0 if y is None else np.asarray(y, float))
    n = len(d)
    mean, sd = d.mean(), d.std(ddof=1)
    se = sd / np.sqrt(n)
    tcrit = stats.t.ppf(0.975, n - 1)
    t, p = stats.ttest_1samp(d, 0.0)
    try:
        w_p = stats.wilcoxon(d).pvalue
    except ValueError:
        w_p = np.nan
    return {
        'n': n, 'mean_diff': mean, 'sd_diff': sd,
        'ci95_low': mean - tcrit * se, 'ci95_high': mean + tcrit * se,
        't': t, 'df': n - 1, 'p_value': p, 'cohens_dz': mean / sd if sd > 0 else np.nan,
        'wilcoxon_p': w_p, 'n_positive': int((d > 0).sum()),
    }


if not os.path.exists(SIM_CSV):
    print(f"Can't find {SIM_CSV}. Mount Drive (cell 25) and make sure the robustness runs were saved.")
else:
    long_df = load_per_commander(SIM_CSV)
    wide = long_df.pivot_table(index=['Run', 'Commander'], columns='Condition',
                               values=[METRIC_A, METRIC_B], aggfunc='first')

    # Common-scale factor: each metric's SD over every row in the file (all conditions, all commanders)
    scale = {m: long_df[m].std(ddof=1) for m in (METRIC_A, METRIC_B)}

    per_cmd = pd.DataFrame(index=wide.index)
    for m in (METRIC_A, METRIC_B):
        for c in ['SameStrat', 'DiffStrat', 'Random']:
            per_cmd[f"{SHORT[m]} {c}"] = wide[(m, c)]
    for gname, (hi, lo) in GAPS.items():
        for m in (METRIC_A, METRIC_B):
            per_cmd[f"{SHORT[m]} gap: {gname}"] = wide[(m, hi)] - wide[(m, lo)]
        per_cmd[f"SBM-PCS: {gname}"] = (per_cmd[f"SBM gap: {gname}"] - per_cmd[f"PCS gap: {gname}"])
    per_cmd = per_cmd.reset_index().drop(columns='Run')

    print(f"Commanders in analysis: {len(per_cmd)}")
    print("\n=== Per-commander raw scores and gaps ===")
    display(per_cmd.round(4))

    rows = []
    for gname, (hi, lo) in GAPS.items():
        g_a = per_cmd[f"SBM gap: {gname}"].to_numpy()
        g_b = per_cmd[f"PCS gap: {gname}"].to_numpy()
        rows.append({'Gap': gname, 'Test': 'SBM gap vs PCS gap (raw, paired)', **paired_summary(g_a, g_b)})
        rows.append({'Gap': gname, 'Test': 'SBM gap vs PCS gap (common scale, paired)',
                     **paired_summary(g_a / scale[METRIC_A], g_b / scale[METRIC_B])})
        rows.append({'Gap': gname, 'Test': 'SBM gap vs 0', **paired_summary(g_a)})
        rows.append({'Gap': gname, 'Test': 'PCS gap vs 0', **paired_summary(g_b)})
    summary = pd.DataFrame(rows)

    print(f"\nCommon-scale divisors (SD over all rows): SBM={scale[METRIC_A]:.4f}, PCS={scale[METRIC_B]:.4f}")
    print("\n=== Paired tests across commanders (two-sided, alpha = 0.05) ===")
    show = summary.copy()
    show['95% CI'] = show.apply(lambda r: f"[{r.ci95_low:.4f}, {r.ci95_high:.4f}]", axis=1)
    show['p_value'] = show['p_value'].map(lambda p: f"{p:.2e}")
    show['wilcoxon_p'] = show['wilcoxon_p'].map(lambda p: f"{p:.2e}" if pd.notna(p) else 'n/a')
    display(show[['Gap', 'Test', 'n', 'mean_diff', 'sd_diff', '95% CI', 't', 'df', 'p_value',
                  'cohens_dz', 'wilcoxon_p', 'n_positive']].round(4))

    # Plain-language readout
    print("\n=== Readout ===")
    for gname in GAPS:
        s = summary[summary['Gap'] == gname].set_index('Test')
        raw_row = s.loc['SBM gap vs PCS gap (raw, paired)']
        cs_row = s.loc['SBM gap vs PCS gap (common scale, paired)']
        print(f"\n{gname}:")
        for t in ['SBM gap vs 0', 'PCS gap vs 0']:
            r = s.loc[t]
            print(f"  {t[:3]} mean gap = {r.mean_diff:+.4f}  95% CI [{r.ci95_low:+.4f}, {r.ci95_high:+.4f}]  "
                  f"p = {r.p_value:.2e}  d_z = {r.cohens_dz:+.2f}  ({r.n_positive}/{int(r.n)} commanders > 0)")
        print(f"  SBM - PCS (raw):          {raw_row.mean_diff:+.4f}  95% CI [{raw_row.ci95_low:+.4f}, "
              f"{raw_row.ci95_high:+.4f}]  t({int(raw_row.df)}) = {raw_row.t:.2f}  p = {raw_row.p_value:.2e}  "
              f"d_z = {raw_row.cohens_dz:+.2f}")
        print(f"  SBM - PCS (common scale): {cs_row.mean_diff:+.4f}  95% CI [{cs_row.ci95_low:+.4f}, "
              f"{cs_row.ci95_high:+.4f}]  t({int(cs_row.df)}) = {cs_row.t:.2f}  p = {cs_row.p_value:.2e}  "
              f"d_z = {cs_row.cohens_dz:+.2f}")

    summary.to_csv(OUT_CSV, index=False)
    per_cmd.to_csv(OUT_CSV.replace('.csv', '_per_commander.csv'), index=False)
    print(f"\nSaved: {OUT_CSV} and the per-commander table next to it.")

Commanders in analysis: 10

=== Per-commander raw scores and gaps ===


,Commander,SBM SameStrat,SBM DiffStrat,SBM Random,PCS SameStrat,PCS DiffStrat,PCS Random,SBM gap: Same vs Diff Strategy,PCS gap: Same vs Diff Strategy,SBM-PCS: Same vs Diff Strategy,SBM gap: Diff Strategy vs Random,PCS gap: Diff Strategy vs Random,SBM-PCS: Diff Strategy vs Random
0,"Kozilek, the Great Distortion",0.8348,0.6303,0.6120,0.9761,0.8498,0.8892,0.2045,0.1263,0.0782,0.0183,-0.0394,0.0577
1,"Gonti, Lord of Luxury",0.7261,0.5760,0.6054,0.9614,0.8236,0.9412,0.1501,0.1378,0.0123,-0.0294,-0.1176,0.0882
2,"Krenko, Mob Boss",0.8346,0.6184,0.5805,0.9821,0.8883,0.9112,0.2161,0.0938,0.1223,0.0379,-0.0229,0.0608
3,Teysa Karlov,0.7705,0.5440,0.5938,0.9710,0.7112,0.9116,0.2265,0.2598,-0.0333,-0.0498,-0.2004,0.1505
4,"Aesi, Tyrant of Gyre Strait",0.7393,0.5413,0.5726,0.9431,0.7373,0.8239,0.1980,0.2058,-0.0077,-0.0313,-0.0865,0.0552
5,"Chatterfang, Squirrel General",0.7769,0.6067,0.5644,0.9546,0.8765,0.9044,0.1702,0.0781,0.0920,0.0423,-0.0279,0.0702
6,"Cloud, Ex-SOLDIER",0.7889,0.5806,0.5723,0.9751,0.8431,0.8779,0.2083,0.1320,0.0763,0.0082,-0.0348,0.0431
7,"Averna, the Chaos Bloom",0.7587,0.5693,0.5437,0.9338,0.8194,0.7852,0.1894,0.1144,0.0749,0.0256,0.0342,-0.0085
8,"Kenrith, the Returned King",0.8944,0.6421,0.5494,0.9880,0.8603,0.8765,0.2523,0.1277,0.1246,0.0927,-0.0162,0.1089
9,"Jodah, the Unifier",0.6143,0.6319,0.5321,0.9339,0.8840,0.8612,-0.0175,0.0499,-0.0674,0.0998,0.0229,0.0769



Common-scale divisors (SD over all rows): SBM=0.1761, PCS=0.0820

=== Paired tests across commanders (two-sided, alpha = 0.05) ===


,Gap,Test,n,mean_diff,sd_diff,95% CI,t,df,p_value,cohens_dz,wilcoxon_p,n_positive
0,Same vs Diff Strategy,"SBM gap vs PCS gap (raw, paired)",10,0.0472,0.0667,"[-0.0005, 0.0950]",2.2381,9,5.20e-02,0.7078,4.88e-02,7
1,Same vs Diff Strategy,"SBM gap vs PCS gap (common scale, paired)",10,-0.5949,0.6277,"[-1.0439, -0.1459]",-2.9971,9,1.50e-02,-0.9478,9.77e-03,2
2,Same vs Diff Strategy,SBM gap vs 0,10,0.1798,0.0749,"[0.1262, 0.2334]",7.5888,9,3.37e-05,2.3998,3.91e-03,9
3,Same vs Diff Strategy,PCS gap vs 0,10,0.1326,0.0608,"[0.0891, 0.1761]",6.8950,9,7.11e-05,2.1804,1.95e-03,10
4,Diff Strategy vs Random,"SBM gap vs PCS gap (raw, paired)",10,0.0703,0.0418,"[0.0404, 0.1002]",5.3197,9,4.81e-04,1.6822,3.91e-03,9
5,Diff Strategy vs Random,"SBM gap vs PCS gap (common scale, paired)",10,0.7174,0.6424,"[0.2578, 1.1769]",3.5314,9,6.40e-03,1.1167,3.91e-03,9
6,Diff Strategy vs Random,SBM gap vs 0,10,0.0214,0.0501,"[-0.0144, 0.0573]",1.3533,9,2.09e-01,0.4279,3.22e-01,7
7,Diff Strategy vs Random,PCS gap vs 0,10,-0.0489,0.0696,"[-0.0987, 0.0009]",-2.2201,9,5.36e-02,-0.7021,3.71e-02,2



=== Readout ===

Same vs Diff Strategy:
  SBM mean gap = +0.1798  95% CI [+0.1262, +0.2334]  p = 3.37e-05  d_z = +2.40  (9/10 commanders > 0)
  PCS mean gap = +0.1326  95% CI [+0.0891, +0.1761]  p = 7.11e-05  d_z = +2.18  (10/10 commanders > 0)
  SBM - PCS (raw):          +0.0472  95% CI [-0.0005, +0.0950]  t(9) = 2.24  p = 5.20e-02  d_z = +0.71
  SBM - PCS (common scale): -0.5949  95% CI [-1.0439, -0.1459]  t(9) = -3.00  p = 1.50e-02  d_z = -0.95

Diff Strategy vs Random:
  SBM mean gap = +0.0214  95% CI [-0.0144, +0.0573]  p = 2.09e-01  d_z = +0.43  (7/10 commanders > 0)
  PCS mean gap = -0.0489  95% CI [-0.0987, +0.0009]  p = 5.36e-02  d_z = -0.70  (2/10 commanders > 0)
  SBM - PCS (raw):          +0.0703  95% CI [+0.0404, +0.1002]  t(9) = 5.32  p = 4.81e-04  d_z = +1.68
  SBM - PCS (common scale): +0.7174  95% CI [+0.2578, +1.1769]  t(9) = 3.53  p = 6.40e-03  d_z = +1.12

Saved: /content/drive/MyDrive/MTGCISE/gap_significance_summary_player_decks.csv and the per-commander table ne